In [ ]:

import os
import math
import random
from typing import Tuple, List, Dict  # Only for type hints, not for instantiation
import copy
import time
import psutil
import inspect
import gc

import h5py
import joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras import mixed_precision

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, top_k_accuracy_score, classification_report
from sklearn.utils.class_weight import compute_class_weight



In [ ]:
from Loss_Func import get_spatio_temporal_focal_loss,huber_loss_tf,label_smoothing_sparse_categorical_crossentropy,weighted_mse_loss
from TCN import TCN,compiled_tcn

In [ ]:
# Set device for TensorFlow
print("TensorFlow version:", tf.__version__)
print('Keras version:', tf.keras.__version__)
print("GPUs available:", tf.config.list_physical_devices('GPU'))
device = "GPU" if tf.config.list_physical_devices('GPU') else "CPU"
print("Device:", device)

In [ ]:
gc.collect()

In [ ]:

# policy = mixed_precision.Policy('mixed_float16')
policy = mixed_precision.Policy('float32')
mixed_precision.set_global_policy(policy)
gc.collect()

## Configurations

In [ ]:
file_path = f'combined_matches.csv'
# file_path = f'combined_matches_2068, 2269, 2417.csv'

USE_PREPROCESSED = True
PREPROCESSED_DIR = './preprocessed_data' # Change path if needed for Kaggle: '/kaggle/input/preprocessed-sequences/'



N_ROWS = 3
N_COLS = 9
NUM_ZONES = N_ROWS * N_COLS

# Sequence & horizon
SEQ_LEN =50    # number of past frames used
FPS = 10            # adjust based on your data (often 25fps for football)
HORIZON_SECONDS =3 # predict location after 3/5 seconds
HORIZON_FRAMES = HORIZON_SECONDS * FPS

# Training params
BATCH_SIZE =256
LR = 5e-4   # 🔧 FIXED: Lowered from 3e-3 to 1e-4 for stability
EPOCHS =60
PATIENCE = 3 # for early stopping

# Enhanced features for better zone prediction
#42
# FEATURE_COLS = [
#     # Position
#     "x_normalized", "y_normalized",

#     # Basic movement
#     "dx", "dy", "speed_normalized", "acceleration",
#     "movement_angle",

#     # # Multi-window velocities (NEW)
#     "dx_avg_3", "dy_avg_3", "dx_avg_5", "dy_avg_5", "dx_avg_10", "dy_avg_10",
#     "speed_avg_3", "speed_avg_5", "speed_avg_10",

#     # Movement trends (NEW)
#     "acceleration_trend", "angle_change", "angle_stability", "speed_change_rate",

#     # Spatial context
#     "distance_from_center", "distance_from_goal_home", "distance_from_goal_away",
#     "distance_from_sideline",

#     # Ball & team
#     "distance_to_ball", "ball_possession_proximity", "team_has_possession", "team_spread",

#     'home_score', 'away_score','nearest_opponent_1', 'nearest_opponent_2', 'nearest_opponent_3', 
#     'defensive_line_distance','team_centroid_distance',

#     "is_goalkeeper","is_defender","is_midfielder","is_forward","has_possession","team_has_possession","opponent_has_possession"


# ]

# 1️⃣ Explicitly separate your feature types
SCALE_COLS = [
    "x_normalized", "y_normalized", "dx", "dy", "speed_normalized", "acceleration",
    "movement_angle", "dx_avg_3", "dy_avg_3", "dx_avg_5", "dy_avg_5", "dx_avg_10", "dy_avg_10",
    "speed_avg_3", "speed_avg_5", "speed_avg_10", "acceleration_trend", "angle_change", 
    "angle_stability", "speed_change_rate", "distance_from_center", "distance_from_goal_home", 
    "distance_from_goal_away", "distance_from_sideline"
]

BINARY_COLS = [
    "is_goalkeeper", "is_defender", "is_midfielder", "is_forward",
    "has_possession", "team_has_possession", "opponent_has_possession"
]


#32
# FEATURE_COLS = [
#     # Position
#     "x_normalized", "y_normalized",

#     # Basic movement
#     "dx_avg_3", "dy_avg_3", "speed_normalized", "acceleration",
#     "movement_angle",

#     # Movement trends (NEW)
#     "acceleration_trend", "angle_change", "angle_stability", "speed_change_rate",

#     # Spatial context
#     "distance_from_center", "distance_from_goal_home", "distance_from_goal_away",
#     "distance_from_sideline",

#     # Ball & team
#     "distance_to_ball", "ball_possession_proximity",  "team_spread",

#     'home_score', 'away_score','nearest_opponent_1', 'nearest_opponent_2', 'nearest_opponent_3', 
#     'defensive_line_distance','team_centroid_distance','position_encoded'

# ]
# 26
FEATURE_COLS = [
    # Position
    "x_normalized", "y_normalized",

    # Basic movement
    "dx", "dy", "speed_normalized", "acceleration",
    "movement_angle",

    # Movement trends (NEW)
    "acceleration_trend", "angle_change", "angle_stability", "speed_change_rate",

    # Spatial context
    "distance_from_center", "distance_from_goal_home", "distance_from_goal_away",
    "distance_from_sideline",

    # Ball & team
    "distance_to_ball", "ball_possession_proximity",  "team_spread",

    'home_score', 'away_score','nearest_opponent_1', 'nearest_opponent_2', 'nearest_opponent_3', 
    'defensive_line_distance','team_centroid_distance','position_encoded'

]

##24
# FEATURE_COLS = [
#     "x_normalized", "y_normalized",
#     "dx", "dy", "speed_normalized", "acceleration",
#     "movement_angle",
#     "dx_avg_3", "dy_avg_3", "dx_avg_5", "dy_avg_5", "dx_avg_10", "dy_avg_10",
#     "speed_avg_3", "speed_avg_5", "speed_avg_10",
#     "acceleration_trend", "angle_change", "angle_stability", "speed_change_rate",
#     "distance_from_center", "distance_from_goal_home", "distance_from_goal_away",
#     "distance_from_sideline",
# ]
#11
# FEATURE_COLS = [
#     "x_normalized", "y_normalized",
#     "dx", "dy", "speed_normalized",
#     "movement_angle",
#     # Keep smoothed averages, drop raw acceleration trends
#     "dx_avg_5", "dy_avg_5", "speed_avg_5",
#     "distance_from_goal_home", "distance_from_center"
# ]

coord_cols=("x_normalized", "y_normalized")

some_number= 42

worker_num=4
USE_ATTENTION_MECHANISM= True


# ═══════════════════════════════════════════════════════════════════════════════
# MODEL SELECTION
# ═══════════════════════════════════════════════════════════════════════════════
# Available MODEL_TYPE options:
#   - "Keras_tcn"     : Keras TCN for classification/regression
#   - "MDN_TCN"       : 🆕 Mixture Density Network (multi-modal probabilistic)
# ═══════════════════════════════════════════════════════════════════════════════

MODEL_TYPE = "MDN_TCN"  # Change to "MDN_TCN" for probabilistic predictions
BACKEND = "keras"       # keras/torch
CO_ORDINATES = True
COMBINED=False

# Configuration for enhanced training
USE_COSINE_LR = True           # Toggle cosine LR vs ReduceLROnPlateau
WARMUP_EPOCHS = 5              # Linear warmup epochs
LABEL_SMOOTHING = 0.1          # Label smoothing for classification (0 = off)
MAX_LR = LR                # Peak learning rate
MIN_LR = 1e-5                  # Final learning rate

MDN_NUM_MIXTURES = 10     # Number of Gaussian components (K)
MDN_SIGMA_MIN = 1e-4      # Minimum sigma to prevent collapse
MDN_LEARNING_RATE = 5e-5  # Lower LR for MDN stability


if MODEL_TYPE == "MDN_TCN":   
    print("⚠️  MDN_TCN selected: CO_ORDINATES auto-set to True")
    CO_ORDINATES = True  # MDN always uses coordinates
    MDN_OUTPUT_DIM = 2


In [ ]:
# import seaborn as sns
# import matplotlib.pyplot as plt

# # Calculate correlation matrix
# corr_matrix = df[FEATURE_COLS].corr()

# # Visualize the correlation matrix
# plt.figure(figsize=(20, 16))
# sns.heatmap(corr_matrix, annot=True, cmap='coolwarm')
# plt.show()

# # Find highly correlated pairs
# threshold = 0.9  # Set a correlation threshold to identify highly correlated features
# high_corr_pairs = [(col1, col2) for col1 in corr_matrix.columns for col2 in corr_matrix.columns if abs(corr_matrix[col1][col2]) > threshold and col1 != col2]
# print(f"Highly correlated pairs: {high_corr_pairs}")

## Keras TCN enhanced

In [ ]:
try:
    # pylint: disable=E0611,E0401
    from tensorflow.keras.utils import register_keras_serializable
  # For recent Keras
except ImportError:
    # pylint: disable=E0611,E0401
    from tensorflow.keras.utils import register_keras_serializable  # For older versions

# pylint: disable=E0611,E0401
from tensorflow.keras import backend as K, Model, Input, optimizers
# pylint: disable=E0611,E0401
from tensorflow.keras import layers
# pylint: disable=E0611,E0401
from tensorflow.keras.layers import Activation, SpatialDropout1D, Lambda
# pylint: disable=E0611,E0401
from tensorflow.keras.layers import Layer, Conv1D, Dense, BatchNormalization, LayerNormalization


## LSTM

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers

class SpatioTemporalLSTM(tf.keras.Model):
    def __init__(self,
                 lstm_units=128,
                 num_layers=2,
                 dropout_rate=0.3,
                 coordinate_targets=False,
                 num_zones=None):

        super().__init__()

        self.coordinate_targets = coordinate_targets

        # Stack LSTMs
        self.lstm_layers = []
        for i in range(num_layers):
            self.lstm_layers.append(
                layers.LSTM(
                    lstm_units,
                    return_sequences=(i < num_layers - 1)
                )
            )

        self.dropout = layers.Dropout(dropout_rate)

        # Output layer depends on task
        if coordinate_targets:
            self.output_layer = layers.Dense(2)  # regression dx, dy
        else:
            self.output_layer = layers.Dense(num_zones, activation="softmax")

    def call(self, inputs, training=False):
        x = inputs

        for lstm in self.lstm_layers:
            x = lstm(x)

        x = self.dropout(x, training=training)
        return self.output_layer(x)

In [ ]:
def LSTMTraining():
    with strategy.scope():
        model = SpatioTemporalLSTM(
            lstm_units=128,
            num_layers=2,
            dropout_rate=0.3,
            coordinate_targets=CO_ORDINATES,
            num_zones=NUM_ZONES
        )

        model.build(input_shape=(None, SEQ_LEN, len(FEATURE_COLS)))

        if CO_ORDINATES:
            model.compile(
                optimizer=tf.keras.optimizers.Adam(1e-3),
                loss='mse',
                metrics=['mae']
            )
        else:
            model.compile(
                optimizer=tf.keras.optimizers.Adam(1e-3),
                loss=tf.keras.losses.SparseCategoricalCrossentropy(),
                metrics=[
                    "accuracy",
                    tf.keras.metrics.SparseTopKCategoricalAccuracy(k=3, name="top3_acc"),
                    tf.keras.metrics.SparseTopKCategoricalAccuracy(k=5, name="top5_acc")
                ]
            )
    return model

def LSTMEval(model):
    if CO_ORDINATES:
        model.compile(
            optimizer=tf.keras.optimizers.Adam(1e-3),
            loss='mse',
            metrics=['mae']
        )
    else:
        model.compile(
            optimizer=tf.keras.optimizers.Adam(1e-3),
            loss=tf.keras.losses.SparseCategoricalCrossentropy(),
            metrics=['accuracy']
        )
    test_results = model.evaluate(
        test_loader,
        steps=test_steps
    )
    print("Test Results:", test_results)


In [ ]:


# @misc{KerasTCN,
#   author = {Philippe Remy},
#   title = {Temporal Convolutional Networks for Keras},
#   year = {2020},
#   publisher = {GitHub},
#   journal = {GitHub repository},
#   howpublished = {\url{https://github.com/philipperemy/keras-tcn}},
# }



def is_power_of_two(num: int):
    return num != 0 and ((num & (num - 1)) == 0)


def adjust_dilations(dilations: list):
    if all([is_power_of_two(i) for i in dilations]):
        return dilations
    else:
        new_dilations = [2 ** i for i in dilations]
        return new_dilations


class ResidualBlock(Layer):

    def __init__(self,
                 dilation_rate: int,
                 nb_filters: int,
                 kernel_size: int,
                 padding: str,
                 activation: str = 'relu',
                 dropout_rate: float = 0,
                 kernel_initializer: str = 'he_normal',
                 use_batch_norm: bool = False,
                 use_layer_norm: bool = False,
                 **kwargs):
        """Defines the residual block for the WaveNet TCN
        Args:
            x: The previous layer in the model
            training: boolean indicating whether the layer should behave in training mode or in inference mode
            dilation_rate: The dilation power of 2 we are using for this residual block
            nb_filters: The number of convolutional filters to use in this block
            kernel_size: The size of the convolutional kernel
            padding: The padding used in the convolutional layers, 'same' or 'causal'.
            activation: The final activation used in o = Activation(x + F(x))
            dropout_rate: Float between 0 and 1. Fraction of the input units to drop.
            kernel_initializer: Initializer for the kernel weights matrix (Conv1D).
            use_batch_norm: Whether to use batch normalization in the residual layers or not.
            use_layer_norm: Whether to use layer normalization in the residual layers or not.
            kwargs: Any initializers for Layer class.
        """

        self.dilation_rate = dilation_rate
        self.nb_filters = nb_filters
        self.kernel_size = kernel_size
        self.padding = padding
        self.activation = activation
        self.dropout_rate = dropout_rate
        self.use_batch_norm = use_batch_norm
        self.use_layer_norm = use_layer_norm
        self.kernel_initializer = kernel_initializer
        self.layers = []
        self.shape_match_conv = None
        self.res_output_shape = None
        self.final_activation = None
        self.batch_norm_layers = []
        self.layer_norm_layers = []

        super(ResidualBlock, self).__init__(**kwargs)

    def _build_layer(self, layer):
        """Helper function for building layer
        Args:
            layer: Appends layer to internal layer list and builds it based on the current output
                   shape of ResidualBlocK. Updates current output shape.
        """
        self.layers.append(layer)
        self.layers[-1].build(self.res_output_shape)
        self.res_output_shape = self.layers[-1].compute_output_shape(self.res_output_shape)

    def build(self, input_shape):

        with K.name_scope(self.name):  # name scope used to make sure weights get unique names
            self.layers = []
            self.res_output_shape = input_shape

            for k in range(2):  # dilated conv block.
                name = 'conv1D_{}'.format(k)
                with K.name_scope(name):  # name scope used to make sure weights get unique names
                    conv = Conv1D(
                        filters=self.nb_filters,
                        kernel_size=self.kernel_size,
                        dilation_rate=self.dilation_rate,
                        padding=self.padding,
                        name=name,
                        kernel_initializer=self.kernel_initializer
                    )
                    self._build_layer(conv)

                with K.name_scope('norm_{}'.format(k)):
                    if self.use_batch_norm:
                        bn_layer = BatchNormalization()
                        self.batch_norm_layers.append(bn_layer)
                        self._build_layer(bn_layer)
                    elif self.use_layer_norm:
                        ln_layer = LayerNormalization()
                        self.layer_norm_layers.append(ln_layer)
                        self._build_layer(ln_layer)

                with K.name_scope('act_and_dropout_{}'.format(k)):
                    self._build_layer(Activation(self.activation, name='Act_Conv1D_{}'.format(k)))
                    self._build_layer(SpatialDropout1D(rate=self.dropout_rate, name='SDropout_{}'.format(k)))

            if self.nb_filters != input_shape[-1]:
                # 1x1 conv to match the shapes (channel dimension).
                name = 'matching_conv1D'
                with K.name_scope(name):
                    # make and build this layer separately because it directly uses input_shape.
                    # 1x1 conv.
                    self.shape_match_conv = Conv1D(
                        filters=self.nb_filters,
                        kernel_size=1,
                        padding='same',
                        name=name,
                        kernel_initializer=self.kernel_initializer
                    )
            else:
                name = 'matching_identity'
                self.shape_match_conv = Lambda(lambda x: x, name=name)

            with K.name_scope(name):
                self.shape_match_conv.build(input_shape)
                self.res_output_shape = self.shape_match_conv.compute_output_shape(input_shape)

            self._build_layer(Activation(self.activation, name='Act_Conv_Blocks'))
            self.final_activation = Activation(self.activation, name='Act_Res_Block')
            self.final_activation.build(self.res_output_shape)  # probably isn't necessary

            # this is done to force Keras to add the layers in the list to self._layers
            for layer in self.layers:
                self.__setattr__(layer.name, layer)
            self.__setattr__(self.shape_match_conv.name, self.shape_match_conv)
            self.__setattr__(self.final_activation.name, self.final_activation)

            super(ResidualBlock, self).build(input_shape)  # done to make sure self.built is set True

    def call(self, inputs, training=None, **kwargs):
        """
        Returns: A tuple where the first element is the residual model tensor, and the second
                 is the skip connection tensor.
        """
        # https://arxiv.org/pdf/1803.01271.pdf  page 4, Figure 1 (b).
        # x1: Dilated Conv -> Norm -> Dropout (x2).
        # x2: Residual (1x1 matching conv - optional).
        # Output: x1 + x2.
        # x1 -> connected to skip connections.
        # x1 + x2 -> connected to the next block.
        #       input
        #     x1      x2
        #   conv1D    1x1 Conv1D (optional)
        #    ...
        #   conv1D
        #    ...
        #       x1 + x2
        x1 = inputs
        for layer in self.layers:
            training_flag = 'training' in dict(inspect.signature(layer.call).parameters)
            x1 = layer(x1, training=training) if training_flag else layer(x1)
        x2 = self.shape_match_conv(inputs)
        x1_x2 = self.final_activation(layers.add([x2, x1], name='Add_Res'))
        return [x1_x2, x1]

    def compute_output_shape(self, input_shape):
        return [self.res_output_shape, self.res_output_shape]


# @register_keras_serializable()
class TCN(Layer):
    """Creates a TCN layer.

        Input shape:
            A 3D tensor with shape (batch_size, timesteps, input_dim).

        Args:
            nb_filters: The number of filters to use in the convolutional layers. Can be a list.
            kernel_size: The size of the kernel to use in each convolutional layer.
            dilations: The list of the dilations. Example is: [1, 2, 4, 8, 16, 32, 64].
            nb_stacks : The number of stacks of residual blocks to use.
            padding: The padding to use in the convolutional layers, 'causal' or 'same'.
            use_skip_connections: Boolean. If we want to add skip connections from input to each residual blocK.
            return_sequences: Boolean. Whether to return the last output in the output sequence, or the full sequence.
            activation: The activation used in the residual blocks o = Activation(x + F(x)).
            dropout_rate: Float between 0 and 1. Fraction of the input units to drop.
            kernel_initializer: Initializer for the kernel weights matrix (Conv1D).
            use_batch_norm: Whether to use batch normalization in the residual layers or not.
            use_layer_norm: Whether to use layer normalization in the residual layers or not.
            go_backwards: Boolean (default False). If True, process the input sequence backwards and
            return the reversed sequence.
            return_state: Boolean. Whether to return the last state in addition to the output. Default: False.
            kwargs: Any other arguments for configuring parent class Layer. For example "name=str", Name of the model.
                    Use unique names when using multiple TCN.
        Returns:
            A TCN layer.
        """

    def __init__(self,
                 nb_filters=64,
                 kernel_size=3,
                 nb_stacks=1,
                 dilations=(1, 2, 4, 8, 16, 32),
                 padding='causal',
                 use_skip_connections=True,
                 dropout_rate=0.0,
                 return_sequences=False,
                 activation='relu',
                 kernel_initializer='he_normal',
                 use_batch_norm=False,
                 use_layer_norm=False,
                 go_backwards=False,
                 return_state=False,
                 **kwargs):
        self.stateful = False  # TCN are not stateful. Keras requires this parameter.
        self.return_sequences = return_sequences
        self.dropout_rate = dropout_rate
        self.use_skip_connections = use_skip_connections
        self.dilations = dilations
        self.nb_stacks = nb_stacks
        self.kernel_size = kernel_size
        self.nb_filters = nb_filters
        self.activation_name = activation
        self.padding = padding
        self.kernel_initializer = kernel_initializer
        self.use_batch_norm = use_batch_norm
        self.use_layer_norm = use_layer_norm
        self.go_backwards = go_backwards
        self.return_state = return_state
        self.skip_connections = []
        self.residual_blocks = []
        self.layers_outputs = []
        self.build_output_shape = None
        self.slicer_layer = None  # in case return_sequence=False
        self.output_slice_index = None  # in case return_sequence=False
        self.padding_same_and_time_dim_unknown = False  # edge case if padding='same' and time_dim = None

        if self.use_batch_norm + self.use_layer_norm > 1:
            raise ValueError('Only one normalization can be specified at once.')

        if isinstance(self.nb_filters, list):
            assert len(self.nb_filters) == len(self.dilations)
            if len(set(self.nb_filters)) > 1 and self.use_skip_connections:
                raise ValueError('Skip connections are not compatible '
                                 'with a list of filters, unless they are all equal.')

        if padding != 'causal' and padding != 'same':
            raise ValueError("Only 'causal' or 'same' padding are compatible for this layer.")

        # initialize parent class
        super(TCN, self).__init__(**kwargs)

    @property
    def receptive_field(self):
        return 1 + 2 * (self.kernel_size - 1) * self.nb_stacks * sum(self.dilations)

    def tolist(self, shape):
        # noinspection PyBroadException
        try:
            return shape.as_list()
        except Exception:
            return list(shape)

    def build(self, input_shape):

        # member to hold current output shape of the layer for building purposes
        self.build_output_shape = input_shape

        # list to hold all the member ResidualBlocks
        self.residual_blocks = []
        total_num_blocks = self.nb_stacks * len(self.dilations)
        if not self.use_skip_connections:
            total_num_blocks += 1  # cheap way to do a false case for below

        for s in range(self.nb_stacks):
            for i, d in enumerate(self.dilations):
                res_block_filters = self.nb_filters[i] if isinstance(self.nb_filters, list) else self.nb_filters
                self.residual_blocks.append(ResidualBlock(dilation_rate=d,
                                                          nb_filters=res_block_filters,
                                                          kernel_size=self.kernel_size,
                                                          padding=self.padding,
                                                          activation=self.activation_name,
                                                          dropout_rate=self.dropout_rate,
                                                          use_batch_norm=self.use_batch_norm,
                                                          use_layer_norm=self.use_layer_norm,
                                                          kernel_initializer=self.kernel_initializer,
                                                          name='residual_block_{}'.format(len(self.residual_blocks))))
                # build newest residual block
                self.residual_blocks[-1].build(self.build_output_shape)
                self.build_output_shape = self.residual_blocks[-1].res_output_shape

        # this is done to force keras to add the layers in the list to self._layers
        for layer in self.residual_blocks:
            self.__setattr__(layer.name, layer)

        self.output_slice_index = None
        if self.padding == 'same':
            time = self.tolist(self.build_output_shape)[1]
            if time is not None:  # if time dimension is defined. e.g. shape = (bs, 500, input_dim).
                self.output_slice_index = int(self.tolist(self.build_output_shape)[1] / 2)
            else:
                # It will known at call time. c.f. self.call.
                self.padding_same_and_time_dim_unknown = True

        else:
            self.output_slice_index = -1  # causal case.
        self.slicer_layer = Lambda(lambda tt: tt[:, self.output_slice_index, :], name='Slice_Output')
        self.slicer_layer.build(self.tolist(self.build_output_shape))

    def compute_output_shape(self, input_shape):
        """
        Overridden in case keras uses it somewhere... no idea. Just trying to avoid future errors.
        """
        if not self.built:
            self.build(input_shape)
        if not self.return_sequences:
            batch_size = self.build_output_shape[0]
            batch_size = batch_size.value if hasattr(batch_size, 'value') else batch_size
            nb_filters = self.build_output_shape[-1]
            return [batch_size, nb_filters]
        else:
            # Compatibility tensorflow 1.x
            return [v.value if hasattr(v, 'value') else v for v in self.build_output_shape]

    def call(self, inputs, training=None, **kwargs):
        x = inputs

        if self.go_backwards:
            # reverse x in the time axis
            x = tf.reverse(x, axis=[1])

        self.layers_outputs = [x]
        self.skip_connections = []
        for res_block in self.residual_blocks:
            try:
                x, skip_out = res_block(x, training=training)
            except TypeError:  # compatibility with tensorflow 1.x
                x, skip_out = res_block(K.cast(x, 'float32'), training=training)
            self.skip_connections.append(skip_out)
            self.layers_outputs.append(x)

        if self.use_skip_connections:
            if len(self.skip_connections) > 1:
                # Keras: A merge layer should be called on a list of at least 2 inputs. Got 1 input.
                x = layers.add(self.skip_connections, name='Add_Skip_Connections')
            else:
                x = self.skip_connections[0]
            self.layers_outputs.append(x)

        if not self.return_sequences:
            # case: time dimension is unknown. e.g. (bs, None, input_dim).
            if self.padding_same_and_time_dim_unknown:
                self.output_slice_index = K.shape(self.layers_outputs[-1])[1] // 2
            x = self.slicer_layer(x)
            self.layers_outputs.append(x)
        return x

    def get_config(self):
        """
        Returns the config of a the layer. This is used for saving and loading from a model
        :return: python dictionary with specs to rebuild layer
        """
        config = super(TCN, self).get_config()
        config['nb_filters'] = self.nb_filters
        config['kernel_size'] = self.kernel_size
        config['nb_stacks'] = self.nb_stacks
        config['dilations'] = self.dilations
        config['padding'] = self.padding
        config['use_skip_connections'] = self.use_skip_connections
        config['dropout_rate'] = self.dropout_rate
        config['return_sequences'] = self.return_sequences
        config['activation'] = self.activation_name
        config['use_batch_norm'] = self.use_batch_norm
        config['use_layer_norm'] = self.use_layer_norm
        config['kernel_initializer'] = self.kernel_initializer
        config['go_backwards'] = self.go_backwards
        config['return_state'] = self.return_state
        return config


def compiled_tcns(num_feat,  # type: int
                 num_classes,  # type: int
                 nb_filters,  # type: int
                 kernel_size,  # type: int
                 dilations,  # type: List[int]
                 nb_stacks,  # type: int
                 max_len,  # type: int
                 output_len=1,  # type: int
                 padding='causal',  # type: str
                 use_skip_connections=False,  # type: bool
                 return_sequences=True,
                 regression=False,  # type: bool
                 dropout_rate=0.05,  # type: float
                 name='tcn',  # type: str,
                 kernel_initializer='he_normal',  # type: str,
                 activation='relu',  # type:str,
                 opt='adam',
                 lr=0.002,
                 use_batch_norm=False,
                 use_layer_norm=False):
    # type: (...) -> Model
    """Creates a compiled TCN model for a given task (i.e. regression or classification).
    Classification uses a sparse categorical loss. Please input class ids and not one-hot encodings.

    Args:
        num_feat: The number of features of your input, i.e. the last dimension of: (batch_size, timesteps, input_dim).
        num_classes: The size of the final dense layer, how many classes we are predicting.
        nb_filters: The number of filters to use in the convolutional layers.
        kernel_size: The size of the kernel to use in each convolutional layer.
        dilations: The list of the dilations. Example is: [1, 2, 4, 8, 16, 32, 64].
        nb_stacks : The number of stacks of residual blocks to use.
        max_len: The maximum sequence length, use None if the sequence length is dynamic.
        padding: The padding to use in the convolutional layers.
        use_skip_connections: Boolean. If we want to add skip connections from input to each residual blocK.
        return_sequences: Boolean. Whether to return the last output in the output sequence, or the full sequence.
        regression: Whether the output should be continuous or discrete.
        dropout_rate: Float between 0 and 1. Fraction of the input units to drop.
        activation: The activation used in the residual blocks o = Activation(x + F(x)).
        name: Name of the model. Useful when having multiple TCN.
        kernel_initializer: Initializer for the kernel weights matrix (Conv1D).
        opt: Optimizer name.
        lr: Learning rate.
        use_batch_norm: Whether to use batch normalization in the residual layers or not.
        use_layer_norm: Whether to use layer normalization in the residual layers or not.
    Returns:
        A compiled keras TCN.
    """

    dilations = adjust_dilations(dilations)

    input_layer = Input(shape=(max_len, num_feat))
    if USE_ATTENTION_MECHANISM:
        x = x = TCN(
    nb_filters=nb_filters,
    kernel_size=kernel_size,
    nb_stacks=nb_stacks,
    dilations=dilations,
    padding=padding,
    use_skip_connections=use_skip_connections,
    dropout_rate=dropout_rate,
    return_sequences=True,  # <-- correct
    activation=activation,
    kernel_initializer=kernel_initializer,
    use_batch_norm=use_batch_norm,
    use_layer_norm=use_layer_norm,
    name=name
)(input_layer)
        score = Dense(1)(x)                     # (batch, time, 1)
        weights = tf.keras.layers.Softmax(axis=1)(score)
        x = tf.keras.layers.Multiply()([x, weights])
        x = tf.keras.layers.Lambda(lambda t: tf.reduce_sum(t, axis=1))(x)
    else:
        x = TCN(nb_filters, kernel_size, nb_stacks, dilations, padding,
                use_skip_connections, dropout_rate, return_sequences,
                activation, kernel_initializer, use_batch_norm, use_layer_norm,
                name=name)(input_layer)

    print('x.shape=', x.shape)

    def get_opt(lr=LR):
        if opt == 'adam':
            return tf.keras.optimizers.Adam(learning_rate=lr, clipnorm=1.)
        elif opt == 'rmsprop':
            return optimizers.RMSprop(learning_rate=lr, clipnorm=1.)
        else:
            raise Exception('Only Adam and RMSProp are available here')

    if not regression:
        # classification
        x = Dense(128, activation='gelu')(x)
        x = Dense(num_classes, activation='softmax')(x)  # ✅ Combine in one layer
        # x = Dense(num_classes)(x)

        output_layer = x
        model = Model(input_layer, output_layer)
        
    else:
        # regression
        x = Dense(num_classes)(x)
        x = Activation('linear')(x)

        output_layer = x
        model = Model(input_layer, output_layer)


    if not regression:
        # loss = tf.keras.losses.SparseCategoricalCrossentropy( )
        # loss=focal_loss_sparse(gamma=2.0, alpha=0.25) 
        # loss= spatial_loss 
        #  loss= SpatialCrossEntropy(alpha=2.0)
        loss=tf.keras.losses.SparseCategoricalCrossentropy( )
        metrics = [
            "sparse_categorical_accuracy",
            tf.keras.metrics.SparseTopKCategoricalAccuracy(k=3),
            tf.keras.metrics.SparseTopKCategoricalAccuracy(k=5)
        ]
    else:
        loss = "mean_squared_error"
        metrics = ["mae"]

    model.compile(get_opt(), loss=loss, metrics=metrics)
    print('model.x = {}'.format(input_layer.shape))
    print('model.y = {}'.format(output_layer.shape))
    tcn_full_summary(model)
    return model


def tcn_full_summary(model: Model, expand_residual_blocks=True):
    import tensorflow as tf
    # 2.6.0-rc1, 2.5.0...
    versions = [int(v) for v in tf.__version__.split('-')[0].split('.')]
    if versions[0] <= 2 and versions[1] < 5:
        layers = model._layers.copy()  # store existing layers
        model._layers.clear()  # clear layers

        for i in range(len(layers)):
            if isinstance(layers[i], TCN):
                for layer in layers[i]._layers:
                    if not isinstance(layer, ResidualBlock):
                        if not hasattr(layer, '__iter__'):
                            model._layers.append(layer)
                    else:
                        if expand_residual_blocks:
                            for lyr in layer._layers:
                                if not hasattr(lyr, '__iter__'):
                                    model._layers.append(lyr)
                        else:
                            model._layers.append(layer)
            else:
                model._layers.append(layers[i])

        model.summary()  # print summary

        # restore original layers
        model._layers.clear()
        [model._layers.append(lyr) for lyr in layers]
    else:
        print('WARNING: tcn_full_summary: Compatible with tensorflow 2.5.0 or below.')
        print('Use tensorboard instead. Example in keras-tcn/tasks/tcn_tensorboard.py.')

In [ ]:
import math

# ─────────────────────────────────────────────────────────────────────────────
# 1. COSINE ANNEALING LR SCHEDULER WITH WARMUP
# ─────────────────────────────────────────────────────────────────────────────

class CosineAnnealingWarmupScheduler(tf.keras.callbacks.Callback):
    """
    Cosine Annealing Learning Rate Scheduler with Linear Warmup.
    
    LR Schedule:
        - Warmup: Linear increase from min_lr to max_lr over warmup_epochs
        - Cosine: Smooth decay from max_lr to min_lr following cosine curve
    
    Benefits:
        - Smooth LR transitions prevent training instability
        - Warmup helps with large batch training
        - Cosine decay often outperforms step decay
    """
    
    def __init__(self, max_lr=1e-3, min_lr=1e-6, warmup_epochs=5, total_epochs=50, 
                 steps_per_epoch=None, verbose=1):
        super().__init__()
        self.max_lr = max_lr
        self.min_lr = min_lr
        self.warmup_epochs = warmup_epochs
        self.total_epochs = total_epochs
        self.steps_per_epoch = steps_per_epoch
        self.verbose = verbose
        self.current_lr = min_lr
        self.history = {'lr': []}
        
    def on_epoch_begin(self, epoch, logs=None):
        if epoch < self.warmup_epochs:
            # Linear warmup
            self.current_lr = self.min_lr + (self.max_lr - self.min_lr) * (epoch / self.warmup_epochs)
        else:
            # Cosine annealing
            progress = (epoch - self.warmup_epochs) / (self.total_epochs - self.warmup_epochs)
            self.current_lr = self.min_lr + 0.5 * (self.max_lr - self.min_lr) * (1 + math.cos(math.pi * progress))
        
        # Set the learning rate (compatible with Keras 3.x and older versions)
        try:
            # Keras 3.x / TF 2.16+
            self.model.optimizer.learning_rate.assign(self.current_lr)
        except (AttributeError, TypeError):
            # Fallback for older Keras versions
            tf.keras.backend.set_value(self.model.optimizer.learning_rate, self.current_lr)
        
        self.history['lr'].append(self.current_lr)
        
        if self.verbose:
            print(f"\n📈 Epoch {epoch+1}: LR = {self.current_lr:.2e}")
    
    def on_train_end(self, logs=None):
        if self.verbose:
            print(f"\n✅ Cosine LR Schedule complete. LR range: {self.max_lr:.2e} → {self.min_lr:.2e}")


class WarmupCosineDecaySchedule(tf.keras.optimizers.schedules.LearningRateSchedule):
    """
    Learning Rate Schedule for use directly with optimizer.
    Combines linear warmup with cosine decay.
    """
    
    def __init__(self, max_lr=1e-3, min_lr=1e-6, warmup_steps=1000, decay_steps=10000):
        super().__init__()
        self.max_lr = max_lr
        self.min_lr = min_lr
        self.warmup_steps = warmup_steps
        self.decay_steps = decay_steps
        
    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        
        # Warmup phase
        warmup_lr = self.min_lr + (self.max_lr - self.min_lr) * (step / self.warmup_steps)
        
        # Cosine decay phase
        decay_step = step - self.warmup_steps
        decay_progress = decay_step / self.decay_steps
        cosine_lr = self.min_lr + 0.5 * (self.max_lr - self.min_lr) * (1 + tf.cos(math.pi * decay_progress))
        
        # Choose based on current step
        return tf.where(step < self.warmup_steps, warmup_lr, cosine_lr)
    
    def get_config(self):
        return {
            'max_lr': self.max_lr,
            'min_lr': self.min_lr,
            'warmup_steps': self.warmup_steps,
            'decay_steps': self.decay_steps
        }

class FP16SafeSparseTopKAccuracy(tf.keras.metrics.SparseTopKCategoricalAccuracy):
    """
    TopK accuracy metric that casts predictions to float32 for FP16 compatibility.
    Fixes: TypeError: Input 'predictions' of 'InTopKV2' Op has type float16
    """
    def update_state(self, y_true, y_pred, sample_weight=None):
        # Cast predictions to float32 for TopK computation
        y_pred = tf.cast(y_pred, tf.float32)
        return super().update_state(y_true, y_pred, sample_weight)




##  Unified training function for Keras backend.

In [ ]:
class GarbageCollectorCallback(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        gc.collect()
        tf.keras.backend.clear_session() # Resets the global state (useful for heavy models)

In [ ]:
from sklearn.metrics import confusion_matrix
import tensorflow as tf

# ───────────────────────────────────────────────────────────────
# Confusion Matrix Callback for generator/dataset
# ───────────────────────────────────────────────────────────────
class ConfusionMatrixCallback(tf.keras.callbacks.Callback):
    def __init__(self, val_loader, num_classes=27, interval=5):
        """
        val_loader: Keras generator or tf.data.Dataset for validation
        num_classes: total number of zones
        interval: compute/plot every `interval` epochs
        """
        super().__init__()
        self.val_loader = val_loader
        self.num_classes = num_classes
        self.interval = interval

    def on_epoch_end(self, epoch, logs=None):
        if (epoch + 1) % self.interval != 0:
            return

        all_preds = []
        all_labels = []

        # Iterate through validation loader
        for batch in self.val_loader:
            x_batch, y_batch = batch
            preds_batch = self.model.predict(x_batch, verbose=0)
            preds_batch = np.argmax(preds_batch, axis=1)

            all_preds.extend(preds_batch)
            # Convert tensor to numpy if needed
            all_labels.extend(y_batch.numpy() if hasattr(y_batch, 'numpy') else y_batch)

        # Compute confusion matrix
        cm = confusion_matrix(all_labels, all_preds, labels=np.arange(self.num_classes))
        # Normalize rows for percentage
        cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        cm_norm = np.nan_to_num(cm_norm)  # handle divisions by zero

        print(f"\nConfusion Matrix (percentages) at Epoch {epoch+1}:")
        print(np.round(cm_norm * 100, 1))

        # Plot heatmap
        plt.figure(figsize=(12, 10))
        sns.heatmap(cm_norm * 100, annot=True, fmt=".1f", cmap="Blues")
        plt.xlabel("Predicted Zone")
        plt.ylabel("True Zone")
        plt.title(f"Normalized Confusion Matrix - Epoch {epoch+1}")
        plt.show()

In [ ]:
def train_model(model, train_loader, val_loader, steps_per_epoch ,val_steps,epochs=EPOCHS, lr=LR,patience=PATIENCE):


    if BACKEND == "keras":
        
        print("═" * 70)
        print("🚀 ENHANCED KERAS TRAINING")
        print("═" * 70)
        print(f"   • Mode: {'Regression' if CO_ORDINATES else 'Classification'}")
        print(f"   • LR Schedule: {'Cosine Annealing' if USE_COSINE_LR else 'ReduceLROnPlateau'}")
        if USE_COSINE_LR:
            print(f"   • LR Range: {MAX_LR:.2e} → {MIN_LR:.2e}")
            print(f"   • Warmup: {WARMUP_EPOCHS} epochs")
        if not CO_ORDINATES and LABEL_SMOOTHING > 0:
            print(f"   • Label Smoothing: {LABEL_SMOOTHING}")
        
        # ─────────────────────────────────────────────────────────────────
        # Recompile with label smoothing for classification
        # ─────────────────────────────────────────────────────────────────
        if not CO_ORDINATES :
            
            y_train = train_df['zone'].values
            model.compile(
                optimizer=tf.keras.optimizers.Adam(
                    learning_rate=MIN_LR if USE_COSINE_LR else lr,
                    # weight_decay=1e-4,
                    clipnorm=.5
                ),
                # loss=label_smoothing_sparse_categorical_crossentropy(LABEL_SMOOTHING),
                     # loss=SpatialCrossEntropy(alpha=2.0),
                    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
                    # loss = "sparse_categorical_crossentropy",
                # loss=get_spatio_temporal_focal_loss(class_alphas=alpha_from_class_counts(y_train=y_train),gamma=2.0),
                metrics=[
                    "sparse_categorical_accuracy",
                    FP16SafeSparseTopKAccuracy(k=3, name='top3_acc'),
                    FP16SafeSparseTopKAccuracy(k=5, name='top5_acc')
                ]
            )
        
        # ─────────────────────────────────────────────────────────────────
        # Build callbacks list
        # ─────────────────────────────────────────────────────────────────
        callbacks = [GarbageCollectorCallback()]
        
        # Choose LR scheduler
        if USE_COSINE_LR and not CO_ORDINATES:
            # Cosine Annealing with Warmup for classification
            callbacks.append(
                CosineAnnealingWarmupScheduler(
                    max_lr=MAX_LR,
                    min_lr=MIN_LR,
                    warmup_epochs=WARMUP_EPOCHS,
                    total_epochs=epochs,
                    verbose=1
                )
            )
        elif USE_COSINE_LR and  CO_ORDINATES:
            callbacks.append(
                CosineAnnealingWarmupScheduler(
                    max_lr=MAX_LR,
                    min_lr=MIN_LR,
                    warmup_epochs=WARMUP_EPOCHS,
                    total_epochs=epochs,
                    verbose=1
                )
            )
        elif not USE_COSINE_LR and  CO_ORDINATES:
            # ReduceLROnPlateau for regression or when cosine is disabled
            callbacks.append(
                tf.keras.callbacks.ReduceLROnPlateau(
                    monitor='val_mae' if CO_ORDINATES else 'val_loss',
                    factor=0.5,
                    patience=3,
                    min_lr=MIN_LR,
                    mode='min' if CO_ORDINATES else 'max',
                    verbose=1
                )
            )
        
        # Early stopping
        callbacks.append(
            tf.keras.callbacks.EarlyStopping(
                monitor='val_mae' if CO_ORDINATES else 'val_loss',
                patience=patience,
                restore_best_weights=True,
                mode='min' if CO_ORDINATES else 'max',
                verbose=1
            )
        )
        
        # Model checkpoint
        checkpoint_path = 'best_model_mae.keras' if CO_ORDINATES else 'best_classification_model.keras'
        callbacks.append(
            tf.keras.callbacks.ModelCheckpoint(
                checkpoint_path,
                monitor='val_mae' if CO_ORDINATES else 'val_loss',
                save_best_only=True,
                mode='min' if CO_ORDINATES else 'max',
                verbose=1
            )
        )
        callbacks.append(
                tf.keras.callbacks.CSVLogger(
                   'training_history.csv', 
                    separator=',', 
                    append=False # Set to True if you are resuming a training run
                )
            )
        
        # ─────────────────────────────────────────────────────────────────
        # Class weights for imbalanced classification
        # ─────────────────────────────────────────────────────────────────
            
      
        
        
        # ─────────────────────────────────────────────────────────────────
        # Train
        # ─────────────────────────────────────────────────────────────────
        print(f"\n🏃 Starting training for {epochs} epochs...")
        
        history = model.fit(
            train_loader,
            validation_data=val_loader,
            epochs=epochs,
            callbacks=callbacks,
            verbose=1,
           
            steps_per_epoch=steps_per_epoch,
            validation_steps=val_steps
        )
        
        # ─────────────────────────────────────────────────────────────────
        # Report results
        # ─────────────────────────────────────────────────────────────────
        print("\n" + "═" * 70)
        print("✅ TRAINING COMPLETE")
        print("═" * 70)
        
        if CO_ORDINATES:
            best_mae = min(history.history.get('val_mae', [float('inf')]))
            print(f"   • Best Val MAE: {best_mae:.4f}")
        else:
            best_acc = max(history.history.get('val_sparse_categorical_accuracy', [0]))
            best_top3 = max(history.history.get('val_top3_acc', [0]))
            best_top5 = max(history.history.get('val_top5_acc', [0]))
            print(f"   • Best Val Accuracy: {best_acc*100:.2f}%")
            print(f"   • Best Top-3 Accuracy: {best_top3*100:.2f}%")
            print(f"   • Best Top-5 Accuracy: {best_top5*100:.2f}%")
        
        print(f"   • Model saved to: {checkpoint_path}")

        return model, history

## Check and Loading Dataset 

In [ ]:
def load_preprocessed():
    """Load preprocessed sequences for Keras backend"""

    print("\n" + "="*80)
    print("📂 LOADING KERAS PREPROCESSED DATA")
    print("="*80)
    files_dir={
    "train_df":'preprocessed_data/train_df.pkl',
    "val_df":'preprocessed_data/val_df.pkl',
    "test_df":'preprocessed_data/test_df.pkl',    
}
    try:

        train_df = pd.read_pickle(files_dir['train_df'])
        val_df = pd.read_pickle(files_dir['val_df'])
        test_df = pd.read_pickle(files_dir['test_df'])
        
       
        print(f"✓ Loaded dataframes:")
        print(f"   Train: {len(train_df)} rows, Val: {len(val_df)} rows, Test: {len(test_df)} rows")
        return {
            'train_df': train_df,
            'val_df': val_df,
            'test_df': test_df,
            'df': pd.concat([train_df, val_df, test_df], ignore_index=True)
        }

    except Exception as e:
        print(f"\n❌ Error loading Keras preprocessed data: {e}")
        return None
def load_dataset_with_memory_optimization(file_path):
    chunk_size = 500_000  # Adjust based on your RAM
    chunks = []
    total_rows = 0
    
    try:
        # Load WITHOUT forcing dtypes - let pandas auto-detect
        for i, chunk in enumerate(pd.read_csv(file_path, chunksize=chunk_size, low_memory=False)):
            
            # Drop any timestamp string columns that aren't needed
            cols_to_drop = []
            for col in chunk.columns:
                # Check if column contains time strings like '0:11.20'
                if chunk[col].dtype == 'object':
                    sample_val = str(chunk[col].dropna().iloc[0]) if len(chunk[col].dropna()) > 0 else ""
                    if ':' in sample_val and col not in ['match_id', 'player_id']:
                        cols_to_drop.append(col)
                        print(f"  Dropping timestamp column: {col}")
            
            if cols_to_drop:
                chunk = chunk.drop(columns=cols_to_drop)
            
            # Downcast numeric columns AFTER loading to save memory
            for col in chunk.select_dtypes(include=['float64']).columns:
                chunk[col] = chunk[col].astype('float32')
            for col in chunk.select_dtypes(include=['int64']).columns:
                chunk[col] = pd.to_numeric(chunk[col], downcast='integer')
            
            chunks.append(chunk)
            total_rows += len(chunk)
            
            # Memory check
            mem_percent = psutil.virtual_memory().percent
            print(f"  Chunk {i+1}: {len(chunk):,} rows | Total: {total_rows:,} | RAM: {mem_percent:.1f}%", end='\r')
            
            # Stop if memory is getting too high
            if mem_percent > 95:
                print(f"\n⚠️  Memory threshold reached at {total_rows:,} rows. Using partial data.")
                break
        
        print(f"\n✓ Loaded {len(chunks)} chunks")
        
        # Concatenate chunks
        df = pd.concat(chunks, ignore_index=True)
        del chunks
        gc.collect()
        
        print(f"Dataset loaded successfully! Shape: {df.shape}")
        print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
        if 'timestamp_seconds' in df.columns:
            print(f"Time range: {df['timestamp_seconds'].min():.2f} to {df['timestamp_seconds'].max():.2f} seconds")

        
    except MemoryError:
        print("❌ MemoryError during loading. Try reducing chunk_size or using fewer matches.")
        raise
    
    # Display basic info
    print(f"Columns: {list(df.columns)}")
    print(f"Unique matches: {df['match_id'].nunique()}")
    print(f"Unique players: {df['player_id'].nunique()}")
    
 



In [ ]:
loaded_data = load_preprocessed()
if loaded_data is not None:
	train_df = loaded_data['train_df']
	val_df = loaded_data['val_df']
	test_df = loaded_data['test_df']
	df = loaded_data['df']
	SKIP_DATA_LOADING=True

else:
    SKIP_DATA_LOADING=False
    print("❌ Failed to load preprocessed sequences. Please check your data files.")


In [ ]:
if not SKIP_DATA_LOADING:
    df =load_dataset_with_memory_optimization(file_path)
   
    print("\nAdding velocity ,enhanced & contextual features...")
    df = add_velocity_features(df)
    gc.collect()

    df = add_contextual_features(df)
    gc.collect()      
        
    print("Checking feature columns...")
    available_features = [col for col in FEATURE_COLS if col in df.columns]
    missing_features = [col for col in FEATURE_COLS if col not in df.columns]
        
    if missing_features:
            print(f"Warning: Missing columns {missing_features}")
            FEATURE_COLS = available_features
            print(f"Updated feature list to {len(FEATURE_COLS)} available features")
        
        # Handle missing values
    print("Handling missing values...")
    initial_rows = len(df)
    df = df.dropna(subset=FEATURE_COLS).reset_index(drop=True)
    final_rows = len(df)
    print(f"Removed {initial_rows - final_rows} rows with missing values")
        
        # Downcast numeric columns to save more memory
    print("Optimizing memory usage...")
    for col in df.select_dtypes(include=['float64']).columns:
            df[col] = df[col].astype('float32')
    for col in df.select_dtypes(include=['int64']).columns:
            df[col] = pd.to_numeric(df[col], downcast='integer')
        
    gc.collect()
        
    final_memory = df.memory_usage(deep=True).sum() / 1024**2
    print(f"Final memory usage: {final_memory:.1f} MB")
    print(f"Dataset processed successfully! Final shape: {df.shape}")

else:
    print("✅ Skipping data loading - using preprocessed sequences")
    print(f"   Dataset shape: {df.shape}")
    print(f"   Features already computed: {len(FEATURE_COLS)}")

In [ ]:
from Data_preprocessing import xy_to_zone_vectorized
if 'zone' not in df.columns:
    print("Computing 'zone' column for class weights calculation...")
    df['zone'] = xy_to_zone_vectorized(
        df['x_normalized'].values,
        df['y_normalized'].values,
        n_rows=N_ROWS,
        n_cols=N_COLS
    )
gc.collect()

In [ ]:
from Data_preprocessing import split_by_match_df
# print("Splitting data by matches...")
train_df, val_df, test_df = split_by_match_df(df, fixed_split=False)


scaler = StandardScaler()
print("Fitting scaler on training data...")

scaler.fit(train_df[FEATURE_COLS].values)

print("✓ Scaler fitted on training data")


In [ ]:
gpus = tf.config.list_physical_devices('GPU')
print(f"\n🔍 GPU Detection:")
print(f"Available GPUs: {len(gpus)}")
for gpu in gpus:
        print(f"  - {gpu}")
# Create distribution strategy for multi-GPU
if len(gpus) > 1:
        print(f"\n🚀 Using MirroredStrategy for {len(gpus)} GPUs")
        strategy = tf.distribute.MirroredStrategy()
else:
        print("\n⚠️  Single/No GPU detected, using default strategy")
        strategy = tf.distribute.get_strategy()

print(f"Number of devices in strategy: {strategy.num_replicas_in_sync}")

    # Adjust batch size for multi-GPU (IMPORTANT!)
GLOBAL_BATCH_SIZE = BATCH_SIZE * strategy.num_replicas_in_sync
print(f"\nBatch size configuration:")
print(f"  • Per-GPU batch size: {BATCH_SIZE}")
print(f"  • Global batch size: {GLOBAL_BATCH_SIZE}")


In [ ]:

def make_tf_dataset(df, batch_size,N_ROWS,N_COLS,scaler, SEQ_LEN,HORIZON_FRAMES,FEATURE_COLS,shuffle=False, coordinate_targets=False, combined=False, repeat=False):

    min_frames_needed = SEQ_LEN + HORIZON_FRAMES
    df = df.groupby(["match_id", "player_id"]).filter(
        lambda x: len(x) >= min_frames_needed
    )
    """Optimized tf.data pipeline with parallel processing and known cardinality"""
    if combined:
        output_types = (
            tf.float32,
            {"zone": tf.int32, "co_ordinate": tf.float32},
        )
        output_shapes = (
            (SEQ_LEN, len(FEATURE_COLS)),
            {"zone": (), "co_ordinate": (2,)},
        )
    else:
        output_types = (
            tf.float32,
            tf.float32 if coordinate_targets else tf.int32,
        )
        output_shapes = (
            (SEQ_LEN, len(FEATURE_COLS)),
            (2,) if coordinate_targets else (),
        )

    # 🔧 PRE-CALCULATE: Count sequences for progress bar
    total_sequences = 0
    for _, player_df in df.groupby(["match_id", "player_id"]):
        total_sequences += max(0, len(player_df) - SEQ_LEN - HORIZON_FRAMES + 1)

    expected_batches = total_sequences // batch_size
    print(f"   Dataset: {total_sequences:,} sequences → {expected_batches} batches")
    ds = tf.data.Dataset.from_generator(
        lambda: keras_sequence_generator_with_scaler(
           df=df,
           feature_cols=FEATURE_COLS,coord_cols=['x_normalized', 'y_normalized'],
           coordinate_targets=coordinate_targets, combined=combined,
           scaler=scaler,horizon=HORIZON_FRAMES,seq_len=SEQ_LEN,n_rows=N_ROWS,n_cols=N_COLS
        ),
        output_types=output_types,
        output_shapes=output_shapes
    )

    if shuffle:
        ds = ds.shuffle(4096)  # 🔧 INCREASED: Better randomization

    # 🔧 OPTIMIZATION: Batch before cache for memory efficiency
    # ensure labels are always correct shape

    def map_fn(x, y):
        if not combined and coordinate_targets:
            y = tf.reshape(y, [-1])
        return x, y

    ds = ds.map(map_fn, num_parallel_calls=tf.data.AUTOTUNE)
    # 🔧 FIXED: drop_remainder=True prevents 'ran out of data' warnings
    ds = ds.batch(batch_size, drop_remainder=False)
    ds = ds.prefetch(tf.data.AUTOTUNE)

    # Only repeat if requested (e.g. for training)
    if repeat:
        ds = ds.repeat()
        
    return ds,expected_batches

def keras_sequence_generator_with_scaler(df, feature_cols, scaler, seq_len, horizon, coord_cols,
                             coordinate_targets, combined=False, n_rows=None, n_cols=None):
    """
    Generator that yields sequences one by one (or in small batches) instead of storing all.
    """
    for _, player_df in df.groupby(["match_id", "player_id"]):
        feats = player_df[feature_cols].values.astype(np.float32)
        if scaler is not None:
            feats = scaler.transform(feats)
        coords = player_df[list(coord_cols)].values.astype(np.float32)

        if combined:
            # COMBINED: yield both zone and coordinate targets as a dict
            zones = xy_to_zone_vectorized(
                player_df["x_normalized"].values,
                player_df["y_normalized"].values,
                n_rows, n_cols
            )
            for i in range(len(player_df) - seq_len - horizon + 1):
                seq = feats[i:i+seq_len]
                t_fut = i + seq_len - 1 + horizon
                zone = zones[t_fut]
                coord = coords[t_fut]
                yield seq, {"zone": np.int32(zone), "co_ordinate": coord.astype(np.float32)}

        elif coordinate_targets:
            # regression: coordinates only
            for i in range(len(player_df) - seq_len - horizon + 1):
                seq = feats[i:i+seq_len]
                t_curr = i + seq_len - 1
                t_fut = t_curr + horizon
                yield seq, coords[t_fut].astype(np.float32)

        else:
            # classification: zone only
            zones = xy_to_zone_vectorized(
                player_df["x_normalized"].values,
                player_df["y_normalized"].values,
                n_rows, n_cols
            )
            for i in range(len(player_df) - seq_len - horizon + 1):
                seq = feats[i:i+seq_len]
                t_fut = i + seq_len - 1 + horizon
                zone = zones[t_fut]
                yield seq, np.int32(zone), 




In [ ]:
train_loader,train_steps_per_epoch = make_tf_dataset(
            train_df,
         BATCH_SIZE,
            shuffle=True,
            coordinate_targets=CO_ORDINATES,
            combined=COMBINED,
            repeat=True,
               HORIZON_FRAMES=HORIZON_FRAMES ,
               FEATURE_COLS=FEATURE_COLS,
               SEQ_LEN=SEQ_LEN,
               N_ROWS=N_ROWS,
               N_COLS=N_COLS,
               scaler=scaler
        )

 # Val/Test should NOT repeat so evaluation stops naturally
 # Also disable shuffle for evaluation to keep it deterministic
val_loader ,val_steps = make_tf_dataset(val_df,BATCH_SIZE, shuffle=False, coordinate_targets=CO_ORDINATES, combined=COMBINED, repeat=False, HORIZON_FRAMES=HORIZON_FRAMES , N_ROWS=N_ROWS,
               N_COLS=N_COLS,
               FEATURE_COLS=FEATURE_COLS,
               SEQ_LEN=SEQ_LEN, scaler=scaler)
test_loader,test_steps = make_tf_dataset(test_df,BATCH_SIZE, shuffle=True, coordinate_targets=CO_ORDINATES, combined=COMBINED, repeat=False, HORIZON_FRAMES=HORIZON_FRAMES , N_ROWS=N_ROWS,
               N_COLS=N_COLS,
               FEATURE_COLS=FEATURE_COLS,
               SEQ_LEN=SEQ_LEN, scaler=scaler)

tf.keras.backend.clear_session()
gc.collect()
print(f"Datasets created in make tf with scaler. Ready for training.")


## Model Initialisation

In [ ]:
print(train_steps_per_epoch,val_steps,test_steps)

In [ ]:
def make_tcn_with_attention():
    with strategy.scope():

        inputs = tf.keras.Input(shape=(SEQ_LEN, len(FEATURE_COLS)))

        # 1️⃣ Use TCN directly
        x = TCN(
            nb_filters=128,
            kernel_size=5,
            nb_stacks=2,
            dilations=[2,4,8,16,32,64],
            padding='causal',
            use_skip_connections=True,
            dropout_rate=0.0,
            return_sequences=True,   # IMPORTANT
            activation='relu',
            kernel_initializer='he_normal',
            use_batch_norm=False,
            use_layer_norm=True
        )(inputs)

        # x shape: (batch, time, filters)

        # # 2️⃣ Temporal Attention
        # score = tf.keras.layers.Dense(1)(x)  # (batch, time, 1)
        # weights = tf.keras.layers.Softmax(axis=1)(score)

        # x = tf.keras.layers.Multiply()([x, weights])
        # x = tf.keras.layers.Lambda(lambda t: tf.reduce_sum(t, axis=1))(x)

        attn = tf.keras.layers.MultiHeadAttention(
        num_heads=4,
        key_dim=32
    )(x, x)
    
        x = tf.keras.layers.Add()([x, attn])
        x = tf.keras.layers.LayerNormalization()(x)
        
        x = tf.keras.layers.GlobalAveragePooling1D()(x)
        # Now shape: (batch, filters)
        num_zones = N_COLS*N_ROWS
        # 3️⃣ Final classifier
        if not CO_ORDINATES:
            
            x = tf.keras.layers.Dense(128, activation='relu')(x)
            x = tf.keras.layers.Dropout(0.3)(x)
            
            # 🔥 Motion representation
            motion = tf.keras.layers.Dense(64, activation='relu')(x)
            motion = tf.keras.layers.Dense(32, activation='relu')(motion)
            outputs = tf.keras.layers.Dense(num_zones, activation='softmax')(motion)
            loss= 'sparse_categorical_crossentropy'
            metrics = [
            "sparse_categorical_accuracy",
            tf.keras.metrics.SparseTopKCategoricalAccuracy(k=3, name="top3_acc"),
    tf.keras.metrics.SparseTopKCategoricalAccuracy(k=5, name="top5_acc")
        ]
        elif CO_ORDINATES:
            outputs = tf.keras.layers.Dense(2, activation='linear')(x)
            loss = 'mse'
            metrics = ["mae"]
        elif COMBINED:
            zone=tf.keras.layers.Dense(num_zones, activation='softmax')(x)
            co_ordinate = tf.keras.layers.Dense(2, activation='linear', name="coord_output")(x)
            outputs = [zone, co_ordinate]
            loss={
                "zone": tf.keras.losses.SparseCategoricalCrossentropy(),
                "co_ordinate": tf.keras.losses.Huber()  # better than MSE
            }
            loss_weights={
                "zone": 1.0,
                "co_ordinate": 0.5   # start smaller
            }
            metrics={
                "zone": [
                    "sparse_categorical_accuracy",
                  tf.keras.metrics.SparseTopKCategoricalAccuracy(k=3, name="top3_acc"),
    tf.keras.metrics.SparseTopKCategoricalAccuracy(k=5, name="top5_acc")
                ],
                "co_ordinate": [
                    tf.keras.metrics.MeanAbsoluteError()
                ]
            }
            
        model = tf.keras.Model(inputs, outputs)

        model.compile(
            optimizer=tf.keras.optimizers.Adam(LR),
            loss=loss,
            metrics=metrics,
            loss_weights=loss_weights if COMBINED else None
        )
    model.summary()
    return model

In [ ]:
# use to load the best classi 11_1s and regression_12_1s
#make sure to switch CO_ORDINATES to relevant
def make_tcn_with_attention_for_current_best():
    with strategy.scope():

        inputs = tf.keras.Input(shape=(SEQ_LEN, len(FEATURE_COLS)))

        # 1️⃣ Use TCN directly
        x = TCN(
            nb_filters=128,
            kernel_size=5,
            nb_stacks=2,
            dilations=[2,4,8,16,32,64],
            padding='causal',
            use_skip_connections=True,
            dropout_rate=0.0,
            return_sequences=True,   # IMPORTANT
            activation='relu',
            kernel_initializer='he_normal',
            use_batch_norm=False,
            use_layer_norm=True
        )(inputs)

        # x shape: (batch, time, filters)

        # 2️⃣ Temporal Attention
        score = tf.keras.layers.Dense(1)(x)  # (batch, time, 1)
        weights = tf.keras.layers.Softmax(axis=1)(score)

        x = tf.keras.layers.Multiply()([x, weights])
        x = tf.keras.layers.Lambda(
    lambda t: tf.reduce_sum(t, axis=1),
    output_shape=(128,)
)(x)

        # Now shape: (batch, filters)
        num_zones = N_COLS*N_ROWS
        # 3️⃣ Final classifier
        if not CO_ORDINATES:
            outputs = tf.keras.layers.Dense(num_zones, activation='softmax')(x)
            loss= 'sparse_categorical_crossentropy'
            metrics = [
            "sparse_categorical_accuracy",
            tf.keras.metrics.SparseTopKCategoricalAccuracy(k=3, name="top3_acc"),
    tf.keras.metrics.SparseTopKCategoricalAccuracy(k=5, name="top5_acc")
        ]
        elif CO_ORDINATES:
            outputs = tf.keras.layers.Dense(2, activation='linear')(x)
            loss = 'mse'
            metrics = ["mae"]
        elif COMBINED:
            zone=tf.keras.layers.Dense(num_zones, activation='softmax')(x)
            co_ordinate = tf.keras.layers.Dense(2, activation='linear', name="coord_output")(x)
            outputs = [zone, co_ordinate]
            loss={
                "zone": tf.keras.losses.SparseCategoricalCrossentropy(),
                "co_ordinate": tf.keras.losses.Huber()  # better than MSE
            }
            loss_weights={
                "zone": 1.0,
                "co_ordinate": 0.5   # start smaller
            }
            metrics={
                "zone": [
                    "sparse_categorical_accuracy",
                  tf.keras.metrics.SparseTopKCategoricalAccuracy(k=3, name="top3_acc"),
    tf.keras.metrics.SparseTopKCategoricalAccuracy(k=5, name="top5_acc")
                ],
                "co_ordinate": [
                    tf.keras.metrics.MeanAbsoluteError()
                ]
            }
            
        model = tf.keras.Model(inputs, outputs)

        model.compile(
            optimizer=tf.keras.optimizers.Adam(LR),
            loss=loss,
            metrics=metrics
        )
    model.summary()
    return model

In [ ]:
# use to load the best classi 13_3s
# #make sure to switch CO_ORDINATES to relevant
def make_tcn_with_attention_regression_3s():
    with strategy.scope():

        inputs = tf.keras.Input(shape=(SEQ_LEN, len(FEATURE_COLS)))

        # 1️⃣ Use TCN directly
        x = TCN(
            nb_filters=128,
            kernel_size=5,
            nb_stacks=2,
            dilations=[2,4,8,16,32,64],
            padding='causal',
            use_skip_connections=True,
            dropout_rate=0.1,
            return_sequences=True,   # IMPORTANT
            activation='relu',
            kernel_initializer='he_normal',
            use_batch_norm=False,
            use_layer_norm=True
        )(inputs)

        # x shape: (batch, time, filters)

        # 2️⃣ Temporal Attention
        # score = tf.keras.layers.Dense(1)(x)  # (batch, time, 1)
        # weights = tf.keras.layers.Softmax(axis=1)(score)

        # x = tf.keras.layers.Multiply()([x, weights])
        # x = tf.keras.layers.Lambda(lambda t: tf.reduce_sum(t, axis=1))(x)

        attn = tf.keras.layers.MultiHeadAttention(
        num_heads=4,
        key_dim=32
    )(x, x)
    
        x = tf.keras.layers.Add()([x, attn])
        x = tf.keras.layers.LayerNormalization()(x)
        
        x = tf.keras.layers.GlobalAveragePooling1D()(x)
        # Now shape: (batch, filters)
        num_zones = N_COLS*N_ROWS
        # 3️⃣ Final classifier
        if not CO_ORDINATES:
            
            x = tf.keras.layers.Dense(128, activation='relu')(x)
            x = tf.keras.layers.Dropout(0.3)(x)
            
            # 🔥 Motion representation
            motion = tf.keras.layers.Dense(64, activation='relu')(x)
            motion = tf.keras.layers.Dense(32, activation='relu')(motion)
            outputs = tf.keras.layers.Dense(num_zones, activation='softmax')(motion)
            loss= 'sparse_categorical_crossentropy'
            metrics = [
            "sparse_categorical_accuracy",
            tf.keras.metrics.SparseTopKCategoricalAccuracy(k=3, name="top3_acc"),
    tf.keras.metrics.SparseTopKCategoricalAccuracy(k=5, name="top5_acc")
        ]
        elif CO_ORDINATES:
            outputs = tf.keras.layers.Dense(2, activation='linear')(x)
            loss = 'mse'
            metrics = ["mae"]
        elif COMBINED:
            zone=tf.keras.layers.Dense(num_zones, activation='softmax')(x)
            co_ordinate = tf.keras.layers.Dense(2, activation='linear', name="coord_output")(x)
            outputs = [zone, co_ordinate]
            loss={
                "zone": tf.keras.losses.SparseCategoricalCrossentropy(),
                "co_ordinate": tf.keras.losses.Huber()  # better than MSE
            }
            loss_weights={
                "zone": 1.0,
                "co_ordinate": 0.5   # start smaller
            }
            metrics={
                "zone": [
                    "sparse_categorical_accuracy",
                  tf.keras.metrics.SparseTopKCategoricalAccuracy(k=3, name="top3_acc"),
    tf.keras.metrics.SparseTopKCategoricalAccuracy(k=5, name="top5_acc")
                ],
                "co_ordinate": [
                    tf.keras.metrics.MeanAbsoluteError()
                ]
            }
            
        model = tf.keras.Model(inputs, outputs)

        model.compile(
            optimizer=tf.keras.optimizers.Adam(LR),
            loss=loss,
            metrics=metrics
        )
    model.summary()
    return model

In [ ]:
def train_model_new(model, train_loader, val_loader, steps_per_epoch ,val_steps,epochs,patience=PATIENCE):
        
        print("═" * 70)
        print("🚀 ENHANCED KERAS TRAINING")
        print("═" * 70)
        print(f"   • Mode: {'Regression' if CO_ORDINATES else 'Classification'}")
        print(f"   • LR Schedule: {'Cosine Annealing' if USE_COSINE_LR else 'ReduceLROnPlateau'}")
        if USE_COSINE_LR:
            print(f"   • LR Range: {MAX_LR:.2e} → {MIN_LR:.2e}")
            print(f"   • Warmup: {WARMUP_EPOCHS} epochs")
        if not CO_ORDINATES and LABEL_SMOOTHING > 0:
            print(f"   • Label Smoothing: {LABEL_SMOOTHING}")
        
        # ─────────────────────────────────────────────────────────────────
        # Build callbacks list
        # ─────────────────────────────────────────────────────────────────
        callbacks = [GarbageCollectorCallback()]
        if USE_COSINE_LR and not CO_ORDINATES:
            callbacks.append(
                CosineAnnealingWarmupScheduler(
                    max_lr=MAX_LR,
                    min_lr=MIN_LR,
                    warmup_epochs=WARMUP_EPOCHS,
                    total_epochs=epochs,
                    verbose=1
                ) # type: ignore
            )
        else:
            callbacks.append(
                tf.keras.callbacks.ReduceLROnPlateau(
                    monitor='val_mae' if CO_ORDINATES else 'val_sparse_categorical_accuracy',
                    factor=0.5,
                    patience=3,
                    min_lr=MIN_LR,
                    mode='min' if CO_ORDINATES else 'max',
                    verbose=1
                )# type: ignore
            )
    
        # Early stopping
        callbacks.append(
            tf.keras.callbacks.EarlyStopping(
                monitor='val_mae' if CO_ORDINATES else 'val_sparse_categorical_accuracy',
                patience=patience,
                restore_best_weights=True,
                mode='min' if CO_ORDINATES else 'max',
                verbose=1
            )# type: ignore
        )
        
        # Model checkpoint
        checkpoint_path = 'best_model_mae.keras' if CO_ORDINATES else 'best_classification_model.keras'
        callbacks.append(
            tf.keras.callbacks.ModelCheckpoint(
                checkpoint_path,
                monitor='val_mae' if CO_ORDINATES else 'val_sparse_categorical_accuracy',
                save_best_only=True,
                mode='min' if CO_ORDINATES else 'max',
                verbose=1
            )# type: ignore
        )
        callbacks.append(
                tf.keras.callbacks.CSVLogger(
                   'training_history.csv', 
                    separator=',', 
                    append=False # Set to True if you are resuming a training run
                )# type: ignore
            )
        callbacks.append(
            tf.keras.callbacks.TerminateOnNaN(                
            )# type: ignore
        )

        print(f"\n🏃 Starting training for {epochs} epochs...")
        classes = np.unique(train_df['zone'])
        weight= compute_class_weight(class_weight='balanced', classes=classes, y=train_df['zone'])
        class_weight = dict(zip(classes, weight))
        history = model.fit(
            train_loader,
            validation_data=val_loader,
            epochs=epochs,
            callbacks=callbacks,
            verbose=1,
            steps_per_epoch=steps_per_epoch,
            validation_steps=val_steps,
            class_weight=class_weight if not CO_ORDINATES else None
        )
        

        if CO_ORDINATES:
            best_mae = min(history.history.get('val_mae', [float('inf')]))
            print(f"   • Best Val MAE: {best_mae:.4f}")
        else:
            best_acc = max(history.history.get('val_sparse_categorical_accuracy', [0]))
            best_top3 = max(history.history.get('val_top3_acc', [0]))
            best_top5 = max(history.history.get('val_top5_acc', [0]))
            print(f"   • Best Val Accuracy: {best_acc*100:.2f}%")
            print(f"   • Best Top-3 Accuracy: {best_top3*100:.2f}%")
            print(f"   • Best Top-5 Accuracy: {best_top5*100:.2f}%")
        
        print(f"   • Model saved to: {checkpoint_path}")

        return model, history

In [ ]:
test_model_best_new=make_tcn_with_attention_for_current_best()

In [ ]:
test_model_3s= make_tcn_with_attention_regression_3s()

In [ ]:
inference_model=make_tcn_with_attention()

In [ ]:
inference_unit,inference_history= train_modelg_new(
    model=inference_model,
    train_loader=train_loader,
    val_loader=val_loader,
    steps_per_epoch=train_steps_per_epoch,
    val_steps=val_steps,
    epochs=EPOCHS,
    patience=PATIENCE
)

In [ ]:
test_model=make_tcn_with_attention_for_current_best()

In [ ]:
test_model.load_weights('../testings/testings/classi_model_13_3s/best_model_mae.keras')

In [ ]:
test_model.save('best_model_mae.keras')

In [ ]:
from tensorflow import keras
model_save_path= '../testings/testings/classi_model_13_3s/best_model_mae.keras'
#build before importing
custom_objects={'TCN': TCN}

iferesne=keras.models.load_model(model_save_path,custom_objects={'TCN':TCN}, safe_mode=False, compile=False)




In [ ]:
# Initialize enhanced model
model_type = MODEL_TYPE
print(f"Initializing {MODEL_TYPE}")

if model_type == "Keras_tcn":
    with strategy.scope():  # ✅ Everything inside scope
        model = compiled_tcn(
            num_feat=len(FEATURE_COLS),
            num_classes = 2 if CO_ORDINATES else NUM_ZONES,
            nb_filters= 32,  # Was [16,16,32,32,64,64]
            kernel_size=7,
            dilations=[1, 2, 4, 8, 16, 32,64], # increase to add more receptive field
            nb_stacks=1,# increase to add more residual blocks
            max_len=SEQ_LEN,
            padding='causal',
            use_skip_connections=True,
            return_sequences=False,
            dropout_rate=0.2,       # Was 0.5 - too aggressive
            activation='relu',
            use_batch_norm=True,
            use_layer_norm=False,
            lr=LR,
            regression=CO_ORDINATES
        )

        # ✅ Re-compile INSIDE strategy.scope() with XLA
        if CO_ORDINATES:
            optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4, clipnorm=1.0)
            optimizer = mixed_precision.LossScaleOptimizer(optimizer)  # ⚡ FP16 speedup
            model.compile(
                optimizer=optimizer,
                loss=weighted_mse_loss(),
                metrics=['mae', tf.keras.metrics.RootMeanSquaredError(name='rmse')],
                jit_compile=False # 🚀 XLA compilation for speed
            )

        # ✅ Get model info INSIDE scope
        total_params = model.count_params()
        trainable_params = sum([np.prod(w.shape) for w in model.trainable_weights])

        print("Keras model compiled inside strategy.scope()")
        print(f"Using {strategy.num_replicas_in_sync} GPU(s)")

# ═══════════════════════════════════════════════════════════════════════════════
# MDN_TCN: Mixture Density Network for multi-modal probabilistic predictions
# ═══════════════════════════════════════════════════════════════════════════════
if model_type == "MDN_TCN":
    print("═" * 60)
    print("🎯 Initializing MDN_TCN Model")
    print("═" * 60)
    with strategy.scope():    
    # Build MDN model (uses build_tcn_mdn_model from MDN components)
        model = build_tcn_mdn_model(
            seq_len=SEQ_LEN,
            num_features=len(FEATURE_COLS),
            num_mixtures=MDN_NUM_MIXTURES,
            tcn_filters=256,              # Single int required for skip connections
            tcn_kernel_size=7,
            tcn_dilations=[1, 2, 4, 8, 16, 32],
            tcn_stacks=2,
            dropout_rate=0.3,
            use_batch_norm=True,
            dense_units=256,
            learning_rate=MDN_LEARNING_RATE
        )
    
    # MDN uses custom training loop, no compile needed
        total_params = model.count_params()
        trainable_params = sum([np.prod(w.shape) for w in model.trainable_weights])
        
        print(f"MDN model built successfully")
        print(f"  • Mixtures (K): {MDN_NUM_MIXTURES}")
        print(f"  • Output: π({MDN_NUM_MIXTURES}), μ({MDN_NUM_MIXTURES}x2), σ({MDN_NUM_MIXTURES}x2)")
        print("═" * 60)


# Print summary
print(f"\n{'═'*60}")
print(f"📊 MODEL SUMMARY")
print(f"{'═'*60}")
print(f"  • Model type: {model_type}")
print(f"  • Total parameters: {total_params:,}")
print(f"  • Trainable parameters: {trainable_params:,}")
print(f"  • Input features: {len(FEATURE_COLS)}")
if model_type == "MDN_TCN":
    print(f"  • Output type: Probabilistic GMM (K={MDN_NUM_MIXTURES} mixtures)")
else:
    print(f"  • Output size: {2 if CO_ORDINATES else NUM_ZONES}")
print(f"{'═'*60}")

##  MODEL TRAINING

In [ ]:
print("Starting enhanced training...")
print(f"Training configuration:")
print(f"Training for {'MDN (probabilistic)' if MODEL_TYPE == 'MDN_TCN' else ('regression (coordinates)' if CO_ORDINATES else 'classification (zones)')}")
print(f"- Epochs: {EPOCHS}")
print(f"- Learning rate: {MDN_LEARNING_RATE if MODEL_TYPE == 'MDN_TCN' else LR}")
print(f"- Batch size: {BATCH_SIZE}")
print(f"- Patience: {PATIENCE}")
print(f"- Model Type: {MODEL_TYPE}")
print(f"- Backend: {BACKEND}")


# ═══════════════════════════════════════════════════════════════════════════════
# MODEL TRAINING - Conditional based on MODEL_TYPE
# ═══════════════════════════════════════════════════════════════════════════════

if MODEL_TYPE == "MDN_TCN":
    # ─────────────────────────────────────────────────────────────────────────
    # MDN uses custom training loop with NLL loss + cosine LR schedule
    # ─────────────────────────────────────────────────────────────────────────
    print("\n🎯 Using MDN custom training loop...")
    # Calculate steps_per_epoch (same as Keras TCN does)
    with strategy.scope():
        trained_model, training_history = train_mdn_model(
            train_loader=train_loader,
            val_loader=val_loader,
            model=model,
            seq_len=SEQ_LEN,
            num_features=len(FEATURE_COLS),
            num_mixtures=MDN_NUM_MIXTURES,
            epochs=EPOCHS,
            patience=PATIENCE,
            learning_rate=MDN_LEARNING_RATE,
            tcn_filters=128,
            tcn_kernel_size=7,
            tcn_dilations=[1, 2, 4, 8, 16, 32],
            dropout_rate=0.3,
            verbose=1,
            steps_per_epoch=train_steps_per_epoch,
            # ── LR Scheduler config ──
            lr_schedule='cosine',     # 'cosine' | 'constant' | 'step'
            warmup_epochs=3,          # linear warmup for first 2 epochs
            min_lr_ratio=0.01,        # decay to 1% of initial LR
        )
    
else:
    # ─────────────────────────────────────────────────────────────────────────
    # Standard training for other model types
    # ─────────────────────────────────────────────────────────────────────────
    trained_model, training_history = train_model(
        model, train_loader, val_loader,train_steps_per_epoch,val_steps,
        epochs=EPOCHS, lr=LR, patience=PATIENCE
    )

print("\n✅ Training completed successfully!")

## LSTM train and eval

In [ ]:
lstmModel=LSTMTraining()
lstmModel.build((None, SEQ_LEN, len(FEATURE_COLS)))
lstmModel.summary()
sample_x, _ = next(iter(train_loader))
lstmModel(sample_x)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

checkpoint_cb = ModelCheckpoint(
    filepath='best_lstm_model.h5',
    monitor='val_loss',
    save_best_only=True,
    save_weights_only=False,
    verbose=1
)
early_stopping = EarlyStopping(
    monitor='val_mae' if CO_ORDINATES else 'val_accuracy',
    patience=5,
    restore_best_weights=True,
    verbose=1
)
lstmModel.fit(
    train_loader,
    validation_data=val_loader,
    epochs=EPOCHS,
    steps_per_epoch=train_steps_per_epoch,
    validation_steps=val_steps,
    callbacks=[checkpoint_cb, early_stopping]
)


In [ ]:
from tensorflow.keras.models import load_model

lstmModel = SpatioTemporalLSTM(
    lstm_units=128,
    num_layers=2,
    dropout_rate=0.3,
    coordinate_targets=CO_ORDINATES,
    num_zones=NUM_ZONES
)

# Build with correct input shape
lstmModel.build(input_shape=(None, SEQ_LEN, len(FEATURE_COLS)))

# Load weights
lstmModel.load_weights('best_lstm_model.h5')


In [ ]:
LSTMEval(lstmModel)

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report, balanced_accuracy_score
import numpy as np

def evaluate_classification_metrics(model, test_loader, test_steps, top_k=5, coordinate_mode=CO_ORDINATES):
    """
    Evaluate classification performance on the test set for a model.
    It computes various metrics like Accuracy, Precision, Recall, F1 Score, etc.
    
    Parameters:
    - model: Trained model (LSTM or TCN in this case).
    - test_loader: Test dataset generator or data loader.
    - test_steps: Number of steps for evaluation.
    - top_k: Number of top zones to consider for classification evaluation.
    - coordinate_mode: Whether to use coordinate-based regression mode or not.

    Returns:
    - metrics_dict: Dictionary containing computed metrics.
    """
    
    all_preds = []
    all_labels = []

    for batch_idx, (batch_X, batch_y) in enumerate(test_loader.take(test_steps)):
        # Get predictions for this batch
        preds = model.predict(batch_X, verbose=0)
        
        # Debugging: Print the shape of the predictions
        print(f"Batch {batch_idx} - preds shape: {preds.shape}")

        # Convert labels to numpy
        labels = batch_y.numpy()

        all_preds.append(preds)
        all_labels.append(labels)

        # ---- Classification Mode ----
        if not coordinate_mode:
            # Multi-class classification: probs shape should be (batch_size, num_classes)
            probs = preds  # Get predicted probabilities (shape: [batch_size, num_classes])

            # For each sample, get the index of the class with the highest probability
            pred_classes = np.argmax(probs, axis=1)  # shape should now be [batch_size]

            true_classes = labels  # For multi-class classification, true labels

            # Ensure both true_classes and pred_classes are 1D arrays with the same length
            true_classes = true_classes.flatten()
            pred_classes = pred_classes.flatten()

            print(f"Batch {batch_idx} - true_classes shape: {true_classes.shape}")
            print(f"Batch {batch_idx} - pred_classes shape: {pred_classes.shape}")

            # Check if the lengths match
            print(f"Batch {batch_idx} - true_classes length: {len(true_classes)}")
            print(f"Batch {batch_idx} - pred_classes length: {len(pred_classes)}")

            batch_acc = accuracy_score(true_classes, pred_classes)

            if batch_idx % 10 == 0:  # Print accuracy for every 10 batches
                print(f"Batch {batch_idx} | Batch Accuracy: {batch_acc:.4f}")

    # Concatenate everything after loop
    all_preds = np.concatenate(all_preds, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)

    # ================================
    # FINAL METRICS (Classification or Coordinate-based)
    # ================================

    # Ensure both all_preds and all_labels are flattened and have consistent lengths
    all_preds = np.argmax(all_preds, axis=1)  # Get predicted classes from probabilities
    all_labels = np.argmax(all_labels, axis=1) if all_labels.ndim > 1 else all_labels  # Convert true labels if necessary

    # Compute classification metrics
    accuracy = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average='macro')
    balanced_acc = balanced_accuracy_score(all_labels, all_preds)

    # Print final metrics
    print("\n📊 Final Test Accuracy:")
    print(f"  Test Accuracy: {accuracy:.4f}")
    print(f"Macro F1 Score: {macro_f1:.4f}")
    print(f"Balanced Accuracy: {balanced_acc:.4f}")
    print(classification_report(all_labels, all_preds, zero_division=0))

    return {
        'accuracy': accuracy,
        'macro_f1': macro_f1,
        'balanced_accuracy': balanced_acc,
        'predictions': all_preds,
        'labels': all_labels,
        'probabilities': all_preds
    }

In [ ]:
# def predict_zone(model, sequence, scaler=None):
#     """
#     Predict the zone for a given sequence using the trained model.
    
#     Parameters:
#     - model: The trained SpatioTemporalLSTM model
#     - sequence: The input sequence (shape: [1, SEQ_LEN, num_features] or [batch_size, SEQ_LEN, num_features])
#     - scaler: Optional, a scaler to normalize the sequence (if used during training)
    
#     Returns:
#     - predicted_zone: Predicted zone index (0 to NUM_ZONES-1)
#     - probabilities: Softmax probabilities for each zone
#     """
    
#     # If scaler is provided, normalize the input sequence
#     if scaler is not None:
#         sequence = scaler.transform(sequence.reshape(-1, sequence.shape[-1])).reshape(sequence.shape)
    
#     # Get predictions (probabilities for each zone)
#     pred_probs = model.predict(sequence)
    
#     # Get predicted zone by taking argmax
#     predicted_zone = np.argmax(pred_probs, axis=-1)  # Index of max probability (zone)
    
#     return predicted_zone, pred_probs

# def batch_predict(model, sequences, batch_size=2, scaler=None):
#     predicted_zones = []
#     pred_probs_all = []

#     for i in range(0, len(sequences), batch_size):
#         batch = sequences[i:i+batch_size]
#         if scaler is not None:
#             batch = scaler.transform(batch.reshape(-1, batch.shape[-1])).reshape(batch.shape)
#         probs = model.predict(batch, verbose=0)
#         pred_zones = np.argmax(probs, axis=-1)
#         predicted_zones.extend(pred_zones)
#         pred_probs_all.extend(probs)
    
#     return np.array(predicted_zones), np.array(pred_probs_all)
# predicted_zone, pred_probs = batch_predict(lstmModel, np.array(sequences[:10]))

result=predict_zone_single(
    model=lstmModel,
    df=df,
    match_id='2417',
    player_id='5585',
    scaler=scaler,
    start_frame=2000,
    top_k=3,
    return_all_probs=True   
    )
print(result)
    # Print results
# for (match_id, player_id, t_sec), zone in zip(metadata, predicted_zone):
#     print(f"Match: {match_id}, Player: {player_id}, Time: {t_sec}s, Predicted Zone: {zone}")

In [ ]:
metrics=evaluate_classification_metrics(lstmModel, test_loader, 100, top_k=5, coordinate_mode=CO_ORDINATES)
print(metrics)

In [ ]:
# MDN MODEL BUILDER - TCN ENCODER + MDN HEAD

# Architecture:
#   Input (B, SEQ_LEN, features)
#      │
#      ▼
#   TCN Encoder (temporal feature extraction)
#      │
#      ▼
#   Dense layers (optional refinement)
#      │
#      ▼
#   MDN Layer → (π, μ, σ)
# ═══════════════════════════════════════════════════════════════════════════════
# MDN OUTPUT LAYER
# ═══════════════════════════════════════════════════════════════════════════════
# 
# Outputs three tensors for Gaussian Mixture Model:
#   π (pi)    : (B, K)    - Mixture weights (sum to 1 via softmax)
#   μ (mu)    : (B, K, 2) - Means for (x, y) per component
#   σ (sigma) : (B, K, 2) - Std deviations (always positive via softplus)
# ═══════════════════════════════════════════════════════════════════════════════

@register_keras_serializable(package='MDN')
class MDNLayer(tf.keras.layers.Layer):
    """
    Mixture Density Network output layer.
    
    Converts encoder output into GMM parameters for probabilistic prediction.
    
    Args:
        num_mixtures: Number of Gaussian components (K)
        output_dim: Dimension of output space (2 for x, y coordinates)
        sigma_min: Minimum std deviation to prevent collapse
    
    Outputs:
        pi: (batch, K) mixture weights summing to 1
        mu: (batch, K, output_dim) means per component
        sigma: (batch, K, output_dim) std deviations (positive)
    """
    
    def __init__(self, num_mixtures=5, output_dim=2, sigma_min=1e-4, **kwargs):
        super(MDNLayer, self).__init__(**kwargs)
        self.num_mixtures = num_mixtures
        self.output_dim = output_dim
        self.sigma_min = sigma_min
        
    def build(self, input_shape):
        hidden_dim = input_shape[-1]
        
        # π head: outputs K logits → softmax for mixture weights
        self.pi_dense = tf.keras.layers.Dense(
            self.num_mixtures,
            kernel_initializer='glorot_uniform',
            name='mdn_pi_logits'
        )
        
        # μ head: outputs K * output_dim values → reshape to (K, output_dim)
        self.mu_dense = tf.keras.layers.Dense(
            self.num_mixtures * self.output_dim,
            kernel_initializer='glorot_uniform',
            name='mdn_mu_raw'
        )
        
        # σ head: outputs K * output_dim values → softplus for positive values
        self.sigma_dense = tf.keras.layers.Dense(
            self.num_mixtures * self.output_dim,
            kernel_initializer='glorot_uniform',
            name='mdn_sigma_raw'
        )
        
        super(MDNLayer, self).build(input_shape)
        
    def call(self, inputs, training=None):
        """
        Forward pass through MDN layer.
        
        Args:
            inputs: (batch, hidden_dim) encoded features from TCN
            
        Returns:
            Tuple of (pi, mu, sigma)
        """
        batch_size = tf.shape(inputs)[0]
        
        # ─────────────────────────────────────────────────────────────────────
        # Mixture weights π: softmax ensures sum = 1
        # ─────────────────────────────────────────────────────────────────────
        pi_logits = self.pi_dense(inputs)  # (B, K)
        pi = tf.nn.softmax(pi_logits, axis=-1)  # (B, K), sum = 1
        
        # ─────────────────────────────────────────────────────────────────────
        # Means μ: unconstrained, reshaped to (B, K, output_dim)
        # ─────────────────────────────────────────────────────────────────────
        mu_raw = self.mu_dense(inputs)  # (B, K * output_dim)
        mu = tf.reshape(mu_raw, (batch_size, self.num_mixtures, self.output_dim))
        
        # ─────────────────────────────────────────────────────────────────────
        # Std deviations σ: softplus ensures positive, add minimum
        # ─────────────────────────────────────────────────────────────────────
        sigma_raw = self.sigma_dense(inputs)  # (B, K * output_dim)
        sigma = tf.nn.softplus(sigma_raw) + self.sigma_min  # Always positive
        sigma = tf.reshape(sigma, (batch_size, self.num_mixtures, self.output_dim))
        
        return pi, mu, sigma
    
    def get_config(self):
        config = super(MDNLayer, self).get_config()
        config.update({
            'num_mixtures': self.num_mixtures,
            'output_dim': self.output_dim,
            'sigma_min': self.sigma_min
        })
        return config
    
    @classmethod
    def from_config(cls, config):
        return cls(**config)


print("✅ MDNLayer class defined")
print(f"   • Outputs: π (B, K), μ (B, K, 2), σ (B, K, 2)")
print(f"   • Uses softmax for weights, softplus for std devs")

def build_tcn_mdn_model(
    seq_len,
    num_features,
    num_mixtures=5,
    tcn_filters=128,              # Changed to single int (required for skip connections)
    tcn_kernel_size=7,
    tcn_dilations=[1, 2, 4, 8, 16, 32],
    tcn_stacks=2,
    dropout_rate=0.3,
    use_batch_norm=True,
    dense_units=256,
    learning_rate=1e-4
):
    """
    Build complete TCN + MDN model for probabilistic coordinate prediction.
    
    Args:
        seq_len: Length of input sequences (e.g., 150)
        num_features: Number of input features (e.g., 26)
        num_mixtures: Number of Gaussian mixture components (K)
        tcn_filters: Number of filters for TCN (single int for skip connections)
        tcn_kernel_size: Kernel size for temporal convolutions
        tcn_dilations: Dilation rates for TCN
        tcn_stacks: Number of TCN stacks
        dropout_rate: Dropout rate
        use_batch_norm: Whether to use batch normalization
        dense_units: Hidden units before MDN layer
        learning_rate: Learning rate for optimizer
        
    Returns:
        Compiled Keras model with MDN outputs
    """
    
    print("═" * 60)
    print("🏗️  Building TCN + MDN Model")
    print("═" * 60)
    
    # ─────────────────────────────────────────────────────────────────────────
    # Input layer
    # ─────────────────────────────────────────────────────────────────────────
    inputs = tf.keras.Input(
        shape=(seq_len, num_features), 
        name='sequence_input'
    )
    print(f"   Input shape: (B, {seq_len}, {num_features})")
    
    # ─────────────────────────────────────────────────────────────────────────
    # TCN Encoder
    # Note: Use single filter count (int) for skip_connections compatibility
    # ─────────────────────────────────────────────────────────────────────────
    
    # If list provided, check if all equal; if not, use last value
    if isinstance(tcn_filters, list):
        if len(set(tcn_filters)) > 1:
            print(f"   ⚠️  Converting varying filters {tcn_filters} to uniform {tcn_filters[-1]}")
            print(f"       (Required for skip connections)")
        tcn_filters = tcn_filters[-1]  # Use the last (largest) value
    
    x = TCN(
        nb_filters=tcn_filters,
        kernel_size=tcn_kernel_size,
        nb_stacks=tcn_stacks,
        dilations=tcn_dilations,
        padding='causal',
        use_skip_connections=True,
        dropout_rate=dropout_rate,
        return_sequences=False,  # Get last timestep encoding
        use_batch_norm=use_batch_norm,
        use_layer_norm=False,
        name='tcn_encoder'
    )(inputs)
    
    print(f"   TCN output: (B, {tcn_filters})")
    
    # ─────────────────────────────────────────────────────────────────────────
    # Dense layers for feature refinement
    # ─────────────────────────────────────────────────────────────────────────
    x = tf.keras.layers.Dense(dense_units, activation='relu', name='dense_1')(x)
    x = tf.keras.layers.Dropout(dropout_rate, name='dropout_1')(x)
    x = tf.keras.layers.Dense(dense_units // 2, activation='relu', name='dense_2')(x)
    x = tf.keras.layers.Dropout(dropout_rate, name='dropout_2')(x)
    
    print(f"   Dense output: (B, {dense_units // 2})")
    
    # ─────────────────────────────────────────────────────────────────────────
    # MDN Output Layer
    # ─────────────────────────────────────────────────────────────────────────
    mdn_layer = MDNLayer(
        num_mixtures=num_mixtures, 
        output_dim=2,  # (x, y)
        sigma_min=MDN_SIGMA_MIN,
        name='mdn_output'
    )
    pi, mu, sigma = mdn_layer(x)
    
    print(f"   MDN outputs:")
    print(f"      π (weights): (B, {num_mixtures})")
    print(f"      μ (means):   (B, {num_mixtures}, 2)")
    print(f"      σ (stdevs):  (B, {num_mixtures}, 2)")
    
    # ─────────────────────────────────────────────────────────────────────────
    # Create model with named outputs
    # ─────────────────────────────────────────────────────────────────────────
    model = tf.keras.Model(
        inputs=inputs,
        outputs={
            'pi': pi,
            'mu': mu, 
            'sigma': sigma
        },
        name='TCN_MDN_Model'
    )
    
    # ─────────────────────────────────────────────────────────────────────────
    # Model summary
    # ─────────────────────────────────────────────────────────────────────────
    total_params = model.count_params()
    print(f"\n   Total parameters: {total_params:,}")
    print("═" * 60)
    
    return model


def compile_mdn_model(model, learning_rate=1e-4):
    """
    Compile MDN model with appropriate optimizer.
    
    Note: MDN uses custom training loop, so we don't compile with loss.
    This function sets up the optimizer for manual training.
    """
    optimizer = tf.keras.optimizers.Adam(
        learning_rate=learning_rate,
        clipnorm=1.0  # Gradient clipping for stability
    )
    return optimizer


print("✅ MDN Model builder functions defined")
print("   • build_tcn_mdn_model: Creates TCN + MDN architecture")
print("   • compile_mdn_model: Sets up optimizer")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# LOAD SAVED MDN MODEL & RUN INFERENCE
# ═══════════════════════════════════════════════════════════════════════════════

import os
import numpy as np
from tensorflow.keras.models import load_model

# ─────────────────────────────────────────────────────────────────────────
# 1. CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────
MDN_MODEL_PATH = '../MDN/mdn_model_Keras_V3_24_100F_2.10NLL(best).keras'

def load_mdn_model_simple(filepath):
    """
    Load MDN model by providing all custom objects to Keras.
    This is the most reliable method for models with custom layers.
    """
    print(f"📦 Loading MDN Model from: {filepath}")
    
    # Define custom objects dict with all necessary components
    custom_objects = {
        'MDNLayer': MDNLayer,  # Custom MDN layer from this notebook
        'TCN': TCN,            # TCN layer
    }
    
    try:
        model = load_model(
            filepath,
            custom_objects=custom_objects,
            safe_mode=False  # Allow loading from different Keras versions
        )
        print(f"✅ Model loaded successfully!")
        return model
        
    except Exception as e:
        print(f"❌ Error loading model: {e}")
        print(f"\n🔧 Diagnostic info:")
        print(f"   File exists: {os.path.exists(filepath)}")
        print(f"   File size: {os.path.getsize(filepath) if os.path.exists(filepath) else 'N/A'} bytes")
        raise

# Check if model file exists
if not os.path.exists(MDN_MODEL_PATH):
    print(f"⚠️  Model file not found: {MDN_MODEL_PATH}")
    print("   To save the current trained model, run:")
    print("   >>> model.save('mdn_model.keras')")
    print("\n   Or update MDN_MODEL_PATH to point to an existing model file.")
else:
    # ─────────────────────────────────────────────────────────────────────
    # 2. LOAD MODEL
    # ─────────────────────────────────────────────────────────────────────
    print("═" * 70)
    print("🔮 LOADING SAVED MDN MODEL")
    print("═" * 70)
    
    loaded_mdn_model = load_mdn_model_simple(MDN_MODEL_PATH)
    _model_loaded = True
    
    print(f"\n📊 Model Summary:")
    print(f"   • Parameters: {loaded_mdn_model.count_params():,}")
    print(f"   • Input shape: {loaded_mdn_model.input_shape}")
    print(f"   • Output keys: {list(loaded_mdn_model.output.keys()) if isinstance(loaded_mdn_model.output, dict) else 'single output'}")


In [ ]:
loaded_mdn_model.summary()

In [ ]:
# MDN INFERENCE UTILITIES

# Functions for extracting predictions from MDN:
#   - Get multiple future positions with probabilities
#   - Sample from the predicted distribution
#   - Select top-K most likely futures

def mdn_predict(model, x_sequence):
    """
    Get MDN predictions from input sequence(s).
    
    Args:
        model: Trained MDN model
        x_sequence: Input data
            - Single: (SEQ_LEN, features) or (1, SEQ_LEN, features)
            - Batch: (B, SEQ_LEN, features)
            
    Returns:
        dict with:
            - 'pi': (B, K) mixture weights
            - 'mu': (B, K, 2) means
            - 'sigma': (B, K, 2) std deviations
    """
   
    # Ensure batch dimension
    if len(x_sequence.shape) == 2:
        x_sequence = np.expand_dims(x_sequence, axis=0)
    
    # Get predictions
    outputs = model(x_sequence, training=False)
    
    return {
        'pi': outputs['pi'].numpy(),
        'mu': outputs['mu'].numpy(),
        'sigma': outputs['sigma'].numpy()
    }


def get_top_k_predictions(pi, mu, sigma, k=3):
    """
    Extract top-K most probable future positions.
    
    Args:
        pi: (K,) mixture weights for one sample
        mu: (K, 2) mixture means
        sigma: (K, 2) mixture std deviations
        k: Number of top predictions to return
        
    Returns:
        List of dicts with keys:
            - 'probability': Mixture weight
            - 'x': Predicted x coordinate
            - 'y': Predicted y coordinate
            - 'x_std': Uncertainty in x
            - 'y_std': Uncertainty in y
    """
    # Sort by probability (descending)
    sorted_indices = np.argsort(pi)[::-1][:k]
    
    results = []
    for idx in sorted_indices:
        results.append({
            'probability': float(pi[idx]),
            'x': float(mu[idx, 0]),
            'y': float(mu[idx, 1]),
            'x_std': float(sigma[idx, 0]),
            'y_std': float(sigma[idx, 1]),
            'component': int(idx)
        })
    
    return results


def sample_from_mdn(pi, mu, sigma, num_samples=100):
    """
    Sample future positions from the predicted GMM distribution.
    
    This generates samples that represent the full predictive distribution,
    useful for Monte Carlo analysis or uncertainty visualization.
    
    Args:
        pi: (K,) mixture weights
        mu: (K, 2) means
        sigma: (K, 2) std deviations
        num_samples: Number of samples to draw
        
    Returns:
        samples: (num_samples, 2) array of sampled (x, y) coordinates
        component_ids: (num_samples,) which component each sample came from
    """
    samples = []
    component_ids = []
    
    for _ in range(num_samples):
        # Choose mixture component based on weights
        k = np.random.choice(len(pi), p=pi)
        
        # Sample from that component's Gaussian
        x = np.random.normal(mu[k, 0], sigma[k, 0])
        y = np.random.normal(mu[k, 1], sigma[k, 1])
        
        samples.append([x, y])
        component_ids.append(k)
    
    return np.array(samples), np.array(component_ids)


def get_expected_position(pi, mu):
    """
    Compute expected (mean) position from mixture.
    
    E[x] = Σ πₖ · μₖ
    
    This is useful for comparison with deterministic predictions.
    
    Args:
        pi: (K,) mixture weights
        mu: (K, 2) means
        
    Returns:
        (2,) array with expected (x, y)
    """
    # Weighted sum of means
    expected = np.sum(pi[:, np.newaxis] * mu, axis=0)
    return expected


def get_most_likely_position(pi, mu):
    """
    Get the single most likely position (mode).
    
    This is simply the mean of the highest-weight component.
    
    Args:
        pi: (K,) mixture weights
        mu: (K, 2) means
        
    Returns:
        (2,) array with mode (x, y)
    """
    best_k = np.argmax(pi)
    return mu[best_k]


def batch_predictions_to_dataframe(predictions, sample_ids=None):
    """
    Convert batch MDN predictions to a pandas DataFrame for analysis.
    
    Args:
        predictions: dict with 'pi', 'mu', 'sigma' from mdn_predict()
        sample_ids: Optional list of sample identifiers
        
    Returns:
        DataFrame with one row per (sample, component)
    """
    pi = predictions['pi']
    mu = predictions['mu']
    sigma = predictions['sigma']
    
    B, K, _ = mu.shape
    
    records = []
    for b in range(B):
        sample_id = sample_ids[b] if sample_ids is not None else b
        for k in range(K):
            records.append({
                'sample_id': sample_id,
                'component': k,
                'probability': pi[b, k],
                'x': mu[b, k, 0],
                'y': mu[b, k, 1],
                'x_std': sigma[b, k, 0],
                'y_std': sigma[b, k, 1]
            })
    
    return pd.DataFrame(records)


print("✅ MDN Inference utilities defined")
print("   • mdn_predict: Get (π, μ, σ) from model")
print("   • get_top_k_predictions: Extract top-K futures")
print("   • sample_from_mdn: Monte Carlo sampling")
print("   • get_expected_position: Weighted mean")
print("   • get_most_likely_position: Mode (highest π)")
print("   • batch_predictions_to_dataframe: For analysis")

In [ ]:


# ─────────────────────────────────────────────────────────────────────────────
# 3. RUN INFERENCE ON TEST DATA
# ─────────────────────────────────────────────────────────────────────────────
if _model_loaded:
    print("\n" + "═" * 70)
    print("🎯 RUNNING INFERENCE")
    print("═" * 70)
    
    # Get a sample batch from test_loader (assumes test_loader is defined)
    try:
        x_sample, y_true_sample = next(iter(test_loader.take(2)))
        print(f"✓ Retrieved batch: x_sample={x_sample.shape}, y_true={y_true_sample.shape}")
    except Exception as e:
        print(f"⚠️  Could not get test data: {e}")
        print("   Using random dummy data for demonstration...")
        x_sample = np.random.randn(1, SEQ_LEN, len(FEATURE_COLS)).astype(np.float32)
        y_true_sample = np.array([[0.5, 0.5]])
    
    # ── Single sample inference ──
    x_single = x_sample[0:1]  # Take first sample
    y_true_single = y_true_sample[0].numpy() if hasattr(y_true_sample[0], 'numpy') else y_true_sample[0]
    
    # Get predictions using mdn_predict utility
    predictions = mdn_predict(loaded_mdn_model, x_single)
    print(predictions.keys())
    pi = predictions['pi'][0]      # (K,)
    mu = predictions['mu'][0]      # (K, 2)
    sigma = predictions['sigma'][0]  # (K, 2)
    
    # ────────────────────────────────────────────────────────────────────────
    # 4. DISPLAY PREDICTIONS
    # ────────────────────────────────────────────────────────────────────────
    print("\n" + "─" * 60)
    print("📍 SINGLE SAMPLE PREDICTIONS")
    print("─" * 60)
    
    # Top-K predictions
    top_k = get_top_k_predictions(pi, mu, sigma, k=min(3, MDN_NUM_MIXTURES))
    
    print(f"\n🎯 Top-{len(top_k)} Predicted Positions:")
    for i, pred in enumerate(top_k, 1):
        print(f"   {i}. ({pred['x']:.3f}, {pred['y']:.3f}) → {pred['probability']*100:.1f}% probability")
        print(f"      Uncertainty: σx={pred['x_std']:.4f}, σy={pred['y_std']:.4f}")
    
    # Expected (mean) position
    expected_pos = get_expected_position(pi, mu)
    print(f"\n📊 Expected Position (weighted mean):")
    print(f"   → ({expected_pos[0]:.3f}, {expected_pos[1]:.3f})")
    
    # Most likely (mode) position
    mode_pos = get_most_likely_position(pi, mu)
    print(f"\n🌟 Most Likely Position (mode):")
    print(f"   → ({mode_pos[0]:.3f}, {mode_pos[1]:.3f})")
    
    # Ground truth comparison
    # print(f"\n✅ Ground Truth:")
    # print(f"   → ({y_true_single[0]:.3f}, {y_true_single[1]:.3f})")
    
    # Error metrics
    expected_error = np.linalg.norm(expected_pos - y_true_single)
    mode_error = np.linalg.norm(mode_pos - y_true_single)
    print(f"\n📏 Errors:")
    print(f"   • Expected position error: {expected_error:.4f}")
    print(f"   • Mode position error:     {mode_error:.4f}")
    
    # ────────────────────────────────────────────────────────────────────────
    # 5. SAMPLE FROM DISTRIBUTION (Monte Carlo)
    # ────────────────────────────────────────────────────────────────────────
    print("\n" + "─" * 60)
    print("🎲 MONTE CARLO SAMPLING")
    print("─" * 60)
    
    samples, component_ids = sample_from_mdn(pi, mu, sigma, num_samples=100)
    
    print(f"   • Generated {len(samples)} samples from predicted GMM")
    print(f"   • Sample mean: ({samples[:, 0].mean():.3f}, {samples[:, 1].mean():.3f})")
    print(f"   • Sample std:  ({samples[:, 0].std():.3f}, {samples[:, 1].std():.3f})")
    
    # Component usage
    unique, counts = np.unique(component_ids, return_counts=True)
    print(f"   • Component usage: {dict(zip(unique, counts))}")
    
    # ────────────────────────────────────────────────────────────────────────
    # 6. BATCH INFERENCE
    # ────────────────────────────────────────────────────────────────────────
    print("\n" + "─" * 60)
    print("📦 BATCH INFERENCE")
    print("─" * 60)
    
    batch_preds = mdn_predict(loaded_mdn_model, x_sample)
    batch_size = batch_preds['pi'].shape[0]
    
    print(f"   • Batch size: {batch_size}")
    print(f"   • π shape: {batch_preds['pi'].shape}")
    print(f"   • μ shape: {batch_preds['mu'].shape}")
    print(f"   • σ shape: {batch_preds['sigma'].shape}")
    
    # Calculate batch errors
    expected_positions = np.array([
        get_expected_position(batch_preds['pi'][i], batch_preds['mu'][i])
        for i in range(batch_size)
    ])
    y_true_np = y_true_sample.numpy() if hasattr(y_true_sample, 'numpy') else y_true_sample
    batch_errors = np.linalg.norm(expected_positions - y_true_np, axis=1)
    
    print(f"\n   Batch MAE (expected positions): {batch_errors.mean():.4f}")
    print(f"   Batch error std: {batch_errors.std():.4f}")
    print(f"   Min error: {batch_errors.min():.4f}, Max error: {batch_errors.max():.4f}")
    
    # ────────────────────────────────────────────────────────────────────────
    # 7. OPTIONAL: VISUALIZATION
    # ────────────────────────────────────────────────────────────────────────
    print("\n" + "─" * 60)
    print("📈 VISUALIZATION")
    print("─" * 60)
    
    try:
        # Get current position from last frame of input sequence
        current_pos = x_single[0, -1, 0:2].numpy() if hasattr(x_single, 'numpy') else x_single[0, -1, 0:2]

        fig = plot_mdn_prediction_on_field(
            pi=pi, 
            mu=mu, 
            sigma=sigma,
            true_pos=y_true_single,
            current_pos=current_pos,
            title="MDN Prediction - Loaded Model"
           
        )
        plt.show()
        print("✅ Visualization displayed")
    except Exception as e:
        print(f"⚠️  Visualization skipped: {e}")
    
    print("\n" + "═" * 70)
    print("✅ INFERENCE COMPLETE")
    print("═" * 70)
    print("\n💡 Usage examples:")
    print("   # Single prediction:")
    print("   >>> preds = mdn_predict(loaded_mdn_model, x_sequence)")
    print("   >>> top_k = get_top_k_predictions(preds['pi'][0], preds['mu'][0], preds['sigma'][0])")
    print("\n   # Full evaluation:")
    print("   >>> metrics = full_mdn_evaluation(loaded_mdn_model, test_loader)")
    

## Evaluation

In [ ]:
expected_timesteps = loaded_mdn_model.input_shape[1]  # 100
X_val_batch, y_val_batch = next(iter(test_loader))

X_val_batch_fixed = np.array([
    x[-expected_timesteps:] if x.shape[0] > expected_timesteps else
    np.pad(x, ((expected_timesteps - x.shape[0], 0), (0, 0)), mode='constant')
    for x in X_val_batch
])

# Take one batch from validation loader

print("="*60)
print("🔍 VALIDATION PREDICTION INSPECTION")
print("="*60)

# ═══════════════════════════════════════════════════════════════════════════════
# MDN MODE - Multi-modal probabilistic predictions
# ═══════════════════════════════════════════════════════════════════════════════
if MODEL_TYPE == "MDN_TCN":
    print(f"Mode: MDN (Multi-modal Probabilistic Predictions)")
    
    # Get MDN predictions
    outputs = loaded_mdn_model(X_val_batch_fixed, training=False)
    pi = outputs['pi'].numpy()      # (B, K)
    mu = outputs['mu'].numpy()      # (B, K, 2)
    sigma = outputs['sigma'].numpy() # (B, K, 2)
    
    # Get true values
    if hasattr(y_val_batch, 'numpy'):
        y_true = y_val_batch.numpy()
    else:
        y_true = np.array(y_val_batch)
    
    print(f"\n📊 Batch Statistics:")
    print(f"  • Samples: {len(y_true)}")
    print(f"  • Mixtures (K): {pi.shape[1]}")
    
    # Show first 3 samples
    print(f"\n📌 First 3 Predictions:")
    for i in range(min(3, len(y_true))):
        print(f"\n  Sample {i+1}:")
        print(f"    True position: ({y_true[i, 0]:.3f}, {y_true[i, 1]:.3f})")
        
        # Get top-3 predictions for this sample
        top3 = get_top_k_predictions(pi[i], mu[i], sigma[i], k=3)
        for j, pred in enumerate(top3):
            print(f"    Pred {j+1}: ({pred['x']:.3f}, {pred['y']:.3f}) "
                  f"- {pred['probability']*100:.1f}% "
                  f"± ({pred['x_std']:.3f}, {pred['y_std']:.3f})")
    
    # Calculate metrics
    print(f"\n📈 Batch Metrics:")
    
    # Best-of-3 MAE
    best_errors = []
    expected_errors = []
    mode_errors = []
    for i in range(len(y_true)):
        distances = np.linalg.norm(mu[i] - y_true[i], axis=-1)
        top_k_indices = np.argsort(pi[i])[::-1][:3]
        best_errors.append(np.min(distances[top_k_indices]))
        expected_errors.append(np.linalg.norm(get_expected_position(pi[i], mu[i]) - y_true[i]))
        mode_errors.append(np.linalg.norm(get_most_likely_position(pi[i], mu[i]) - y_true[i]))
    
    print(f"  • Best-of-3 MAE:  {np.mean(best_errors):.4f}")
    print(f"  • Expected MAE:   {np.mean(expected_errors):.4f}")
    print(f"  • Mode MAE:       {np.mean(mode_errors):.4f}")
    
    # Component diversity
    mean_active = np.mean([np.sum(pi[i] > 0.1) for i in range(len(pi))])
    mean_max_weight = np.mean(np.max(pi, axis=1))
    print(f"  • Avg active components: {mean_active:.2f}")
    print(f"  • Avg max weight: {mean_max_weight:.2%}")

# ═══════════════════════════════════════════════════════════════════════════════
# COORDINATE REGRESSION MODE
# ═══════════════════════════════════════════════════════════════════════════════
elif CO_ORDINATES:
    # Make predictions
    preds = trained_model.predict(X_val_batch, verbose=0)
    
    # --- REGRESSION MODE (Coordinates) ---
    print(f"Mode: REGRESSION (Predicting x, y coordinates)")

    # Get values (handle tensors)
    if hasattr(y_val_batch, 'numpy'):
        y_true = y_val_batch.numpy()
    else:
        y_true = np.array(y_val_batch)

    # preds is already numpy from model.predict

    print(f"\nTrue Coords (First 5):\n{y_true[:5]}")
    print(f"\nPred Coords (First 5):\n{preds[:5]}")

    # Calculate Errors
    errors = y_true - preds
    mae = np.mean(np.abs(errors))
    mse = np.mean(errors**2)
    euclidean_dist = np.sqrt(np.sum(errors**2, axis=1))

    print("-" * 30)
    print(f"Batch MAE:  {mae:.4f}")
    print(f"Batch RMSE: {np.sqrt(mse):.4f}")
    print(f"Mean Euclidean Dist: {np.mean(euclidean_dist):.4f}")

# ═══════════════════════════════════════════════════════════════════════════════
# ZONE CLASSIFICATION MODE
# ═══════════════════════════════════════════════════════════════════════════════
else:
    # Make predictions
    preds = trained_model.predict(X_val_batch, verbose=0)
    
    # --- CLASSIFICATION MODE (Zones) ---
    print(f"Mode: CLASSIFICATION (Predicting Zones)")

    pred_zones = np.argmax(preds, axis=1)

    # Handle potential tensor/numpy difference in labels
    if hasattr(y_val_batch, 'numpy'):
        true_zones = y_val_batch.numpy().flatten()
    else:
        true_zones = y_val_batch.flatten()

    print(f"True Zones (First 20):      {true_zones[:20]}")
    print(f"Predicted Zones (First 20): {pred_zones[:20]}")

    # Check if predictions are 'stuck'
    unique_preds = np.unique(pred_zones)
    print(f"\nUnique predicted zones in batch: {len(unique_preds)}")
    print(f"Most common prediction: {np.bincount(pred_zones).argmax()} (Count: {np.max(np.bincount(pred_zones))}/{len(pred_zones)})")

    # --- NEW: Spatial Error Analysis ---
    def zone_to_rc(zones, n_cols):
        return np.stack([zones // n_cols, zones % n_cols], axis=1)

    true_rc = zone_to_rc(true_zones, N_COLS)
    pred_rc = zone_to_rc(pred_zones, N_COLS)

    # Manhattan distance (rows + cols difference)
    grid_distances = np.abs(true_rc - pred_rc).sum(axis=1)

    print(f"\nBatch Accuracy: {np.mean(pred_zones == true_zones):.4f}")
    print(f"Mean Grid Distance Error: {grid_distances.mean():.2f} zones")
    print(f"Max Grid Distance Error:  {grid_distances.max()} zones")

    # Check probability of true class
    true_class_probs = preds[np.arange(len(preds)), true_zones]
    print(f"Mean Probability of True Zone: {true_class_probs.mean():.4f}")

print("="*60)

In [ ]:
try:
    preds = model.predict(val_loader, verbose=0)
    print(f"Predictions shape: {preds.shape}")
except Exception as e:
    print(f"Error during prediction: {e}")
    print("This may be due to model architecture or data issues. Please check the model and data compatibility.")
    print("using loaded model for prediction...")
    preds=loaded_mdn_model.predict(X_val_batch, verbose=0)

print(preds.keys())


In [ ]:
# Get predictions
if MODEL_TYPE == "MDN_TCN":
    no_val = 1000
    expected_len = loaded_mdn_model.input_shape[1]

    collected = 0
    X_list = []
    y_list = []

    for X_batch, y_batch in test_loader:
        X_np = X_batch.numpy()
        y_np = y_batch.numpy()
        
        X_list.append(X_np)
        y_list.append(y_np)
        
        collected += X_np.shape[0]
        if collected >= no_val:
            break

    # Concatenate and slice to exact amount
    X_val_all = np.concatenate(X_list, axis=0)[:no_val]
    y_val_all = np.concatenate(y_list, axis=0)[:no_val]

    # Fix sequence length
    X_val_all = X_val_all[:, -expected_len:, :]

    print("Actual number of samples used:", len(X_val_all))

    # Predict
    preds = loaded_mdn_model.predict(X_val_all, verbose=0)

    pi = preds['pi']
    mu = preds['mu']

    pred_values = np.sum(pi[..., None] * mu, axis=1)

    mse = np.mean((pred_values.squeeze() - y_val_all) ** 2)
    print(f"Test MSE ({len(X_val_all)} samples): {mse:.4f}")





In [ ]:

# ═══════════════════════════════════════════════════════════════════════════════
# MDN NEGATIVE LOG-LIKELIHOOD (NLL) EVALUATION
# ═══════════════════════════════════════════════════════════════════════════════

def compute_gaussian_pdf(y, mu, sigma, epsilon=1e-8):
    """
    Compute Gaussian probability density function.
    
    Args:
        y: Target values (B, output_dim)
        mu: Means for each component (B, K, output_dim)
        sigma: Standard deviations (B, K, output_dim)
        epsilon: Small value to prevent division by zero
        
    Returns:
        Gaussian PDF values (B, K)
    """
    # Expand y to match mu dimensions: (B, 1, output_dim)
    y_expanded = tf.expand_dims(y, axis=1)  # (B, 1, output_dim)
    
    # (y - mu)^2 / (2 * sigma^2)
    exponent = -0.5 * tf.reduce_sum(
        ((y_expanded - mu) / (sigma + epsilon)) ** 2, 
        axis=-1
    )  # (B, K)
    
    # Normalize: 1 / (sqrt(2*pi) * prod(sigma))
    normalization = 1.0 / (tf.sqrt(2 * np.pi) ** 2 * tf.reduce_prod(sigma + epsilon, axis=-1))  # (B, K)
    
    # PDF = normalization * exp(exponent)
    pdf = normalization * tf.exp(exponent)  # (B, K)
    
    return pdf


def compute_mdn_nll(y_true, pi, mu, sigma, epsilon=1e-8):
    """
    Compute Negative Log-Likelihood for MDN with mixture of Gaussians.
    
    NLL = -log(sum_k(pi_k * N(y | mu_k, sigma_k)))
    
    Args:
        y_true: Ground truth targets (B, output_dim)
        pi: Mixture weights (B, K)
        mu: Means for each component (B, K, output_dim)
        sigma: Standard deviations (B, K, output_dim)
        epsilon: Small value for numerical stability
        
    Returns:
        NLL value (scalar)
    """
    # Compute Gaussian PDF for each component
    pdf = compute_gaussian_pdf(y_true, mu, sigma, epsilon=epsilon)  # (B, K)
    
    # Mixture likelihood: sum_k(pi_k * pdf_k)
    mixture_likelihood = tf.reduce_sum(pi * pdf, axis=1)  # (B,)
    
    # Add epsilon for numerical stability
    mixture_likelihood = tf.maximum(mixture_likelihood, epsilon)
    
    # NLL = -log(mixture_likelihood)
    nll = -tf.math.log(mixture_likelihood)  # (B,)
    
    return tf.reduce_mean(nll), nll


def evaluate_mdn_nll(model, loader, num_batches=None, verbose=True):
    """
    Evaluate MDN model on a dataset and compute average NLL.
    
    Args:
        model: Trained MDN model
        loader: tf.data.Dataset yielding (X, y) pairs
        num_batches: Number of batches to evaluate (None = all)
        verbose: Print statistics
        
    Returns:
        Dictionary with NLL metrics
    """
    all_nll = []
    all_targets = []
    all_preds = {}
    
    print("\n" + "═" * 70)
    print("🔬 MDN MODEL EVALUATION")
    print("═" * 70)
    
    batch_count = 0
    
    for X_batch, y_batch in loader:
        if num_batches and batch_count >= num_batches:
            break
        
        # Get predictions
        preds = model(X_batch, training=False)
        pi = preds['pi'].numpy()      # (B, K)
        mu = preds['mu'].numpy()      # (B, K, output_dim)
        sigma = preds['sigma'].numpy()  # (B, K, output_dim)
        
        y_true = y_batch.numpy()      # (B, output_dim)
        
        # Compute NLL
        mean_nll, batch_nll = compute_mdn_nll(
            tf.constant(y_true, dtype=tf.float32),
            tf.constant(pi, dtype=tf.float32),
            tf.constant(mu, dtype=tf.float32),
            tf.constant(sigma, dtype=tf.float32)
        )
        
        all_nll.extend(batch_nll.numpy())
        all_targets.append(y_true)
        batch_count += 1
        
        if verbose and batch_count % 5 == 0:
            print(f"Batch {batch_count} | Mean NLL: {mean_nll.numpy():.4f}")
    
    # Aggregate results
    all_nll = np.array(all_nll)
    all_targets = np.concatenate(all_targets, axis=0)
    
    results = {
        'nll_values': all_nll,
        'mean_nll': np.mean(all_nll),
        'std_nll': np.std(all_nll),
        'min_nll': np.min(all_nll),
        'max_nll': np.max(all_nll),
        'median_nll': np.median(all_nll),
        'targets': all_targets,
        'num_samples': len(all_nll)
    }
    
    if verbose:
        print("\n📊 NLL Statistics:")
        print(f"   Mean NLL:   {results['mean_nll']:.4f}")
        print(f"   Std NLL:    {results['std_nll']:.4f}")
        print(f"   Min NLL:    {results['min_nll']:.4f}")
        print(f"   Max NLL:    {results['max_nll']:.4f}")
        print(f"   Median NLL: {results['median_nll']:.4f}")
        print(f"   Samples:    {results['num_samples']}")
        print("═" * 70)
    
    return results


# ─────────────────────────────────────────────────────────────────────────
# Evaluate on Validation Set
# ─────────────────────────────────────────────────────────────────────────

if _model_loaded and MODEL_TYPE == "MDN_TCN":
    print("\n📈 EVALUATING MDN MODEL NLL ON VALIDATION SET\n")
    
    # Evaluate on first 100 batches of validation data
    val_nll_results = evaluate_mdn_nll(
        loaded_mdn_model, 
        val_loader.take(100),
        verbose=True
    )
    
    print("\n📈 EVALUATING MDN MODEL NLL ON TEST SET\n")
    
    # Evaluate on first 100 batches of test data
    test_nll_results = evaluate_mdn_nll(
        loaded_mdn_model, 
        test_loader.take(100),
        verbose=True
    )
    
    print("\n" + "=" * 70)
    print("📊 COMPARISON: VALIDATION vs TEST")
    print("=" * 70)
    print(f"Validation Mean NLL: {val_nll_results['mean_nll']:.4f}")
    print(f"Test Mean NLL:       {test_nll_results['mean_nll']:.4f}")
    print(f"Difference:          {abs(val_nll_results['mean_nll'] - test_nll_results['mean_nll']):.4f}")
    print("=" * 70)


In [ ]:

# ═══════════════════════════════════════════════════════════════════════════════
# NLL VISUALIZATION AND ANALYSIS
# ═══════════════════════════════════════════════════════════════════════════════

def plot_nll_analysis(validation_results, test_results):
    """
    Visualize NLL analysis: distributions, histograms, and statistics.
    """
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # ─────────────────────────────────────────────────────────────────────
    # 1. NLL Distribution Comparison
    # ─────────────────────────────────────────────────────────────────────
    axes[0, 0].hist(validation_results['nll_values'], bins=50, alpha=0.6, label='Validation', color='blue', edgecolor='black')
    axes[0, 0].hist(test_results['nll_values'], bins=50, alpha=0.6, label='Test', color='red', edgecolor='black')
    axes[0, 0].axvline(validation_results['mean_nll'], color='blue', linestyle='--', linewidth=2, label=f'Val Mean: {validation_results["mean_nll"]:.3f}')
    axes[0, 0].axvline(test_results['mean_nll'], color='red', linestyle='--', linewidth=2, label=f'Test Mean: {test_results["mean_nll"]:.3f}')
    axes[0, 0].set_xlabel('NLL Value', fontsize=11)
    axes[0, 0].set_ylabel('Frequency', fontsize=11)
    axes[0, 0].set_title('NLL Distribution: Validation vs Test', fontsize=12, fontweight='bold')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # ─────────────────────────────────────────────────────────────────────
    # 2. Box Plot Comparison
    # ─────────────────────────────────────────────────────────────────────
    box_data = [validation_results['nll_values'], test_results['nll_values']]
    bp = axes[0, 1].boxplot(
        box_data,
        labels=['Validation', 'Test'],
        patch_artist=True,
        showmeans=True
    )
    for patch, color in zip(bp['boxes'], ['lightblue', 'lightcoral']):
        patch.set_facecolor(color)
    axes[0, 1].set_ylabel('NLL Value', fontsize=11)
    axes[0, 1].set_title('NLL Distribution: Box Plot', fontsize=12, fontweight='bold')
    axes[0, 1].grid(True, alpha=0.3, axis='y')
    
    # ─────────────────────────────────────────────────────────────────────
    # 3. Cumulative Distribution
    # ─────────────────────────────────────────────────────────────────────
    val_sorted = np.sort(validation_results['nll_values'])
    test_sorted = np.sort(test_results['nll_values'])
    axes[1, 0].plot(val_sorted, np.linspace(0, 1, len(val_sorted)), label='Validation', linewidth=2)
    axes[1, 0].plot(test_sorted, np.linspace(0, 1, len(test_sorted)), label='Test', linewidth=2)
    axes[1, 0].set_xlabel('NLL Value', fontsize=11)
    axes[1, 0].set_ylabel('Cumulative Probability', fontsize=11)
    axes[1, 0].set_title('Cumulative Distribution Function (CDF)', fontsize=12, fontweight='bold')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # ─────────────────────────────────────────────────────────────────────
    # 4. Statistics Table
    # ─────────────────────────────────────────────────────────────────────
    axes[1, 1].axis('off')
    
    stats_text = f"""
    ╔══════════════════════════════════════════╗
    ║         NLL EVALUATION METRICS           ║
    ╠══════════════════════════════════════════╣
    ║ VALIDATION SET                           ║
    ║ ────────────────────────────────────────── ║
    ║ Mean NLL:     {validation_results['mean_nll']:>24.4f}  ║
    ║ Std NLL:      {validation_results['std_nll']:>24.4f}  ║
    ║ Min NLL:      {validation_results['min_nll']:>24.4f}  ║
    ║ Max NLL:      {validation_results['max_nll']:>24.4f}  ║
    ║ Median NLL:   {validation_results['median_nll']:>24.4f}  ║
    ║ Samples:      {validation_results['num_samples']:>24.0f}  ║
    ╠══════════════════════════════════════════╣
    ║ TEST SET                                 ║
    ║ ────────────────────────────────────────── ║
    ║ Mean NLL:     {test_results['mean_nll']:>24.4f}  ║
    ║ Std NLL:      {test_results['std_nll']:>24.4f}  ║
    ║ Min NLL:      {test_results['min_nll']:>24.4f}  ║
    ║ Max NLL:      {test_results['max_nll']:>24.4f}  ║
    ║ Median NLL:   {test_results['median_nll']:>24.4f}  ║
    ║ Samples:      {test_results['num_samples']:>24.0f}  ║
    ╚══════════════════════════════════════════╝
    """
    
    axes[1, 1].text(0.05, 0.95, stats_text, transform=axes[1, 1].transAxes,
                    fontsize=9, verticalalignment='top', fontfamily='monospace',
                    bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))
    
    plt.tight_layout()
    plt.show()


def compute_nll_percentiles(nll_results):
    """
    Compute percentiles for NLL values to understand distribution better.
    """
    nll_values = nll_results['nll_values']
    
    percentiles = [5, 10, 25, 50, 75, 90, 95, 99]
    percentile_values = np.percentile(nll_values, percentiles)
    
    print("\n📊 NLL PERCENTILES:")
    print("─" * 50)
    for p, v in zip(percentiles, percentile_values):
        print(f"   {p:2d}th percentile: {v:.4f}")
    print("─" * 50)
    
    return dict(zip(percentiles, percentile_values))


# ─────────────────────────────────────────────────────────────────────────
# Visualize and Analyze Results
# ─────────────────────────────────────────────────────────────────────────

if _model_loaded and MODEL_TYPE == "MDN_TCN":
    print("\n" + "=" * 70)
    print("📊 DETAILED NLL ANALYSIS")
    print("=" * 70)
    
    # Compute percentiles
    val_percentiles = compute_nll_percentiles(val_nll_results)
    test_percentiles = compute_nll_percentiles(test_nll_results)
    
    # Plot analysis
    plot_nll_analysis(val_nll_results, test_nll_results)
    
    # Additional metrics
    print("\n📈 GENERALIZATION ANALYSIS:")
    print("─" * 70)
    
    # Check for overfitting/underfitting
    mean_diff = test_nll_results['mean_nll'] - val_nll_results['mean_nll']
    if abs(mean_diff) < 0.1:
        status = "✅ Good generalization"
    elif mean_diff > 0.5:
        status = "⚠️  Potential overfitting"
    else:
        status = "⚠️  Check model performance"
    
    print(f"   Validation Mean NLL: {val_nll_results['mean_nll']:.4f}")
    print(f"   Test Mean NLL:       {test_nll_results['mean_nll']:.4f}")
    print(f"   Difference:          {mean_diff:+.4f}")
    print(f"   Status:              {status}")
    print("─" * 70)


In [ ]:

# ═══════════════════════════════════════════════════════════════════════════════
# COMPONENT-WISE NLL ANALYSIS
# ═══════════════════════════════════════════════════════════════════════════════

def analyze_mixture_components(model, loader, num_batches=50):
    """
    Analyze which mixture components contribute most to the NLL.
    
    Args:
        model: Trained MDN model
        loader: tf.data.Dataset
        num_batches: Number of batches to analyze
        
    Returns:
        Dictionary with component analysis
    """
    component_contributions = []
    mixture_weights = []
    
    batch_count = 0
    
    for X_batch, y_batch in loader:
        if batch_count >= num_batches:
            break
        
        # Get predictions
        preds = model(X_batch, training=False)
        pi = preds['pi'].numpy()        # (B, K)
        mu = preds['mu'].numpy()        # (B, K, output_dim)
        sigma = preds['sigma'].numpy()  # (B, K, output_dim)
        
        y_true = y_batch.numpy()        # (B, output_dim)
        
        # Compute PDF for each component
        y_expanded = np.expand_dims(y_true, axis=1)  # (B, 1, output_dim)
        exponent = -0.5 * np.sum(((y_expanded - mu) / sigma) ** 2, axis=-1)  # (B, K)
        normalization = 1.0 / (np.sqrt(2 * np.pi) ** 2 * np.prod(sigma, axis=-1))  # (B, K)
        pdf = normalization * np.exp(exponent)  # (B, K)
        
        # Component contributions: pi_k * pdf_k
        contrib = pi * pdf  # (B, K)
        component_contributions.append(contrib)
        mixture_weights.append(pi)
        
        batch_count += 1
    
    # Aggregate results
    all_contrib = np.concatenate(component_contributions, axis=0)  # (N_samples, K)
    all_weights = np.concatenate(mixture_weights, axis=0)         # (N_samples, K)
    
    # Component statistics
    mean_contrib = np.mean(all_contrib, axis=0)
    std_contrib = np.std(all_contrib, axis=0)
    mean_weight = np.mean(all_weights, axis=0)
    std_weight = np.std(all_weights, axis=0)
    
    # Which components are most active
    active_components = np.argsort(-mean_contrib)
    
    return {
        'mean_contributions': mean_contrib,
        'std_contributions': std_contrib,
        'mean_weights': mean_weight,
        'std_weights': std_weight,
        'active_components': active_components,
        'all_contributions': all_contrib,
        'all_weights': all_weights
    }


def plot_component_analysis(analysis_results):
    """
    Visualize mixture component analysis.
    """
    K = len(analysis_results['mean_weights'])
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # ─────────────────────────────────────────────────────────────────────
    # 1. Component Contributions
    # ─────────────────────────────────────────────────────────────────────
    components = np.arange(K)
    mean_contrib = analysis_results['mean_contributions']
    std_contrib = analysis_results['std_contributions']
    
    axes[0].bar(components, mean_contrib, yerr=std_contrib, 
               capsize=5, alpha=0.7, color='steelblue', edgecolor='black')
    axes[0].set_xlabel('Mixture Component (k)', fontsize=11)
    axes[0].set_ylabel('Mean Contribution (π_k × PDF_k)', fontsize=11)
    axes[0].set_title('Component Contributions to NLL', fontsize=12, fontweight='bold')
    axes[0].grid(True, alpha=0.3, axis='y')
    axes[0].set_xticks(components)
    
    # ─────────────────────────────────────────────────────────────────────
    # 2. Mixture Weights Distribution
    # ─────────────────────────────────────────────────────────────────────
    mean_weights = analysis_results['mean_weights']
    std_weights = analysis_results['std_weights']
    
    axes[1].bar(components, mean_weights, yerr=std_weights,
               capsize=5, alpha=0.7, color='coral', edgecolor='black')
    axes[1].set_xlabel('Mixture Component (k)', fontsize=11)
    axes[1].set_ylabel('Mean Mixture Weight (π_k)', fontsize=11)
    axes[1].set_title('Learned Mixture Weights', fontsize=12, fontweight='bold')
    axes[1].grid(True, alpha=0.3, axis='y')
    axes[1].set_xticks(components)
    
    plt.tight_layout()
    plt.show()
    
    # Print component ranking
    print("\n📊 MOST ACTIVE MIXTURE COMPONENTS:")
    print("─" * 60)
    active = analysis_results['active_components']
    contributions = analysis_results['mean_contributions']
    weights = analysis_results['mean_weights']
    
    for rank, comp_id in enumerate(active[:5], 1):
        print(f"   Rank {rank}: Component {comp_id}")
        print(f"      Mean Contribution: {contributions[comp_id]:.6f}")
        print(f"      Mean Weight (π):   {weights[comp_id]:.4f}")
    print("─" * 60)


# ─────────────────────────────────────────────────────────────────────────
# Component Analysis
# ─────────────────────────────────────────────────────────────────────────

if _model_loaded and MODEL_TYPE == "MDN_TCN":
    print("\n" + "=" * 70)
    print("🔍 MIXTURE COMPONENT ANALYSIS")
    print("=" * 70)
    
    # Analyze components
    component_analysis = analyze_mixture_components(
        loaded_mdn_model,
        val_loader.take(50),
        num_batches=50
    )
    
    # Visualize
    plot_component_analysis(component_analysis)
    
    # Summary
    print("\n📈 Component Summary:")
    print(f"   Total components: {MDN_NUM_MIXTURES}")
    print(f"   Average component weight: {np.mean(component_analysis['mean_weights']):.4f}")
    print(f"   Max component weight: {np.max(component_analysis['mean_weights']):.4f}")
    print(f"   Min component weight: {np.min(component_analysis['mean_weights']):.4f}")


## MDN Negative Log-Likelihood (NLL) Evaluation Guide

### What is NLL?

**Negative Log-Likelihood (NLL)** measures how well the model's predicted probability distribution matches the actual data:

$$\text{NLL} = -\log\left(\sum_{k=1}^{K} \pi_k \mathcal{N}(y \mid \mu_k, \sigma_k)\right)$$

Where:
- $\pi_k$ = mixture weight for component $k$ (probability of choosing component $k$)
- $\mu_k$ = mean of Gaussian component $k$
- $\sigma_k$ = standard deviation of Gaussian component $k$
- $\mathcal{N}$ = Gaussian (normal) probability density function
- $K$ = number of mixture components

### Interpretation

| Metric | Meaning | Good Range |
|--------|---------|-----------|
| **Mean NLL** | Average negative log-likelihood across samples | Lower is better (ideally < 2.0) |
| **Std NLL** | Standard deviation (variability) of NLL | Lower indicates consistent predictions |
| **Min NLL** | Best prediction (lowest probability loss) | Shows model's best case |
| **Max NLL** | Worst prediction (highest probability loss) | Shows model's failure cases |
| **Median NLL** | Middle value (robust to outliers) | Should be close to mean |

### Model Quality Indicators

1. **If Mean NLL is Low** ✅
   - Model assigns high probability to correct targets
   - Good uncertainty quantification
   
2. **If Std NLL is Low** ✅
   - Consistent predictions across samples
   - Reliable model behavior

3. **If Val NLL ≈ Test NLL** ✅
   - Good generalization (no overfitting)
   - Model should perform well on new data

4. **If Test NLL >> Val NLL** ⚠️
   - Possible overfitting
   - Model memorized training data

### Component Analysis

The mixture components analysis shows:
- **Which components are learned** (non-zero weights)
- **Component contributions** to likelihood
- **Multi-modality degree** (how many modes are used)

A good MDN should:
- Use multiple components (not collapse to single Gaussian)
- Have balanced component weights
- Show diverse means/variances across components


## TCN Classification Model load


In [ ]:
from tensorflow.keras.models import load_model
import joblib

# You must provide the custom classes AND custom functions in a dictionary
custom_objects = {
    'TCN': TCN,
    'ResidualBlock': ResidualBlock,
    # 'spatial_loss': spatial_loss,  # for Model(classi)(3s)
    # "loss": spatial_loss, # for best_classification_model & TCN-Classification(3s)
    "loss":'sparse_categorical_crossentropy', # for best_classification_model & TCN-Classification (3s)
    'FP16SafeSparseTopKAccuracy': FP16SafeSparseTopKAccuracy #for best_classification_model & TCN-Classification (3s)
    
}
# model_save_path= 'TCN-Classification/classification_model_V3_64Z_24_0.017MC.keras'
# model_save_path= '../TCN-Classification/classification_model_V3_64Z_24_0.0172MC.keras'
# model_save_path= 'TCN-Classification/classification_model_V2_64Z_24.keras'
# model_save_path= 'TCN-Classification/classification_model_V2_27Z_42_50F_0.058MC(best).keras'
# model_save_path= 'TCN-Classification/classification_model_V2_27Z_42_50F_0.060MC(best).keras'

model_save_path= '../TCN-Regression/best_model_mae.keras'
# model_save_path= 'testings/testings/classi_model_9_best/best_classification_model_V2_27Z_24.keras'
# model_save_path= '../testings/testings/classi_model_11_1s_best/best_classification_model.keras'



reloaded_model_Classification = load_model(model_save_path, custom_objects=custom_objects,safe_mode=False )
# reloaded_model_Classification.summary()

  # your architecture code



print("Model reloaded successfully!")

In [ ]:
reloaded_model_Classification.summary()
gc.collect()

In [ ]:
# ==== PLOT TRAINING HISTORY ====
# Access the actual dictionary inside the History object
history_dict = training_history.history

plt.figure(figsize=(15, 5))

# --- 1. Training & Validation Loss (regression) ---
plt.subplot(1, 2, 1)
# Use the dictionary to get the values.
# Note: Keras usually uses 'loss' and 'val_loss' (not 'train_loss')
plt.plot(history_dict.get('loss', []), label='Train Loss', color='blue')

val_loss = history_dict.get('val_loss', None)
if val_loss is not None:
    plt.plot(val_loss, label='Val Loss', color='red')

plt.title('Training & Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss (MSE)')
plt.legend()
plt.grid(True)

# --- 2. MAE Metrics (Since you are doing Coordinates) ---
if CO_ORDINATES:
    plt.subplot(1, 2, 2)
    plt.plot(history_dict.get('mae', []), label='Train MAE', color='blue')
    plt.plot(history_dict.get('val_mae', []), label='Val MAE', color='red')
    plt.title('Mean Absolute Error (Pitch Units)')
    plt.xlabel('Epoch')
    plt.ylabel('MAE')
    plt.legend()
    plt.grid(True)

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf

def inspect_predictions_backend(
    model,
    loader,
    backend='keras',
    top_k=3,
    num_samples=256,
    coordinates=CO_ORDINATES,
    n_cols=N_COLS
):
    """
    Inspect predictions on first batch of validation data.
    Supports:
        - Classification (zone prediction)
        - Coordinate regression (dx, dy)
    """

    # Get first batch
    sample_input, sample_target = next(iter(loader))

    # ------------------------
    # KERAS BACKEND
    # ------------------------
    if backend == 'keras':

        # Convert tensors to numpy
        if isinstance(sample_input, tf.Tensor):
            sample_input = sample_input.numpy()

        if isinstance(sample_target, tf.Tensor):
            sample_target = sample_target.numpy()
        elif isinstance(sample_target, dict):
            sample_target = list(sample_target.values())[0].numpy()

        preds_np = model(sample_input, training=False).numpy()

        # ============================================================
        # REGRESSION MODE (Coordinates)
        # ============================================================
        if coordinates:
            targets_np = sample_target

            results = pd.DataFrame({
                'True_dx': targets_np[:num_samples, 0],
                'Pred_dx': preds_np[:num_samples, 0],
                'True_dy': targets_np[:num_samples, 1],
                'Pred_dy': preds_np[:num_samples, 1]
            })

            results['Error_X'] = abs(results['True_dx'] - results['Pred_dx'])
            results['Error_Y'] = abs(results['True_dy'] - results['Pred_dy'])
            mae_x = results["Error_X"].mean()
            mae_y = results["Error_Y"].mean()
            

            print(f"\n--- SANITY CHECK (First {num_samples} Samples) ---")
            print(results.round(4))

            print("\nPrediction Statistics:")
            print(f"Max Pred Value: {preds_np.max():.4f}")
            print(f"Min Pred Value: {preds_np.min():.4f}")
            print(f"Avg Pred Value: {preds_np.mean():.4f}")
            print("MAE X:", mae_x)
            print("MAE Y:", mae_y)

            return results

        # ============================================================
        # CLASSIFICATION MODE
        # ============================================================
        else:
            # Convert logits → probabilities
            probs_np = tf.nn.softmax(preds_np, axis=1).numpy()

            # Predicted class
            pred_class = np.argmax(probs_np, axis=1)

            # True class
            if sample_target.ndim > 1:
                true_class = np.argmax(sample_target, axis=1)
            else:
                true_class = sample_target.squeeze()

            # Top-K
            topk_indices = np.argsort(probs_np, axis=1)[:, -top_k:][:, ::-1]
            topk_probs = np.take_along_axis(probs_np, topk_indices, axis=1)

            pred_confidence = probs_np[np.arange(len(pred_class)), pred_class]

            results = pd.DataFrame({
                "True_Zone": true_class[:num_samples],
                "Pred_Zone": pred_class[:num_samples],
                "Pred_Prob": pred_confidence[:num_samples],
                "TopK_Zones": [topk_indices[i] for i in range(num_samples)],
                "TopK_Probs": [topk_probs[i] for i in range(num_samples)]
            })

            # Optional grid distance
            if n_cols is not None:
                def zone_to_rc(zones):
                    return np.stack([zones // n_cols, zones % n_cols], axis=1)

                true_rc = zone_to_rc(true_class[:num_samples])
                pred_rc = zone_to_rc(pred_class[:num_samples])
                grid_distances = np.abs(true_rc - pred_rc).sum(axis=1)

                results["Grid_Distance"] = grid_distances

            # Print results
            print(f"\n--- SANITY CHECK (First {num_samples} Samples) ---")
            print(results.round(4))

            print("\nConfidence Statistics:")
            print(f"Mean Confidence: {pred_confidence[:num_samples].mean():.4f}")
            print(f"Max Confidence: {pred_confidence[:num_samples].max():.4f}")
            print(f"Min Confidence: {pred_confidence[:num_samples].min():.4f}")
            print("Number of classes:", probs_np.shape[1])
            print("Random chance:", 1 / probs_np.shape[1])

            if n_cols is not None:
                print(f"Mean Grid Distance: {results['Grid_Distance'].mean():.2f}")
                print(f"Max Grid Distance: {results['Grid_Distance'].max()}")

            return results     


results_insp = inspect_predictions_backend(reloaded_model_Classification, test_loader.take(100), backend='keras', top_k=5)


In [ ]:
def constant_position_baseline(loader, use_last_n=3):
    """
    Constant Position Baseline:
    Predict the average of the last `use_last_n` frames as future position.

    Args:
        loader: tf.data.Dataset yielding (x_seq, y_true)
        use_last_n: int, number of last frames to average
    Returns:
        mae_x, mae_y: mean absolute error per axis
    """
    maes_x = []
    maes_y = []

    for x_seq, y_true in loader:
        # x_seq shape: (batch, seq_len, features)
        # assume last two features are x,y
        x_seq = x_seq.numpy()
        y_true = y_true.numpy()
        
        # Take last `use_last_n` frames and compute mean
        last_n_frames = x_seq[:, -use_last_n:, -2:]  # shape: (batch, use_last_n, 2)
        pred = last_n_frames.mean(axis=1)           # average across last N frames → shape: (batch, 2)

        error = np.abs(pred - y_true)
        maes_x.append(error[:, 0])
        maes_y.append(error[:, 1])

    mae_x = np.mean(np.concatenate(maes_x))
    mae_y = np.mean(np.concatenate(maes_y))
    return mae_x, mae_y
def constant_velocity_baseline(loader, horizon_frames):
    maes_x = []
    maes_y = []

    for x_seq, y_true in loader:
        x_seq = x_seq.numpy()
        y_true = y_true.numpy()

        x_idx = 0
        y_idx = 1

        x_t_std = x_seq[:, -1, [x_idx, y_idx]]
        x_prev_std = x_seq[:, -2, [x_idx, y_idx]]

        mean_xy  = scaler.mean_[[x_idx, y_idx]]
        scale_xy = scaler.scale_[[x_idx, y_idx]]

        x_t = x_t_std * scale_xy + mean_xy
        x_prev = x_prev_std * scale_xy + mean_xy
        x_t = x_t_std * scale_xy + mean_xy
        x_prev = x_prev_std * scale_xy + mean_xy

        velocity = x_t - x_prev  # per frame velocity
        pred = x_t + horizon_frames * velocity

        # optional: clip to [0,1]
        pred = np.clip(pred, 0.0, 1.0)

        error = np.abs(pred - y_true)

        maes_x.append(error[:, 0])
        maes_y.append(error[:, 1])

    return np.mean(np.concatenate(maes_x)), np.mean(np.concatenate(maes_y))
if CO_ORDINATES and MODEL_TYPE != "MDN_TCN":
    take_amt=100
    mae_x_baseline, mae_y_baseline = constant_position_baseline(test_loader.take(take_amt), SEQ_LEN)
    mae_x_baseline_velo, mae_y_baseline_velo =constant_velocity_baseline(test_loader.take(take_amt), horizon_frames=HORIZON_FRAMES)

    print(f"Constant Position Baseline MAE - X: {mae_x_baseline:.4f}, Y: {mae_y_baseline:.4f}")
    print(f"Constant Velocity Baseline MAE - X: {mae_x_baseline_velo:.4f}, Y: {mae_y_baseline_velo:.4f}")

In [ ]:
for x_seq, y_true in test_loader.take(1):
   print("Feature dim:", x_seq.shape[-1])
   def label_feature_indices(feature_cols, feature_values=None):
       idx_to_feature = {i: name for i, name in enumerate(feature_cols)}
       for i, name in idx_to_feature.items():
           if feature_values is None:
               print(f"[{i}] {name}")
           else:
               print(f"[{i}] {name}: {float(feature_values[i]):.6f}")
       return idx_to_feature

   feature_index_map = label_feature_indices(FEATURE_COLS, x_seq[0, -1])



In [ ]:
import numpy as np

def compute_ade_fde(loader, model, num_samples=20):
    """
    Compute ADE (Average Displacement Error) and FDE (Final Displacement Error)
    for a regression model predicting coordinates (dx, dy).

    Args:
        loader: tf.data.Dataset yielding (x_seq, y_true)
        model: Keras model
        coordinates: True if regression task (dx, dy)
        num_samples: limit number of samples (optional)

    Returns:
        ade: average displacement error
        fde: final displacement error (same as ADE for single-step prediction)
    """
    all_errors = []

    for batch_idx, (x_seq, y_true) in enumerate(loader):
        # Limit number of samples if requested
        if num_samples is not None and batch_idx * x_seq.shape[0] >= num_samples:
            break

        # Convert tensors to numpy
        if isinstance(x_seq, tf.Tensor):
            x_seq = x_seq.numpy()
        if isinstance(y_true, tf.Tensor):
            y_true = y_true.numpy()
        elif isinstance(y_true, dict):
            y_true = list(y_true.values())[0].numpy()

        # Forward pass
        preds = model(x_seq, training=False).numpy()  # shape: (batch, 2)

        # Compute Euclidean distance per sample
        errors = np.linalg.norm(preds - y_true, axis=1)
        all_errors.append(errors)

    all_errors = np.concatenate(all_errors)

    ade = all_errors.mean()  # average displacement error
    fde = all_errors[-1]     # final displacement error (last sample in loader)

    return ade, fde

if CO_ORDINATES :
    ade, fde = compute_ade_fde(test_loader, reloaded_model_Classification)
    print("ADE:", ade, "FDE:", fde)

In [ ]:
gc.collect()

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, balanced_accuracy_score, classification_report

import numpy as np


# ==== EVALUATION SECTION ====
print("\n" + "="*80)
print("📊 MODEL EVALUATION ON TEST SET")
print("="*80)

model=reloaded_model_Classification
# Load best model
if BACKEND == "keras":
    try:
        with strategy.scope():
            
            model = model
        print(f"✅ Loaded best Keras model")
    except:
        print("⚠️  Using trained model (best model file not found)")
        model = trained_model

# Evaluate model

def evaluate_model_unified(model, loader, backend='keras', coordinate_mode=CO_ORDINATES, print_every=10):
    
    all_preds = []
    all_labels = []
    
    if backend == 'keras':
        print("Evaluating Keras model...\n")

    for batch_idx, (batch_X, batch_y) in enumerate(loader, start=1):
        
        # Get predictions for this batch
        preds = model.predict(batch_X, verbose=0)
        
        # Convert labels to numpy
        labels = batch_y.numpy()
        
        all_preds.append(preds)
        all_labels.append(labels)
        
        # ---- Classification Mode ----
        if not coordinate_mode:
            pred_classes = np.argmax(preds, axis=1)
            true_classes = np.argmax(labels, axis=1) if labels.ndim > 1 else labels
            
            batch_acc = accuracy_score(true_classes, pred_classes)
            
            if batch_idx % print_every == 0:
                print(f"Batch {batch_idx} | Batch Accuracy: {batch_acc:.4f}")
    
    # Concatenate everything
    all_preds = np.concatenate(all_preds, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)

    # ================================
    # FINAL METRICS
    # ================================
    
    if coordinate_mode:
        mae = np.mean(np.abs(all_preds - all_labels))
        mse = np.mean((all_preds - all_labels) ** 2)
        mae_x = np.mean(np.abs(all_preds[:, 0] - all_labels[:, 0]))
        mae_y = np.mean(np.abs(all_preds[:, 1] - all_labels[:, 1]))  # Corrected: should be all_labels[:, 1] for Y

        rmse = np.sqrt(mse)

        print("\n📊 Final Coordinate Metrics:")
        print(f"  Overall MAE: {mae:.4f}")
        print(f"  Overall RMSE: {rmse:.4f}")

        return {
            'mae': mae,
            'rmse': rmse,
            'predictions': all_preds,
            'mae_y': mae_y,
            'mae_x': mae_x,
            'labels': all_labels
        }
    
    else:
        pred_classes = np.argmax(all_preds, axis=1)
        true_classes = np.argmax(all_labels, axis=1) if all_labels.ndim > 1 else all_labels
        
        accuracy = accuracy_score(true_classes, pred_classes)
        macro_f1 = f1_score(true_classes, pred_classes, average='macro')
        balanced_acc = balanced_accuracy_score(true_classes, pred_classes)
        

        print("\n📊 Final Test Accuracy:")
        print(f"  Test Accuracy: {accuracy:.4f}")
        print(f"Macro F1 Score: {macro_f1:.4f}")
        print(f"Balanced Accuracy: {balanced_acc:.4f}")
        print(classification_report(true_classes, pred_classes, zero_division=0))

        return {
            'accuracy': accuracy,
            'predictions': pred_classes,
            'labels': true_classes,
            'probabilities': all_preds
        }

# Run evaluation
results = evaluate_model_unified(model, val_loader.take(100), backend=BACKEND, coordinate_mode=CO_ORDINATES)


print("\n" + "="*80)
print("✅ EVALUATION COMPLETE")
print("="*80)

In [ ]:
results.keys()
results['mae_x'], results['mae_y']
# results['predictions'].shape, results['labels'].shape

In [ ]:
from sklearn.metrics import top_k_accuracy_score
y_true = results['labels']
y_score = results['probabilities']  # shape (n_samples, 64)

all_labels = np.arange(y_score.shape[1])
top3_acc = top_k_accuracy_score(y_true, y_score, k=3, labels=all_labels)
top5_acc = top_k_accuracy_score(y_true, y_score, k=5, labels=all_labels)

print("Top-3 Accuracy:", top3_acc)

print("Top-5 Accuracy:", top5_acc)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix
# ==== VISUALIZATION OF RESULTS ====

def visualize_predictions_Classification(results):
    print("\n📊 Generating Confusion Matrix...")

    cm = confusion_matrix(results['labels'], results['predictions'])
    
    # Normalize per row (recall-focused)
    cm = cm.astype("float") / cm.sum(axis=1, keepdims=True)
    cm = np.nan_to_num(cm)

    fig, ax = plt.subplots(figsize=(16, 14))
    im = ax.imshow(cm, interpolation='nearest', cmap='viridis')
    
    cbar = ax.figure.colorbar(im, ax=ax)
    cbar.ax.set_ylabel('Recall (Normalized)', rotation=-90, va="bottom")

    # Show fewer ticks
    ticks = np.arange(0, cm.shape[0], 4)
    ax.set_xticks(ticks)
    ax.set_yticks(ticks)
    ax.set_xticklabels(ticks)
    ax.set_yticklabels(ticks)

    ax.set_title('Normalized Confusion Matrix - Zone Prediction')
    ax.set_ylabel('True Zone')
    ax.set_xlabel('Predicted Zone')

    plt.tight_layout()
    plt.show()


if CO_ORDINATES:
    print("\n📊 Visualizing Coordinate Predictions...")

    # Sample for visualization
    sample_size = min(100, len(results['predictions']))
    sample_indices = np.random.choice(len(results['predictions']), sample_size, replace=False)

    fig, axes = plt.subplots(1, 2, figsize=(15, 6))

    # Scatter plot: Predicted vs True
    axes[0].scatter(results['labels'][sample_indices, 0], results['predictions'][sample_indices, 0],
                   alpha=0.5, label='X-coordinate', s=30)
    axes[0].scatter(results['labels'][sample_indices, 1], results['predictions'][sample_indices, 1],
                   alpha=0.5, label='Y-coordinate', s=30)
    axes[0].plot([0, 1], [0, 1], 'r--', label='Perfect prediction', linewidth=2)
    axes[0].set_xlabel('True Values', fontsize=12)
    axes[0].set_ylabel('Predicted Values', fontsize=12)
    axes[0].set_title('Predicted vs True Coordinates', fontsize=14, fontweight='bold')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Error distribution
    errors = np.sqrt((results['predictions'][:, 0] - results['labels'][:, 0])**2 +
                     (results['predictions'][:, 1] - results['labels'][:, 1])**2)
    axes[1].hist(errors, bins=50, edgecolor='black', alpha=0.7, color='steelblue')
    axes[1].set_xlabel('Euclidean Distance Error', fontsize=12)
    axes[1].set_ylabel('Frequency', fontsize=12)
    axes[1].set_title(f'Prediction Error Distribution\nMean Error: {errors.mean():.4f}',
                     fontsize=14, fontweight='bold')
    axes[1].grid(True, alpha=0.3, axis='y')

    plt.tight_layout()
    # plt.savefig(f'{output_dir}/prediction_visualization.png', dpi=150, bbox_inches='tight')
    plt.show()

    # print(f"✅ Visualization saved to {output_dir}/prediction_visualization.png")
else:
    visualize_predictions_Classification(results)
    


In [ ]:
mae = np.mean(np.abs(results['predictions'] - results['labels']))
print(f"Overall MAE: {mae:.4f}")
within_1 = np.mean(np.abs(results['labels'] - results['predictions'] ) <= 1)
print(f"Percentage of predictions within 1 zone: {within_1:.2%}")

In [ ]:
# ==== FINAL SUMMARY ====
print("\n" + "="*80)
print("🏆 TRAINING & EVALUATION SUMMARY")
print("="*80)

print("\n📊 Dataset Information:")
print(f"  • Total sequences: {len(train_df) + len(val_df) + len(test_df):,}")
print(f"  • Training: {len(train_df):,}")
print(f"  • Validation: {len(val_df):,}")
print(f"  • Test: {len(test_df):,}")

print("\n🧠 Model Configuration:")
print(f"  • Model Type: {MODEL_TYPE}")
print(f"  • Backend: {BACKEND}")
print(f"  • Sequence Length: {SEQ_LEN} frames")
print(f"  • Horizon: {HORIZON_SECONDS}s ({HORIZON_FRAMES} frames)")
print(f"  • Features: {len(FEATURE_COLS)}")
print(f"  • Prediction Mode: {'Coordinates (Regression)' if CO_ORDINATES else 'Zones (Classification)'}")

print("\n📈 Training Configuration:")
print(f"  • Batch Size: {BATCH_SIZE}")
if BACKEND == 'keras':
    print(f"  • Global Batch Size (multi-GPU): {GLOBAL_BATCH_SIZE}")
print(f"  • Learning Rate: {LR}")
# if BACKEND == 'keras':
#     print(f"  • Epochs Completed: {len(training_history.history.get('loss', []))}")
# else:
#     print(f"  • Epochs Completed: {len(training_history.get('train_loss', []))}")

print("\n🎯 Final Performance:")
if CO_ORDINATES:
    print(f"  • Test MAE: {results['mae']:.4f}")
    print(f"  • Test RMSE: {results['rmse']:.4f}")
    print(f"  • X-coordinate MAE: {results['mae_x']:.4f}")
    print(f"  • Y-coordinate MAE: {results['mae_y']:.4f}")
else:
    print(f"  • Test Accuracy: {results['accuracy']:.4f} ({results['accuracy']*100:.2f}%)")

print("\n💾 Saved Files:")

print("\n" + "="*80)
print("✅ ALL PROCESSING COMPLETE!")
print("="*80)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_heatmap_matplotlib(data,
                            title="Heatmap",
                            xlabel="",
                            ylabel="",
                            cmap="YlOrRd",
                            annot=True,
                            fmt=".3f",
                            cbar_label="",
                            figsize=(8, 6),
                            ax=None):
    """
    Seaborn-like heatmap using pure matplotlib.
    """

    if ax is None:
        fig, ax = plt.subplots(figsize=figsize)

    im = ax.imshow(data, cmap=cmap, origin="upper", aspect="auto")

    # Colorbar
    cbar = plt.colorbar(im, ax=ax)
    if cbar_label:
        cbar.set_label(cbar_label)

    # Ticks
    ax.set_xticks(np.arange(data.shape[1]))
    ax.set_yticks(np.arange(data.shape[0]))

    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontsize=14, fontweight='bold')

    # Annotation
    if annot:
        for i in range(data.shape[0]):
            for j in range(data.shape[1]):
                ax.text(j, i,
                        format(data[i, j], fmt),
                        ha="center",
                        va="center",
                        color="black" if data[i, j] < data.max()/2 else "white",
                        fontsize=9)

    plt.tight_layout()
    return ax
def plot_zone_probability_grid(prob_grid,
                               title="Zone Probability Heatmap",
                               xlabel="Field Width (Zones)",
                               ylabel="Field Length (Zones)",
                               cmap="YlOrRd"):
    """
    Styled zone probability grid using matplotlib only.
    """

    fig, ax = plt.subplots(figsize=(10, 6))

    im = ax.imshow(prob_grid, cmap=cmap, origin='upper')

    # Add grid lines
    ax.set_xticks(np.arange(-0.5, prob_grid.shape[1], 1), minor=True)
    ax.set_yticks(np.arange(-0.5, prob_grid.shape[0], 1), minor=True)
    ax.grid(which="minor", color="white", linestyle='-', linewidth=1)
    ax.tick_params(which="minor", bottom=False, left=False)

    # Annotate
    for i in range(prob_grid.shape[0]):
        for j in range(prob_grid.shape[1]):
            ax.text(j, i,
                    f"{prob_grid[i, j]:.3f}",
                    ha="center",
                    va="center",
                    fontsize=8,
                    color="black" if prob_grid[i, j] < prob_grid.max()/2 else "white")

    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label("Probability")

    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontsize=14, fontweight="bold")

    plt.tight_layout()
    plt.show()
def plot_topk_probabilities(top_zones, top_probs):
    """
    Seaborn-style bar plot for top-k probabilities using matplotlib.
    """

    fig, ax = plt.subplots(figsize=(8, 4))

    bars = ax.bar(range(len(top_zones)), top_probs)

    ax.set_xticks(range(len(top_zones)))
    ax.set_xticklabels([f"Zone {z}" for z in top_zones])
    ax.set_ylabel("Probability")
    ax.set_title("Top-K Zone Predictions", fontweight="bold")

    # Annotate bars
    for bar, prob in zip(bars, top_probs):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height(),
                f"{prob:.3f}",
                ha='center',
                va='bottom')

    plt.tight_layout()
    plt.show()
def draw_field(ax, n_rows, n_cols):
    """
    Draw normalized football field with zone grid.
    """

    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_facecolor('#2d8a2d')

    # Outer border
    ax.add_patch(plt.Rectangle((0, 0), 1, 1,
                               fill=False,
                               edgecolor='white',
                               linewidth=2))

    # Mid line
    ax.axvline(0.5, color='white', linestyle='--', alpha=0.6)

    # Zone lines
    for r in range(1, n_rows):
        ax.axhline(r / n_rows, color='white', linewidth=0.5, alpha=0.3)

    for c in range(1, n_cols):
        ax.axvline(c / n_cols, color='white', linewidth=0.5, alpha=0.3)

    ax.set_aspect('equal')


## UNIFIED PREDICTION FUNCTIONS

In [ ]:


# ============================================================================
# UNIFIED PREDICTION FUNCTIONS
# ============================================================================
# Supports: MDN_TCN (Keras), Keras_tcn (classification/regression), PyTorch
# ============================================================================



import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import seaborn as sns
from matplotlib.patches import Ellipse, Arc  # Correct import for Arc


print(f"Using {len(FEATURE_COLS)} features for prediction (same as training)")
print(f"Model Type: {MODEL_TYPE} | Backend: {BACKEND} | Coordinates: {CO_ORDINATES}")

def get_continuous_ranges(frame_pairs):
    frames = sorted(set(f[0] for f in frame_pairs))

    ranges = []
    start = frames[0]
    prev = frames[0]

    for f in frames[1:]:
        if f != prev + 1:
            ranges.append((start, prev))
            start = f
        prev = f

    ranges.append((start, prev))
    return ranges

def get_player_frame_ranges(df, match_id, player_id):
    player_data = df[
        (df["match_id"] == match_id) &
        (df["player_id"] == player_id)
    ].sort_values("frame_number")

    if player_data.empty:
        return []

    frames = player_data["frame_number"].values

    ranges = []
    start = frames[0]
    prev = frames[0]

    for f in frames[1:]:
        if f != prev + 1:  # gap detected
            ranges.append((int(start), int(prev)))
            start = f
        prev = f

    # append final range
    ranges.append((int(start), int(prev)))

    return get_continuous_ranges(ranges)

def get_player_time_ranges_seconds(df, match_id, player_id, fps=10):
    player_data = df[
        (df["match_id"] == match_id) &
        (df["player_id"] == player_id)
    ].sort_values("frame_number")

    if player_data.empty:
        return {}

    frames = player_data["frame_number"]

    # Identify gaps in continuity
    group_id = (frames.diff() != 1).cumsum()

    ranges = (
        player_data
        .groupby(group_id)["frame_number"]
        .agg(["min", "max"])
        .values
    )

    # Convert to seconds and store in dict
    time_ranges = {}

    for i, (start, end) in enumerate(ranges):
        time_ranges[f"segment_{i+1}"] = {
            "start_second": round(start / fps, 2),
            "end_second": round(end / fps, 2),
            "duration_seconds": round((end - start) / fps, 2)
        }

    return time_ranges

def get_player_sequence_features(
    df,
    match_id,
    player_id,
    start_time_seconds=None,
    seq_len=SEQ_LEN
):

    player_data = df[
        (df["match_id"] == match_id) &
        (df["player_id"] == player_id)
    ].copy()

    if len(player_data) < seq_len:
        print(f"Insufficient data for player {player_id} in match {match_id}. "
              f"Need {seq_len} frames, have {len(player_data)}")
        return None, None

    player_data = player_data.sort_values("frame_number").reset_index(drop=True)

    if start_time_seconds is None:
        start_idx = len(player_data) - seq_len

    else:
        if "timestamp_seconds" not in player_data.columns:
            print("Error: 'timestamp_seconds' column not found.")
            return None, None

        timestamps = player_data["timestamp_seconds"].values

        # Find nearest timestamp index
        nearest_idx = np.abs(timestamps - start_time_seconds).argmin()

        # The last possible valid start index
        max_valid_start = len(player_data) - seq_len

        if nearest_idx <= max_valid_start:
            start_idx = nearest_idx
        else:
            # Move forward to the last possible valid start
            if max_valid_start < 0:
                print("No valid starting point available.")
                return None, None
            start_idx = max_valid_start

    end_idx = start_idx + seq_len
    sequence_data = player_data.iloc[start_idx:end_idx]

    # Feature selection
    available_features = [col for col in FEATURE_COLS if col in sequence_data.columns]
    features_to_use = available_features if len(available_features) < len(FEATURE_COLS) else FEATURE_COLS

    features = sequence_data[features_to_use].values.astype(np.float32)

    current_frame = int(sequence_data.iloc[-1]["frame_number"])
    prediction_frame = current_frame + HORIZON_FRAMES

    time_info = {
        "current_frame": current_frame,
        "prediction_frame": prediction_frame,
        "sequence_start_frame": int(sequence_data.iloc[0]["frame_number"]),
        "sequence_end_frame": current_frame,
    }

    if "timestamp_seconds" in sequence_data.columns:
        time_info["sequence_start_timestamp_sec"] = float(sequence_data.iloc[0]["timestamp_seconds"])
        time_info["current_timestamp_sec"] = float(sequence_data.iloc[-1]["timestamp_seconds"])

    if "x_normalized" in sequence_data.columns and \
       "y_normalized" in sequence_data.columns:
        time_info["current_x"] = float(sequence_data.iloc[-1]["x_normalized"])
        time_info["current_y"] = float(sequence_data.iloc[-1]["y_normalized"])

    predicted_frame_data = player_data[
        player_data["frame_number"] == prediction_frame
    ]

    if not predicted_frame_data.empty and \
       "x_normalized" in predicted_frame_data.columns and \
       "y_normalized" in predicted_frame_data.columns:

        predicted_x_normalized = float(predicted_frame_data.iloc[0]["x_normalized"])
        predicted_y_normalized = float(predicted_frame_data.iloc[0]["y_normalized"])

        time_info["predicted_true_x"] = predicted_x_normalized * 105
        time_info["predicted_true_y"] = predicted_y_normalized * 68

    return features, time_info
def predict_for_player(model, df, match_id, player_id, scaler=None,
                       start_frame=None, top_k=5, return_all_probs=False):
    """
    Unified prediction function for all model types and backends.
    """
    # Get player sequence
    player_sequence, time_info = get_player_sequence_features(
        df, match_id, player_id, start_frame
    )

    if player_sequence is None:
        return None

    # Apply scaling
    if scaler is not None:
        seq_len, num_features = player_sequence.shape
        if num_features != scaler.n_features_in_:
            print(f"⚠️  Feature mismatch! Sequence: {num_features}, Scaler: {scaler.n_features_in_}")
            return None
        player_sequence_flat = player_sequence.reshape(-1, num_features)
        player_sequence_scaled = scaler.transform(player_sequence_flat)
        player_sequence = player_sequence_scaled.reshape(seq_len, num_features)

    # ─── MDN_TCN ─────────────────────────────────────────────────────────────
    if MODEL_TYPE == "MDN_TCN":
        input_tensor = np.expand_dims(player_sequence, axis=0)
        outputs = model({"sequence_input": input_tensor}, training=False)

        pi = outputs['pi'].numpy()[0]
        mu = outputs['mu'].numpy()[0]
        sigma = outputs['sigma'].numpy()[0]

        sorted_indices = np.argsort(pi)[::-1][:top_k]

        predictions = []
        for idx in sorted_indices:
            predictions.append({
                'probability': float(pi[idx]),
                'x': float(mu[idx, 0]),
                'y': float(mu[idx, 1]),
                'x_std': float(sigma[idx, 0]),
                'y_std': float(sigma[idx, 1]),
                'component': int(idx),
                'zone': int(xy_to_zone_vectorized(
                    np.array([mu[idx, 0]]),
                    np.array([mu[idx, 1]]),
                    N_ROWS, N_COLS
                )[0])
            })

        expected_pos = np.sum(pi[:, np.newaxis] * mu, axis=0)
        mode_pos = mu[np.argmax(pi)]

        results = {
            'player_id': player_id,
            'match_id': match_id,
            'model_type': 'MDN_TCN',
            'predictions': predictions,
            'expected_position': {'x': float(expected_pos[0]), 'y': float(expected_pos[1])},
            'mode_position': {'x': float(mode_pos[0]), 'y': float(mode_pos[1])},
            'expected_zone': int(xy_to_zone_vectorized(
                np.array([expected_pos[0]]), np.array([expected_pos[1]]), N_ROWS, N_COLS
            )[0]),
            'pi': pi,
            'mu': mu,
            'sigma': sigma,
            'horizon_seconds': HORIZON_SECONDS,
            **time_info
        }
        return results

    # ─── Keras models ────────────────────────────────────────────────────────
    elif BACKEND == 'keras':
        input_tensor = np.expand_dims(player_sequence, axis=0)
        preds = model.predict(input_tensor, verbose=0)

        if CO_ORDINATES:
            pred_x = float(preds[0][0])
            pred_y = float(preds[0][1])
            pred_zone = int(xy_to_zone_vectorized(
                np.array([pred_x]), np.array([pred_y]), N_ROWS, N_COLS
            )[0])

            results = {
                'player_id': player_id,
                'match_id': match_id,
                'model_type': 'Keras_regression',
                'predicted_x': pred_x,
                'predicted_y': pred_y,
                'predicted_zone': pred_zone,
                'predicted_zone_row': pred_zone // N_COLS,
                'predicted_zone_col': pred_zone % N_COLS,
                'horizon_seconds': HORIZON_SECONDS,**time_info
            }
            return results
        else:
            probs = preds[0]
            top_indices = np.argsort(probs)[-top_k:][::-1]
            top_probs = probs[top_indices]

            results = {
                'player_id': player_id,
                'match_id': match_id,
                'model_type': 'Keras_classification',
                'top_zones': top_indices,
                'top_probabilities': top_probs,
                'top_coordinates': [(int(z // N_COLS), int(z % N_COLS)) for z in top_indices],
                'horizon_seconds': HORIZON_SECONDS,
                **time_info
            }
            if return_all_probs:
                results['all_probabilities'] = probs
            return results


def visualize_prediction(prediction_results, title=None):
    """Unified visualization for all prediction types with full football field."""
    if prediction_results is None:
        print("No prediction results to visualize")
        return

    model_type = prediction_results.get('model_type', '')

    # Constants for pitch size (in meters)
    PITCH_LENGTH = 105  # meters (length of the pitch)
    PITCH_WIDTH = 68    # meters (width of the pitch)

    # ─── Initialize Plot ─────────────────────────────────────────────────────
    fig, ax = plt.subplots(1, 1, figsize=(14, 9))
    ax.set_xlim(-1, PITCH_LENGTH + 1)
    ax.set_ylim(-1, PITCH_WIDTH + 1)
    ax.set_facecolor('#2d8a2d')

    # Pitch Boundary
    ax.add_patch(patches.Rectangle((0, 0), PITCH_LENGTH, PITCH_WIDTH, fill=False, edgecolor='white', linewidth=2))

    # Midline
    ax.axvline(x=PITCH_LENGTH / 2, color='white', linewidth=1, linestyle='-', alpha=0.5)

    # Center circle
    center_circle = plt.Circle((PITCH_LENGTH / 2, PITCH_WIDTH / 2), 9.15, color='white', fill=False, linewidth=1)
    ax.add_patch(center_circle)

    # Penalty areas (18-yard boxes)
    ax.add_patch(patches.Rectangle((0, (PITCH_WIDTH - 40) / 2), 16.5, 40, fill=False, color='white', linewidth=2))  # Left penalty box
    ax.add_patch(patches.Rectangle((PITCH_LENGTH - 16.5, (PITCH_WIDTH - 40) / 2), 16.5, 40, fill=False, color='white', linewidth=2))  # Right penalty box

    # Goal areas (6-yard boxes)
    ax.add_patch(patches.Rectangle((0, (PITCH_WIDTH - 20) / 2), 5.5, 20, fill=False, color='white', linewidth=2))  # Left goal box
    ax.add_patch(patches.Rectangle((PITCH_LENGTH - 5.5, (PITCH_WIDTH - 20) / 2), 5.5, 20, fill=False, color='white', linewidth=2))  # Right goal box

    # Goals (6m goalposts)
    ax.plot([0, 0], [(PITCH_WIDTH - 7) / 2, (PITCH_WIDTH + 7) / 2], color='white', lw=5)  # Left goalpost
    ax.plot([PITCH_LENGTH, PITCH_LENGTH], [(PITCH_WIDTH - 7) / 2, (PITCH_WIDTH + 7) / 2], color='white', lw=5)  # Right goalpost

    # Corner arcs (radius 1m)
    ax.add_patch(Arc((0, 0), 2, 2, theta1=0, theta2=90, color='white', lw=2))  # Bottom-left corner arc
    ax.add_patch(Arc((PITCH_LENGTH, 0), 2, 2, theta1=90, theta2=180, color='white', lw=2))  # Bottom-right corner arc
    ax.add_patch(Arc((0, PITCH_WIDTH), 2, 2, theta1=270, theta2=360, color='white', lw=2))  # Top-left corner arc
    ax.add_patch(Arc((PITCH_LENGTH, PITCH_WIDTH), 2, 2, theta1=180, theta2=270, color='white', lw=2))  # Top-right corner arc
    # Check if true_x and true_y are available in prediction_results
    if 'predicted_true_x' in prediction_results and 'predicted_true_y' in prediction_results:
        true_x = prediction_results['predicted_true_x']   # Scale to pitch size
        true_y = prediction_results['predicted_true_y']   # Scale to pitch size
        
        # Plot true positions (using blue color for true positions)
        ax.scatter(true_x, true_y, c='green', s=100, marker='o', edgecolors='black', linewidth=2, zorder=10, label='True Position')
    
    # Add a title and any additional context you'd like
   
    ax.set_xlabel('X (meters)', fontsize=12)
    ax.set_ylabel('Y (meters)', fontsize=12)
    ax.set_aspect('equal')
    ax.legend(loc='upper right', fontsize=10)   
    # ─── Displaying Prediction Based on Model Type ─────────────────────────────

    # ─── MDN Model: Show Uncertainty (Ellipses) ──────────────────────────────
    if 'MDN' in model_type:
        # Draw current position
        if 'current_x' in prediction_results:
            current_x = prediction_results['current_x'] * PITCH_LENGTH
            current_y = prediction_results['current_y'] * PITCH_WIDTH
            ax.scatter(current_x, current_y,
                    c='white', s=200, marker='o',
                    edgecolors='black', linewidth=2,
                    zorder=20, label='Current Position')

        # -----------------------------------------------------------------------
        # 1️⃣ Create grid over pitch (in normalized space 0–1)
        # -----------------------------------------------------------------------
        grid_res_x = 200
        grid_res_y = 130

        x = np.linspace(0, 1, grid_res_x)
        y = np.linspace(0, 1, grid_res_y)
        X, Y = np.meshgrid(x, y)

        density = np.zeros_like(X)

        # -----------------------------------------------------------------------
        # 2️⃣ Compute full MDN mixture density
        # -----------------------------------------------------------------------
        pi = prediction_results['pi']
        mu = prediction_results['mu']
        sigma = prediction_results['sigma']

        for k in range(len(pi)):
            pi_k = pi[k]
            mu_x = mu[k, 0]
            mu_y = mu[k, 1]

            # Numerical stability (very important)
            sigma_x = max(sigma[k, 0], 1e-4)
            sigma_y = max(sigma[k, 1], 1e-4)

            gaussian = (
                1 / (2 * np.pi * sigma_x * sigma_y)
                * np.exp(
                    -(
                        ((X - mu_x) ** 2) / (2 * sigma_x**2)
                        + ((Y - mu_y) ** 2) / (2 * sigma_y**2)
                    )
                )
            )

            density += pi_k * gaussian

        # -----------------------------------------------------------------------
        # 3️⃣ Plot heatmap (blue → red)
        # -----------------------------------------------------------------------
        heatmap = ax.imshow(
            density,
            extent=[0, PITCH_LENGTH, 0, PITCH_WIDTH],
            origin='lower',
            cmap='jet',      # Blue → Green → Yellow → Red
            alpha=0.75,
            aspect='auto'
        )

        # Add colorbar
        cbar = plt.colorbar(heatmap, ax=ax, fraction=0.035, pad=0.02)
        cbar.set_label("Probability Density", fontsize=11)

        # -----------------------------------------------------------------------
        # 4️⃣ Expected position marker
        # -----------------------------------------------------------------------
        exp = prediction_results['expected_position']
        ax.scatter(exp['x'] * PITCH_LENGTH,
                exp['y'] * PITCH_WIDTH,
                c='black', s=120, marker='X',
                edgecolors='white', linewidth=2,
                zorder=25,
                label='Expected Position')

        ax.set_xlabel('X (meters)', fontsize=12)
        ax.set_ylabel('Y (meters)', fontsize=12)
        ax.set_aspect('equal')
        ax.legend(loc='upper right', fontsize=10)
        ax.set_title(title or f"MDN Density Prediction - Player {prediction_results['player_id']}",
             fontsize=14, fontweight='bold')
    # ─── Regression Model: Show Predicted Position without Zones ─────────────
    elif 'regression' in model_type:
        if 'current_x' in prediction_results:
            current_x = prediction_results['current_x'] * PITCH_LENGTH
            current_y = prediction_results['current_y'] * PITCH_WIDTH
            ax.scatter(current_x, current_y, c='yellow', s=200, marker='o', edgecolors='black', linewidth=2,
                       zorder=15, label='Current Position')
            ax.annotate('', xy=(prediction_results['predicted_x'] * PITCH_LENGTH, prediction_results['predicted_y'] * PITCH_WIDTH),
                        xytext=(current_x, current_y),
                        arrowprops=dict(arrowstyle='->', color='white', lw=2))

        ax.scatter(prediction_results['predicted_x'] * PITCH_LENGTH, prediction_results['predicted_y'] * PITCH_WIDTH,
                  c='red', s=250, marker='X', edgecolors='white', linewidth=2,
                  zorder=20, label=f'Predicted ({HORIZON_SECONDS}s ahead)')
        ax.set_xlabel('X (meters)', fontsize=12)
        ax.set_ylabel('Y (meters)', fontsize=12)
        ax.legend(loc='upper right', fontsize=12)
        ax.set_aspect('equal')
        ax.set_title(title or f"Coordinate Prediction - Player {prediction_results['player_id']}",
                     fontsize=14, fontweight='bold')

    # ─── Classification Model: Show Zone Probability Heatmap ────────────────
    else:
        # Initialize a grid to represent the pitch, dividing it into N_ROWS x N_COLS zones
        prob_grid = np.zeros((N_ROWS, N_COLS))
        current_x = prediction_results['current_x'] * PITCH_LENGTH
        current_y = prediction_results['current_y'] * PITCH_WIDTH

        # ─── Step 1: Draw the grid on the pitch ──────────────────────────────────
        grid_width = PITCH_LENGTH / N_COLS  # Width of each cell in meters
        grid_height = PITCH_WIDTH / N_ROWS  # Height of each cell in meters
        
        # Draw the grid lines
        for i in range(1, N_COLS):  # Vertical grid lines
            ax.axvline(x=i * grid_width, color='black', linewidth=1, linestyle='-', alpha=0.6)
        for i in range(1, N_ROWS):  # Horizontal grid lines
            ax.axhline(y=i * grid_height, color='black', linewidth=1, linestyle='-', alpha=0.6)
        
        # ─── Step 2: Plot the top zones and their probabilities ───────────────────
        for zone_idx, prob in zip(prediction_results['top_zones'], prediction_results['top_probabilities']):
            row = zone_idx // N_COLS  # Row index
            col = zone_idx % N_COLS   # Column index
            prob_grid[row, col] = prob
            
            # Compute the center of the cell for each zone
            center_x = (col + 0.5) * grid_width
            center_y = (row + 0.5) * grid_height
            
            # Plot the probability as a colored square or rectangle in the grid
            ax.add_patch(patches.Rectangle(
                (col * grid_width, row * grid_height),
                grid_width, grid_height,
                linewidth=0.5,  # Thinner borders
                edgecolor='white', 
                facecolor=plt.cm.YlOrRd(prob),  # Color based on probability
                alpha=0.5  # Transparency (0.0 = completely transparent, 1.0 = completely opaque)
                ))

            # Annotate the probability in the center of the cell
            ax.annotate(f"{prob*100:.1f}%", 
                        xy=(center_x, center_y), ha='center', va='center',
                        fontsize=10, fontweight='bold', color='black', 
                        bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.5))
   
        ax.add_patch(patches.Rectangle(
            (current_x - 1.5, current_y - 1.5),  # Adjusting position for the rectangle size (1.5 meters on each side)
            2,  # Width of the rectangle
            2,  # Height of the rectangle
            linewidth=1,  # Border thickness
            edgecolor='white', 
            facecolor='red',  # White color for the current position
            alpha=1.0  # Solid white, no transparency
        ))

        # First, create the title text
        title_text = title or f'Zone Predictions - Player {prediction_results["player_id"]}'
        starting_frame_text = f"Starting Frame: {prediction_results['current_frame']}"

        # Combine title and the current zone text
        full_title = f"{title_text} | {starting_frame_text}"

        # Set the title with the current zone and starting frame
        ax.set_title(full_title, fontsize=14, fontweight='bold', loc='left')


        # Add the Prediction Frame info near the title
        ax.text(PITCH_LENGTH - 1, PITCH_WIDTH - 1, f"Prediction Frame: {prediction_results['prediction_frame']}",
                color='white', fontsize=12, ha='right', va='top', 
                bbox=dict(facecolor='black', alpha=0.7, edgecolor='white', boxstyle='round,pad=0.3'))

        # Tight layout and show the plot
        plt.xlabel('Field Width (Zones)')
        plt.ylabel('Field Length (Zones)')
        plt.tight_layout()
        plt.show()
                
    # Time context print
    print(f"\n⏱️  Time Context:")
    print(f"   Sequence: {prediction_results.get('sequence_start_frame', '?')} → "
          f"{prediction_results.get('sequence_end_frame', '?')}")
    print(f"   Prediction frame: {prediction_results.get('prediction_frame', '?')}")
    print(f"   Horizon: {prediction_results.get('horizon_seconds', HORIZON_SECONDS)}s ahead")

def predict_for_multiple_players(model, df, match_id, player_ids, scaler=None, top_k=3):
    """Predict for multiple players in a match."""
    results = {}
    for player_id in player_ids:
        print(f"Predicting for player {player_id}...")
        prediction = predict_for_player(model, df, match_id, player_id, scaler, top_k=top_k)
        if prediction is not None:
            results[player_id] = prediction
            if MODEL_TYPE == "MDN_TCN":
                top = prediction['predictions'][0]
                print(f"  Top: ({top['x']:.3f}, {top['y']:.3f}) - {top['probability']*100:.1f}%")
            elif CO_ORDINATES:
                print(f"  Predicted: ({prediction['predicted_x']:.3f}, {prediction['predicted_y']:.3f})")
            else:
                print(f"  Top zone: {prediction['top_zones'][0]} ({prediction['top_probabilities'][0]:.4f})")
        else:
            print(f"  ❌ Could not predict for player {player_id}")
    return results


def predict_player_next_zone(match_id, player_id, top_k=3):
    """Convenience function for quick predictions."""
    # Use 'model' if 'trained_model' doesn't exist (e.g., after loading)
    _model = trained_model if 'trained_model' in dir() else model
    result = predict_for_player(_model, df, match_id, player_id, scaler=scaler, top_k=top_k)
    if result:
        if MODEL_TYPE == "MDN_TCN":
            print(f"Player {player_id} in Match {match_id} ({HORIZON_SECONDS}s ahead):")
            for i, pred in enumerate(result['predictions'][:top_k]):
                print(f"  {i+1}. ({pred['x']:.3f}, {pred['y']:.3f}) → Zone {pred['zone']} | {pred['probability']*100:.1f}%")
        elif CO_ORDINATES:
            print(f"Player {player_id} → ({result['predicted_x']:.3f}, {result['predicted_y']:.3f}) Zone {result['predicted_zone']}")
        else:
            for i, (zone, prob, coords) in enumerate(zip(
                result['top_zones'], result['top_probabilities'], result['top_coordinates']
            )):
                print(f"  {i+1}. Zone {zone} (Row {coords[0]}, Col {coords[1]}): {prob:.3f}")
        return result
    else:
        print(f"No prediction available for Player {player_id}")
        return None



print("✅ Unified prediction functions loaded"
      )
print(f"   Model: {MODEL_TYPE} | Backend: {BACKEND}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import seaborn as sns
from matplotlib.patches import Ellipse, Arc

print(f"Using {len(FEATURE_COLS)} features for prediction (same as training)")
print(f"Model Type: {MODEL_TYPE} | Backend: {BACKEND} | Coordinates: {CO_ORDINATES}")

# ============================================================================
# MULTI-STEP PREDICTION FUNCTIONS (n seconds ahead)
# ============================================================================

def get_player_sequence_features_multistep(
    df,
    match_id,
    player_id,
    start_time_seconds=None,
    seq_len=SEQ_LEN,
    n_steps=1  # Number of steps to predict (1 step = HORIZON_SECONDS)
):
    """
    Get player sequence features and ground truth for multiple future steps.
    
    Args:
        n_steps: Number of prediction steps (e.g., n_steps=3 predicts next 3 seconds if HORIZON_SECONDS=1)
    
    Returns:
        features: Input sequence
        time_info: Dictionary with prediction frames and ground truth positions for each step
    """
    df.head()
    player_data = df[
        (df["match_id"] == match_id) &
        (df["player_id"] == player_id)
    ].copy()

    min_frames_needed = seq_len + (HORIZON_FRAMES * n_steps)
    print(len(player_data))
    if len(player_data) < min_frames_needed:
        print(f"Insufficient data for player {player_id} in match {match_id}. "
              f"Need {min_frames_needed} frames, have {len(player_data)}")
        return None, None

    player_data = player_data.sort_values("frame_number").reset_index(drop=True)

    if start_time_seconds is None:
        start_idx = len(player_data) - min_frames_needed
    else:
        if "timestamp_seconds" not in player_data.columns:
            print("Error: 'timestamp_seconds' column not found.")
            return None, None

        timestamps = player_data["timestamp_seconds"].values
        nearest_idx = np.abs(timestamps - start_time_seconds).argmin()
        max_valid_start = len(player_data) - min_frames_needed

        if nearest_idx <= max_valid_start:
            start_idx = nearest_idx
        else:
            if max_valid_start < 0:
                print("No valid starting point available.")
                return None, None
            start_idx = max_valid_start

    end_idx = start_idx + seq_len
    sequence_data = player_data.iloc[start_idx:end_idx]

    available_features = [col for col in FEATURE_COLS if col in sequence_data.columns]
    features_to_use = available_features if len(available_features) < len(FEATURE_COLS) else FEATURE_COLS

    features = sequence_data[features_to_use].values.astype(np.float32)

    current_frame = int(sequence_data.iloc[-1]["frame_number"])
    
    # Initialize time info with current position
    time_info = {
        "current_frame": current_frame,
        "sequence_start_frame": int(sequence_data.iloc[0]["frame_number"]),
        "sequence_end_frame": current_frame,
        "n_steps": n_steps,
        "prediction_frames": [],
        "ground_truth_positions": []
    }

    if "timestamp_seconds" in sequence_data.columns:
        time_info["sequence_start_timestamp_sec"] = float(sequence_data.iloc[0]["timestamp_seconds"])
        time_info["current_timestamp_sec"] = float(sequence_data.iloc[-1]["timestamp_seconds"])

    if "x_normalized" in sequence_data.columns and "y_normalized" in sequence_data.columns:
        time_info["current_x"] = float(sequence_data.iloc[-1]["x_normalized"])
        time_info["current_y"] = float(sequence_data.iloc[-1]["y_normalized"])

    # Collect ground truth for each prediction step
    for step in range(1, n_steps + 1):
        prediction_frame = current_frame + (HORIZON_FRAMES * step)
        time_info["prediction_frames"].append(prediction_frame)
        
        predicted_frame_data = player_data[player_data["frame_number"] == prediction_frame]
        
        if not predicted_frame_data.empty and \
           "x_normalized" in predicted_frame_data.columns and \
           "y_normalized" in predicted_frame_data.columns:
            
            pred_x_norm = float(predicted_frame_data.iloc[0]["x_normalized"])
            pred_y_norm = float(predicted_frame_data.iloc[0]["y_normalized"])
            
            time_info["ground_truth_positions"].append({
                "frame": prediction_frame,
                "x": pred_x_norm * 105,
                "y": pred_y_norm * 68,
                "x_norm": pred_x_norm,
                "y_norm": pred_y_norm,
                "step": step,
                "time_ahead_seconds": step * HORIZON_SECONDS
            })
        else:
            time_info["ground_truth_positions"].append(None)

    return features, time_info


def predict_for_player_multistep(model, df, match_id, player_id, scaler=None,
                                  start_frame=None, n_steps=3, top_k=5, 
                                  return_all_probs=False):
    """
    Predict player position for multiple future time steps.
    
    Args:
        n_steps: Number of future steps to predict (e.g., 3 steps = 3 seconds if HORIZON_SECONDS=1)
    
    Returns:
        Dictionary with predictions for each step
    """
    # Get player sequence with multi-step ground truth
    player_sequence, time_info = get_player_sequence_features_multistep(
        df=df, match_id=match_id, player_id=player_id, n_steps=n_steps,seq_len=SEQ_LEN, start_time_seconds=start_frame
    )

    if player_sequence is None:
        return None

    # Apply scaling
    if scaler is not None:
        seq_len, num_features = player_sequence.shape
        if num_features != scaler.n_features_in_:
            print(f"⚠️  Feature mismatch! Sequence: {num_features}, Scaler: {scaler.n_features_in_}")
            return None
        player_sequence_flat = player_sequence.reshape(-1, num_features)
        player_sequence_scaled = scaler.transform(player_sequence_flat)
        player_sequence = player_sequence_scaled.reshape(seq_len, num_features)

    # Store predictions for each step
    all_predictions = []
    current_sequence = player_sequence.copy()
    
    # Iteratively predict n steps
    for step in range(n_steps):
        input_tensor = np.expand_dims(current_sequence, axis=0)
        
        # ─── MDN_TCN ─────────────────────────────────────────────────────
        if MODEL_TYPE == "MDN_TCN":
            outputs = model({"sequence_input": input_tensor}, training=False)
            pi = outputs['pi'].numpy()[0]
            mu = outputs['mu'].numpy()[0]
            sigma = outputs['sigma'].numpy()[0]

            sorted_indices = np.argsort(pi)[::-1][:top_k]
            predictions = []
            
            for idx in sorted_indices:
                predictions.append({
                    'probability': float(pi[idx]),
                    'x': float(mu[idx, 0]),
                    'y': float(mu[idx, 1]),
                    'x_std': float(sigma[idx, 0]),
                    'y_std': float(sigma[idx, 1]),
                    'component': int(idx),
                    'zone': int(xy_to_zone_vectorized(
                        np.array([mu[idx, 0]]),
                        np.array([mu[idx, 1]]),
                        N_ROWS, N_COLS
                    )[0])
                })

            expected_pos = np.sum(pi[:, np.newaxis] * mu, axis=0)
            mode_pos = mu[np.argmax(pi)]
            
            step_result = {
                'step': step + 1,
                'time_ahead_seconds': (step + 1) * HORIZON_SECONDS,
                'predictions': predictions,
                'expected_position': {'x': float(expected_pos[0]), 'y': float(expected_pos[1])},
                'mode_position': {'x': float(mode_pos[0]), 'y': float(mode_pos[1])},
                'pi': pi,
                'mu': mu,
                'sigma': sigma
            }
            
            # Update sequence for next prediction (autoregressive)
            # Use expected position as next input
            next_features = current_sequence[-1].copy()
            next_features[0] = expected_pos[0]  # x_normalized
            next_features[1] = expected_pos[1]  # y_normalized
            current_sequence = np.vstack([current_sequence[1:], next_features])

        # ─── Keras Classification/Regression ────────────────────────────
        elif BACKEND == 'keras':
            preds = model.predict(input_tensor, verbose=0)

            if CO_ORDINATES:
                pred_x = float(preds[0][0])
                pred_y = float(preds[0][1])
                pred_zone = int(xy_to_zone_vectorized(
                    np.array([pred_x]), np.array([pred_y]), N_ROWS, N_COLS
                )[0])

                step_result = {
                    'step': step + 1,
                    'time_ahead_seconds': (step + 1) * HORIZON_SECONDS,
                    'predicted_x': pred_x,
                    'predicted_y': pred_y,
                    'predicted_zone': pred_zone
                }
                
                # Update sequence
                next_features = current_sequence[-1].copy()
                next_features[0] = pred_x
                next_features[1] = pred_y
                current_sequence = np.vstack([current_sequence[1:], next_features])
                
            else:  # Classification
                probs = preds[0]
                top_indices = np.argsort(probs)[-top_k:][::-1]
                top_probs = probs[top_indices]

                step_result = {
                    'step': step + 1,
                    'time_ahead_seconds': (step + 1) * HORIZON_SECONDS,
                    'top_zones': top_indices,
                    'top_probabilities': top_probs,
                    'top_coordinates': [(int(z // N_COLS), int(z % N_COLS)) for z in top_indices]
                }
                
                if return_all_probs:
                    step_result['all_probabilities'] = probs
                
                # Update sequence with most likely zone center
                top_zone = top_indices[0]
                pred_x, pred_y = zone_to_xy_center(top_zone, N_ROWS, N_COLS)
                next_features = current_sequence[-1].copy()
                next_features[0] = pred_x
                next_features[1] = pred_y
                current_sequence = np.vstack([current_sequence[1:], next_features])

        all_predictions.append(step_result)

    # Compile final results
    results = {
        'player_id': player_id,
        'match_id': match_id,
        'model_type': MODEL_TYPE,
        'n_steps': n_steps,
        'horizon_seconds': HORIZON_SECONDS,
        'predictions': all_predictions,
        **time_info
    }

    return results


def zone_to_xy_center(zone_idx, n_rows, n_cols):
    """Helper to find the physical center of a zone index."""
    row = zone_idx // n_cols
    col = zone_idx % n_cols
    x = (col + 0.5) / n_cols
    y = (row + 0.5) / n_rows
    return x, y


def visualize_prediction_multistep(prediction_results, title=None):
    """Visualize multi-step predictions on football field."""
    if prediction_results is None:
        print("No prediction results to visualize")
        return

    model_type = prediction_results.get('model_type', '')
    n_steps = prediction_results.get('n_steps', 1)
    
    PITCH_LENGTH = 105
    PITCH_WIDTH = 68

    fig, ax = plt.subplots(1, 1, figsize=(16, 10))
    ax.set_xlim(-1, PITCH_LENGTH + 1)
    ax.set_ylim(-1, PITCH_WIDTH + 1)
    ax.set_facecolor('#2d8a2d')

    # Draw pitch
    ax.add_patch(patches.Rectangle((0, 0), PITCH_LENGTH, PITCH_WIDTH, fill=False, edgecolor='white', linewidth=2))
    ax.axvline(x=PITCH_LENGTH / 2, color='white', linewidth=1, linestyle='-', alpha=0.5)
    center_circle = plt.Circle((PITCH_LENGTH / 2, PITCH_WIDTH / 2), 9.15, color='white', fill=False, linewidth=1)
    ax.add_patch(center_circle)
    ax.add_patch(patches.Rectangle((0, (PITCH_WIDTH - 40) / 2), 16.5, 40, fill=False, color='white', linewidth=2))
    ax.add_patch(patches.Rectangle((PITCH_LENGTH - 16.5, (PITCH_WIDTH - 40) / 2), 16.5, 40, fill=False, color='white', linewidth=2))
    ax.add_patch(patches.Rectangle((0, (PITCH_WIDTH - 20) / 2), 5.5, 20, fill=False, color='white', linewidth=2))
    ax.add_patch(patches.Rectangle((PITCH_LENGTH - 5.5, (PITCH_WIDTH - 20) / 2), 5.5, 20, fill=False, color='white', linewidth=2))
    ax.plot([0, 0], [(PITCH_WIDTH - 7) / 2, (PITCH_WIDTH + 7) / 2], color='white', lw=5)
    ax.plot([PITCH_LENGTH, PITCH_LENGTH], [(PITCH_WIDTH - 7) / 2, (PITCH_WIDTH + 7) / 2], color='white', lw=5)
    ax.add_patch(Arc((0, 0), 2, 2, theta1=0, theta2=90, color='white', lw=2))
    ax.add_patch(Arc((PITCH_LENGTH, 0), 2, 2, theta1=90, theta2=180, color='white', lw=2))
    ax.add_patch(Arc((0, PITCH_WIDTH), 2, 2, theta1=270, theta2=360, color='white', lw=2))
    ax.add_patch(Arc((PITCH_LENGTH, PITCH_WIDTH), 2, 2, theta1=180, theta2=270, color='white', lw=2))

    # Plot current position
    if 'current_x' in prediction_results:
        current_x = prediction_results['current_x'] * PITCH_LENGTH
        current_y = prediction_results['current_y'] * PITCH_WIDTH
        ax.scatter(current_x, current_y, c='white', s=300, marker='o',
                   edgecolors='black', linewidth=3, zorder=20, label='Current Position')

    # Color schemes for different steps
    colors = plt.cm.Reds(np.linspace(0.3, 0.9, n_steps))
    
    # ─── MDN Model ──────────────────────────────────────────────────────
    if 'MDN' in model_type:
        for idx, pred_step in enumerate(prediction_results['predictions']):
            step_num = pred_step['step']
            time_ahead = pred_step['time_ahead_seconds']
            
            # Plot expected position
            exp_pos = pred_step['expected_position']
            exp_x = exp_pos['x'] * PITCH_LENGTH
            exp_y = exp_pos['y'] * PITCH_WIDTH
            
            ax.scatter(exp_x, exp_y, c=[colors[idx]], s=200, marker='X',
                       edgecolors='black', linewidth=2, zorder=15,
                       label=f'Prediction +{time_ahead}s')
            
            # Plot trajectory line
            if idx == 0:
                ax.plot([current_x, exp_x], [current_y, exp_y],
                        color=colors[idx], linewidth=2, linestyle='--', alpha=0.7)
            else:
                prev_exp = prediction_results['predictions'][idx-1]['expected_position']
                prev_x = prev_exp['x'] * PITCH_LENGTH
                prev_y = prev_exp['y'] * PITCH_WIDTH
                ax.plot([prev_x, exp_x], [prev_y, exp_y],
                        color=colors[idx], linewidth=2, linestyle='--', alpha=0.7)
            
            # Annotate time
            ax.annotate(f'+{time_ahead}s', xy=(exp_x, exp_y), 
                        xytext=(5, 5), textcoords='offset points',
                        fontsize=10, fontweight='bold', color='white',
                        bbox=dict(boxstyle='round,pad=0.3', facecolor=colors[idx], alpha=0.7))

    # ─── Regression Model ───────────────────────────────────────────────
    elif 'regression' in model_type:
        for idx, pred_step in enumerate(prediction_results['predictions']):
            pred_x = pred_step['predicted_x'] * PITCH_LENGTH
            pred_y = pred_step['predicted_y'] * PITCH_WIDTH
            time_ahead = pred_step['time_ahead_seconds']
            
            ax.scatter(pred_x, pred_y, c=[colors[idx]], s=200, marker='X',
                       edgecolors='white', linewidth=2, zorder=15,
                       label=f'Prediction +{time_ahead}s')
            
            # Draw trajectory
            if idx == 0:
                ax.plot([current_x, pred_x], [current_y, pred_y],
                        color=colors[idx], linewidth=2, linestyle='--', alpha=0.7)
            else:
                prev_x = prediction_results['predictions'][idx-1]['predicted_x'] * PITCH_LENGTH
                prev_y = prediction_results['predictions'][idx-1]['predicted_y'] * PITCH_WIDTH
                ax.plot([prev_x, pred_x], [prev_y, pred_y],
                        color=colors[idx], linewidth=2, linestyle='--', alpha=0.7)
            
            ax.annotate(f'+{time_ahead}s', xy=(pred_x, pred_y),
                        xytext=(5, 5), textcoords='offset points',
                        fontsize=10, fontweight='bold', color='white',
                        bbox=dict(boxstyle='round,pad=0.3', facecolor=colors[idx], alpha=0.7))

    # ─── Classification Model ──────────────────────────────────────────
    else:
        grid_width = PITCH_LENGTH / N_COLS
        grid_height = PITCH_WIDTH / N_ROWS
        
        for i in range(1, N_COLS):
            ax.axvline(x=i * grid_width, color='black', linewidth=1, linestyle='-', alpha=0.6)
        for i in range(1, N_ROWS):
            ax.axhline(y=i * grid_height, color='black', linewidth=1, linestyle='-', alpha=0.6)
        
        for idx, pred_step in enumerate(prediction_results['predictions']):
            top_zone = pred_step['top_zones'][0]
            top_prob = pred_step['top_probabilities'][0]
            time_ahead = pred_step['time_ahead_seconds']
            
            row = top_zone // N_COLS
            col = top_zone % N_COLS
            
            center_x = (col + 0.5) * grid_width
            center_y = (row + 0.5) * grid_height
            
            # Highlight zone
            ax.add_patch(patches.Rectangle(
                (col * grid_width, row * grid_height),
                grid_width, grid_height,
                linewidth=2,
                edgecolor=colors[idx],
                facecolor=colors[idx],
                alpha=0.3
            ))
            
            # Mark center
            ax.scatter(center_x, center_y, c=[colors[idx]], s=200, marker='X',
                       edgecolors='white', linewidth=2, zorder=15,
                       label=f'Prediction +{time_ahead}s')
            
            # Annotate
            ax.annotate(f'+{time_ahead}s\n{top_prob*100:.1f}%', 
                        xy=(center_x, center_y),
                        ha='center', va='center',
                        fontsize=10, fontweight='bold', color='white',
                        bbox=dict(boxstyle='round,pad=0.3', facecolor='black', alpha=0.7))

    # Plot ground truth trajectory if available
    if prediction_results.get('ground_truth_positions'):
        gt_positions = [gt for gt in prediction_results['ground_truth_positions'] if gt is not None]
        if gt_positions:
            gt_x = [gt['x'] for gt in gt_positions]
            gt_y = [gt['y'] for gt in gt_positions]
            
            # Plot ground truth path
            all_x = [current_x] + gt_x
            all_y = [current_y] + gt_y
            ax.plot(all_x, all_y, color='green', linewidth=3, linestyle=':', 
                    alpha=0.8, label='Ground Truth Path', zorder=5)
            
            # Mark ground truth positions
            for gt in gt_positions:
                ax.scatter(gt['x'], gt['y'], c='green', s=150, marker='o',
                           edgecolors='black', linewidth=2, zorder=10)

    ax.set_xlabel('X (meters)', fontsize=12)
    ax.set_ylabel('Y (meters)', fontsize=12)
    ax.set_aspect('equal')
    ax.legend(loc='upper right', fontsize=10, framealpha=0.9)
    
    title_text = title or f"Multi-Step Prediction ({n_steps}x{HORIZON_SECONDS}s = {n_steps*HORIZON_SECONDS}s ahead) - Player {prediction_results['player_id']}"
    ax.set_title(title_text, fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    # Print summary
    print(f"\n⏱️  Multi-Step Prediction Summary:")
    print(f"   • Total steps: {n_steps}")
    print(f"   • Time horizon: {n_steps * HORIZON_SECONDS} seconds")
    print(f"   • Current frame: {prediction_results.get('current_frame', '?')}")
    print(f"   • Prediction frames: {prediction_results.get('prediction_frames', [])}")


# Convenience function
def predict_player_trajectory(model, df, match_id, player_id, scaler=None, 
                               start_frame=None, n_seconds=3, top_k=5):
    """
    Predict player trajectory for n seconds ahead.
    
    Args:
        n_seconds: Number of seconds to predict (will be converted to steps)
    """
    n_steps = int(n_seconds / HORIZON_SECONDS)
    if n_steps < 1:
        n_steps = 1
    
    result = predict_for_player_multistep(
        model, df, match_id, player_id, 
        scaler=scaler, start_frame=start_frame, 
        n_steps=n_steps, top_k=top_k
    )
    
    if result:
        print(f"\n🎯 Trajectory Prediction for Player {player_id}")
        print(f"   Total prediction time: {n_seconds}s ({n_steps} steps)")
        
        for pred in result['predictions']:
            step = pred['step']
            time_ahead = pred['time_ahead_seconds']
            print(f"\n   Step {step} (+{time_ahead}s):")
            
            if MODEL_TYPE == "MDN_TCN":
                exp = pred['expected_position']
                print(f"      Expected: ({exp['x']:.3f}, {exp['y']:.3f})")
            elif CO_ORDINATES:
                print(f"      Position: ({pred['predicted_x']:.3f}, {pred['predicted_y']:.3f})")
            else:
                print(f"      Top zone: {pred['top_zones'][0]} ({pred['top_probabilities'][0]:.3f})")
    
    return result


print("✅ Multi-step prediction functions loaded")
print(f"   Use predict_player_trajectory(model, df, match_id, player_id, n_seconds=3)")
print(f"   Or predict_for_player_multistep(..., n_steps=3)")

In [ ]:
# Example usage:

# 1. Predict 3 seconds ahead (3 steps if HORIZON_SECONDS=1)
result = predict_player_trajectory(
    model=inference_model,  # or loaded_mdn_model
    df=df,
    match_id='2417',
    player_id='12788',
    scaler=scaler,
    start_frame=2,
    n_seconds=3  # Predict 3 seconds into the future
)

# 2. Visualize the multi-step prediction
if result:
    visualize_prediction_multistep(result)

# 3. Or use the lower-level function directly
result = predict_for_player_multistep(
    model=inference_model,
    df=df,
    match_id='2417',
    player_id='12788',
    scaler=scaler,
    start_frame=100,
    n_steps=5,  # 5 prediction steps
    top_k=3
)

In [ ]:
# ============================================================================
# PREDICTION DEMONSTRATIONS
# ============================================================================
# Use existing output_dir or set default for local/Colab
print("\n" + "="*80)
print("🎯 PREDICTION DEMONSTRATIONS")
print("="*80)

available_matches = df['match_id'].unique()
print(f"\nAvailable matches: {available_matches}")

demo_match = available_matches[2]
demo_players = df[df['match_id'] == demo_match]['player_id'].unique()[:]
print(f"Using match {demo_match} | Demo players: {demo_players}")


# ── Example 1: Single player ────────────────────────────────────────────────
print("\n" + "="*80)
print("EXAMPLE 1: Single Player Prediction")
print("="*80)
model= loaded_mdn_model if MODEL_TYPE=="MDN_TCN" else test_model_best_new
demo_player = demo_players[-4]

prediction_result = predict_for_player(
    model, df, demo_match, demo_player,
   scaler=scaler,start_frame=1500, top_k=5, return_all_probs=True,
)

if prediction_result:
    visualize_prediction(prediction_result)
else:
    print(f"❌ Could not predict for player {demo_player}")

# ── Example 2: Multiple players ─────────────────────────────────────────────
print("\n" + "="*80)
print("EXAMPLE 2: Multiple Players")
print("="*80)

multi_results = predict_for_multiple_players(
    model, df, demo_match, demo_players[:3],
    scaler=scaler, top_k=3
)

# ── Example 3: Quick prediction ──────────────────────────────────────────────
print("\n" + "="*80)
print("EXAMPLE 3: Quick Prediction")
print("="*80)

test_result = predict_player_next_zone(demo_match, demo_players[1])

# ── Save results ─────────────────────────────────────────────────────────────
print("\n💾 Saving prediction examples...")


print(f"Use predict_player_next_zone(match_id, player_id) for quick predictions")

print("\n" + "="*80)
print("✅ Prediction system ready!")

In [ ]:
train_df[['x_normalized', 'y_normalized']].agg(['min', 'max'])

In [ ]:
train_df[['x_normalized','y_normalized']].mean()


In [ ]:
train_df[['x_normalized','y_normalized']].std()

In [ ]:
# ==== FINAL EVALUATION (Backend-Aware) ====
import numpy as np
from sklearn.metrics import accuracy_score, top_k_accuracy_score, classification_report


# Check if we are using MDN model
if MODEL_TYPE == "MDN_TCN":
    print("="*60)
    print("📊 MDN-SPECIFIC EVALUATION")
    print("="*60)
    print("Detected MDN model. Using probabilistic evaluation metrics (NLL, MAE) instead of classification accuracy.")
    print("To see classification metrics, you would need to convert coordinates to zones first.")
    
    # Use the MDN-specific evaluation function defined earlier
    # Use 'model' if 'trained_model' doesn't exist
    _eval_model = trained_model if 'trained_model' in dir() else loaded_mdn_model
    metrics = full_mdn_evaluation(_eval_model, test_loader.take(20), k=3)
    
elif not CO_ORDINATES and MODEL_TYPE != "MDN_TCN":
    # ────────────────────────────────────────────────────────────────────────
    # CLASSIFICATION EVALUATION LOGIC
    # ────────────────────────────────────────────────────────────────────────
    def evaluate_with_metrics(model, loader):
        """Backend-agnostic evaluation function."""
        all_preds = []
        all_labels = []
        all_probs = []

        if BACKEND == 'keras':
            print("Evaluating with Keras...")
            # Keras: Iterate through tf.data.Dataset
            for batch_x, batch_y in loader:
                probs = model.predict(batch_x, verbose=0)
                preds = np.argmax(probs, axis=1)

                all_probs.append(probs)
                all_preds.append(preds)
                all_labels.append(batch_y.numpy())

            all_probs = np.concatenate(all_probs, axis=0)
            all_preds = np.concatenate(all_preds, axis=0)
            all_labels = np.concatenate(all_labels, axis=0)

        # Calculate standard accuracy
        acc = accuracy_score(all_labels, all_preds)

        # Prepare full class list based on model output shape
        num_classes = all_probs.shape[1]
        all_classes = np.arange(num_classes)

        # Ensure k <= num_classes
        k = min(3, num_classes)

        # Pass the full label set to avoid ValueError
        top3_acc = top_k_accuracy_score(all_labels, all_probs, k=k, labels=all_classes)

        return acc, top3_acc, all_probs, all_preds, all_labels

    def evaluate_detailed(model, loader, zone_names):
        """Wrapper for detailed report using the generic evaluator."""
        _, _, _, preds, labels = evaluate_with_metrics(model, loader)

        print("\nDetailed Classification Report:")
        # Use a subset of labels if zone_names provided to avoid errors if some classes are missing
        unique_labels = np.unique(labels)
        target_names = [zone_names[i] for i in unique_labels] if zone_names else None

        report = classification_report(labels, preds, target_names=target_names, zero_division=0)
        print(report)

        return accuracy_score(labels, preds), preds, labels


    # ----------------------------------------------------------------------------
    # RUN EVALUATION
    # ----------------------------------------------------------------------------
    print("Evaluating model on test set...")

    # Use 'model' if 'trained_model' doesn't exist
    _eval_model = trained_model if 'trained_model' in dir() else model

    loader=test_loader.take(5)
    # Detailed evaluation
    test_acc, test_top3_acc, test_probs, _, _ = evaluate_with_metrics(_eval_model, loader)

    print(f"\nFinal Test Results:")
    print(f"Test Accuracy: {test_acc:.4f}")
    print(f"Test Top-3 Accuracy: {test_top3_acc:.4f}")

    # Generate zone names for detailed reporting
    zone_names = [f"Zone_{i}" for i in range(NUM_ZONES)]

    # Detailed classification report
    # We pass None for zone_names to let sklearn handle missing classes gracefully
    detailed_acc, test_preds, test_labels = evaluate_detailed(_eval_model, loader, None)

    # Performance summary
    print("\n" + "="*60)
    print("MODEL PERFORMANCE SUMMARY")
    print("="*60)
    print(f"Grid Configuration: {N_ROWS} x {N_COLS} = {NUM_ZONES} zones")
    print(f"Prediction Horizon: {HORIZON_SECONDS} seconds ({HORIZON_FRAMES} frames)")
    print(f"Features Used: {len(FEATURE_COLS)}")
    print(f"Model Type: {MODEL_TYPE}")
    print(f"Backend: {BACKEND}")



    print("\nAccuracy Metrics:")
    print(f"- Top-1 Accuracy: {test_acc:.4f}")
    print(f"- Top-3 Accuracy: {test_top3_acc:.4f}")
    print("="*60)

In [ ]:
import numpy as np

def inspect_mdn_output(model, player_sequence, top_k=5):
    """Inspect MDN model output (pi, mu, sigma) for the player sequence prediction"""
    # Expand dimensions to match model's input shape
    input_tensor = np.expand_dims(player_sequence, axis=0)
    
    # Get MDN output
    outputs = model(input_tensor, training=False)
    
    pi = outputs['pi'].numpy()[0]  # Mixture weights
    mu = outputs['mu'].numpy()[0]  # Means (x, y)
    sigma = outputs['sigma'].numpy()[0]  # Standard deviations
    
    print("Mixture weights (pi):", pi)
    print("Means (mu):", mu)
    print("Standard deviations (sigma):", sigma)

    # If top_k is requested, sort and print the top_k predictions
    sorted_indices = np.argsort(pi)[::-1][:top_k]
    for idx in sorted_indices:
        print(f"Component {idx}:")
        print(f"  Probability (pi): {pi[idx]}")
        print(f"  Mean (mu): {mu[idx]}")
        print(f"  Std Dev (sigma): {sigma[idx]}")

    return pi, mu, sigma

# Example usage:
# Assuming player_sequence is the input sequence for a player
# model: trained MDN model
pi, mu, sigma = inspect_mdn_output(model, player_sequence)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.patches import Ellipse
import numpy as np

def plot_mdn_density(pi, mu, sigma, PITCH_LENGTH=105, PITCH_WIDTH=68):
    """Visualize the MDN density as a heatmap on a football field"""
    
    # Create grid for density calculation
    grid_res_x = 200
    grid_res_y = 130
    x = np.linspace(0, 1, grid_res_x)
    y = np.linspace(0, 1, grid_res_y)
    X, Y = np.meshgrid(x, y)

    # Calculate the density based on the MDN components
    density = np.zeros_like(X)
    
    for k in range(len(pi)):
        mu_x = mu[k, 0]
        mu_y = mu[k, 1]
        sigma_x = max(sigma[k, 0], 1e-4)  # Ensure no division by zero
        sigma_y = max(sigma[k, 1], 1e-4)

        # 2D Gaussian distribution for each component
        gaussian = (1 / (2 * np.pi * sigma_x * sigma_y)) * np.exp(
            -(((X - mu_x) ** 2) / (2 * sigma_x**2) + ((Y - mu_y) ** 2) / (2 * sigma_y**2))
        )
        
        density += pi[k] * gaussian

    # Normalize the density
    density /= np.max(density)

    # Plot the density
    fig, ax = plt.subplots(figsize=(14, 9))
    ax.set_xlim(-1, PITCH_LENGTH + 1)
    ax.set_ylim(-1, PITCH_WIDTH + 1)
    ax.set_facecolor('#2d8a2d')
    
    # Draw the pitch boundary and other elements
    ax.add_patch(patches.Rectangle((0, 0), PITCH_LENGTH, PITCH_WIDTH, fill=False, edgecolor='white', linewidth=2))
    ax.axvline(x=PITCH_LENGTH / 2, color='white', linewidth=1, linestyle='-', alpha=0.5)
    ax.add_patch(plt.Circle((PITCH_LENGTH / 2, PITCH_WIDTH / 2), 9.15, color='white', fill=False, linewidth=1))

    # Visualize the density as a heatmap
    heatmap = ax.imshow(
        density,
        extent=[0, PITCH_LENGTH, 0, PITCH_WIDTH],
        origin='lower',
        cmap='jet',  # Color map: Blue → Green → Yellow → Red
        alpha=0.75,
        aspect='auto'
    )
    
    cbar = plt.colorbar(heatmap, ax=ax, fraction=0.035, pad=0.02)
    cbar.set_label("Probability Density", fontsize=11)

    ax.set_xlabel('X (meters)', fontsize=12)
    ax.set_ylabel('Y (meters)', fontsize=12)
    ax.set_aspect('equal')
    ax.set_title(f"MDN Density Prediction", fontsize=14, fontweight='bold')

    plt.tight_layout()
    plt.show()

# Example usage:
# Assuming pi, mu, sigma are extracted from the MDN output
plot_mdn_density(pi, mu, sigma)

In [ ]:
player_sequence, time_info = get_player_sequence_features(
        df, demo_match, demo_player, 100
    )
x_input = {"sequence_input": np.expand_dims(player_sequence, axis=0)}
raw_x = x_input["sequence_input"]

## ENSEMBLED TCN and MDN+TCN

In [ ]:
import numpy as np

def ensemble_predict(mdn_model, class_model, x_input, n_rows=N_ROWS, n_cols=N_COLS):
    """
    Ensembles MDN (Coordinates) and TCN (Zones).
    """
    # 1. Get MDN Prediction (Continuous Point)
    mdn_preds = mdn_model.predict(x_input, verbose=0)
    pi, mu = mdn_preds['pi'], mdn_preds['mu']
    
    # Extract Max-Pi Mode
    max_pi_idx = np.argmax(pi, axis=1)
    batch_idx = np.arange(len(max_pi_idx))
    x_raw = mu[batch_idx, max_pi_idx, 0]
    y_raw = mu[batch_idx, max_pi_idx, 1]
    
    # 2. Get Classification Prediction (Categorical Zones)
    class_probs = class_model.predict(x_input, verbose=0)
    top_1_zone = np.argmax(class_probs, axis=1)
    
    # 3. Validation: Does MDN point live in a logical zone?
    # Convert MDN (x,y) back to a zone index
    mdn_zone = xy_to_zone_vectorized(x_raw, y_raw, n_rows, n_cols)
    
    final_x, final_y = [], []
    
    for i in range(len(x_raw)):
        # Check if MDN and Classification agree (Top-3 check)
        top_3_zones = np.argsort(class_probs[i])[-3:]
        
        if mdn_zone[i] in top_3_zones:
            # AGREEMENT: Use high-precision MDN coordinate
            final_x.append(x_raw[i])
            final_y.append(y_raw[i])
        else:
            # DISAGREEMENT: MDN is an outlier. Snap to Class Center
            # Get center of the Top-1 predicted zone
            zx, zy = zone_to_xy_center(top_1_zone[i], n_rows, n_cols)
            final_x.append(zx)
            final_y.append(zy)
            
    return np.array(final_x), np.array(final_y), mdn_zone, top_1_zone

def zone_to_xy_center(zone_idx, n_rows, n_cols):
    """Helper to find the physical center of a zone index."""
    row = zone_idx // n_cols
    col = zone_idx % n_cols
    # Center of the cell in 0-1 normalized space
    x = (col + 0.5) / n_cols
    y = (row + 0.5) / n_rows
    return x, y


predicts=ensemble_predict(loaded_mdn_model, reloaded_model_Classification, raw_x)
print(predicts)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def plot_ensemble_prediction(mdn_xy, mdn_zone, class_zone, n_rows=N_ROWS, n_cols=N_COLS):
    fig, ax = plt.subplots(figsize=(12, 8))
    
    # 1. Draw Pitch (Normalized 0-1 space)
    ax.set_facecolor('#22312b') # Pitch Green
    plt.plot([0, 1, 1, 0, 0], [0, 0, 1, 1, 0], color="white", lw=2) # Boundaries
    plt.axvline(0.5, color="white", lw=2) # Halfway line
    
    # 2. Draw the Classification Zone (Box)
    c_row = class_zone // n_cols
    c_col = class_zone % n_cols
    rect = patches.Rectangle((c_col/n_cols, c_row/n_rows), 1/n_cols, 1/n_rows, 
                             linewidth=2, edgecolor='yellow', facecolor='yellow', alpha=0.3, label='Class. Predicted Zone')
    ax.add_patch(rect)
    
    # 3. Plot the MDN Coordinate (Point)
    ax.scatter(mdn_xy[0], mdn_xy[1], color='cyan', s=150, edgecolors='white', 
               marker='*', zorder=5, label=f'MDN Point (Zone {mdn_zone})')

    # Add labels and formatting
    plt.title(f"Ensemble Forecast: MDN (Zone {mdn_zone}) vs Classification (Zone {class_zone})", color='white', fontsize=15)
    plt.legend(loc='upper right')
    plt.xlim(-0.05, 1.05)
    plt.ylim(-0.05, 1.05)
    plt.gca().invert_yaxis() # Match your dataframe's y-coord system if 0 is top
    plt.show()

# Run it with your previous output
# (array([0.3888]), array([0.8333]), array([18]), array([21]))
mdn_x = predicts[0][0]
mdn_y = predicts[1][0]
mdn_z = predicts[2][0]
cl_z  = predicts[3][0]

plot_ensemble_prediction((mdn_x, mdn_y), mdn_z, cl_z)

In [ ]:
def calculate_prediction_error_final(predicts, time_info):
    # Standard Pitch Dimensions
    pitch_length = 105.0
    pitch_width = 68.0

    # 1. Prediction (Always normalized 0-1)
    pred_x, pred_y = predicts[0][0], predicts[1][0]
    
    # 2. Extract Ground Truth
    tx = time_info['predicted_true_x']
    ty = time_info['predicted_true_y']
    print(pred_x, pred_y,tx, ty)
    # 3. AUTO-DETECTION: Is the ground truth already in meters or normalized?
    if tx > 1.0 or ty > 1.0:
        # If values are large, they are likely already METERS
        actual_x_m, actual_y_m = tx, ty
    else:
        # If values are 0-1, convert them to METERS
        actual_x_m, actual_y_m = tx * pitch_length, ty * pitch_width

    # 4. Convert Prediction to METERS
    pred_x_m, pred_y_m = pred_x * pitch_length, pred_y * pitch_width

    # 5. Calculate Euclidean Error in Meters
    error_meters = np.sqrt((pred_x_m - actual_x_m)**2 + (pred_y_m - actual_y_m)**2)
    
    print(f"═" * 35)
    print(f"🎯 Forecast (Meters): ({pred_x_m:.2f}, {pred_y_m:.2f})")
    print(f"📍 Reality (Meters):  ({actual_x_m:.2f}, {actual_y_m:.2f})")
    print(f"📏 Distance Error:    {error_meters:.2f} meters")
    print(f"═" * 35)
    
    return error_meters

# To use it, pass both the model predictions and your time_info dictionary:
error = calculate_prediction_error_final(predicts, time_info)

In [ ]:
x_batch, y_batch = next(iter(val_loader))
y_batch_sample = y_batch[0:1] 
print(y_batch_sample)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.colors import LinearSegmentedColormap

def generate_scouting_heatmap(model, player_sequences, actual_coords=None):
    """
    Generates a 3-second 'Future Heatmap' for a player trajectory.
    """
    fig, ax = plt.subplots(figsize=(14, 9))
    
    # 1. Pitch Setup
    ax.set_facecolor('#1a2421') 
    plt.plot([0, 1, 1, 0, 0], [0, 0, 1, 1, 0], color="white", alpha=0.5)
    plt.axvline(0.5, color="white", alpha=0.3)

    # 2. Run Predictions for the sequence
    # player_sequences shape: (N_frames, SEQ_LEN, num_features)
    preds = model.predict(player_sequences, verbose=0)
    pi, mu, sigma = preds['pi'], preds['mu'], preds['sigma']

    # 3. Plot Probability 'Bubbles'
    # We use alpha to show the 'flow' over time
    for t in range(len(pi)):
        # Get the most likely mixture (max pi)
        k = np.argmax(pi[t])
        
        # Draw an ellipse representing the model's uncertainty (2-sigma)
        # Using sigma for width/height
        ellipse = patches.Ellipse(
            (mu[t, k, 0], mu[t, k, 1]), 
            width=sigma[t, k, 0] * 2, 
            height=sigma[t, k, 1] * 2,
            angle=0, color='cyan', alpha=0.1, zorder=2
        )
        ax.add_patch(ellipse)
        
        # Plot the center point trajectory
        ax.scatter(mu[t, k, 0], mu[t, k, 1], color='cyan', s=10, alpha=0.3)

    # 4. Plot Ground Truth (if available) to see the 'Drift'
    if actual_coords is not None:
        ax.plot(actual_coords[:, 0], actual_coords[:, 1], color='orange', 
                lw=2, label='Actual Path', linestyle='--')
        ax.scatter(actual_coords[-1, 0], actual_coords[-1, 1], color='orange', s=100, marker='X')

    plt.title("Scouting Forecast: Predicted 3-Second Movement Intensity", color='white', fontsize=16)
    plt.legend()
    plt.gca().invert_yaxis()
    plt.axis('off')
    plt.show()

generate_scouting_heatmap(loaded_mdn_model, demo_sequences)